In [ ]:
%%configure
{ "vCores": { "parameterName": "pipelinecore", "defaultValue": 2 }}

# TPC-DS concurrency ladder — Direct Lake over mirrored Databricks vs Direct Lake on OneLake

The arms, over the **same rows**, written different ways:

| model | arm |
|---|---|
| `tpcds_sf{sf}_default` | Direct Lake over the **mirrored Databricks** catalog; the same rows written by Databricks with a plain `saveAsTable` under the configuration-only recipe -- Optimized Writes at a fixed bin (one ~6M-row row group per file), dictionary kept, **no sort** |
| `tpcds_sf{sf}_vorder` | Direct Lake on OneLake; the same rows rewritten by **Fabric Spark** with V-Order, ZSTD, Optimize Write at 1 GB, partitioned by date and Z-ordered |

All are `directLakeOnly`, so a query Direct Lake cannot serve fails rather than quietly falling
back to the SQL endpoint and logging a pushdown time that would read as a slow layout.

The protocol is the **white paper's own**: ONE model, `runs` (3) consecutive load tests over it.
The arm's semantic model is deleted, recreated from the paper's TMDL (embedded below) and
reframed; then the 24-query suite runs three times at `concurrent_threads` readers, back to back:

| run | `cache` | readers | what it is |
|---|---|---|---|
| 1 | `0` | `concurrent_threads` | the model is seconds old, so nothing is resident: this run **pays the transcode**, query by query, exactly as their Run 1 does |
| 2 | `1` | `concurrent_threads` | warm |
| 3 | `1` | `concurrent_threads` | warm — and run 1 → run 3 is the **warming curve**, which is the whole point |

Their published per-query P50s (`paper/loadtest_p50_per_run.csv`) carry the same `Run` column, so
ours sit beside theirs run by run. At SF100/20 users their V-Order arm warms (99.5 / 5.0 / 6.2 s)
while their mirrored arm never does (114.8 / 142.7 / 166.9 s, degrading) — a fixed mirroring tax
cannot produce that shape, but layout can, and that is what this measures.

The ONE deviation from their protocol: **the model is deleted after run 3**, so nothing holds
capacity memory into the next arm. A run that FAILS keeps its model, so there is something left to
inspect.

There is no separate transcode probe any more. Running one first would page in every column the
suite reads and flatten run 1 into run 2, destroying the curve. `TRANSCODE_DAX` is still embedded
below and can be run by hand against a model nothing else has touched.

Every query filters its fact on a randomised `cache_buster` value, so no thread is served another
thread's cached result.

Parameters below are overridden per rung by `run_benchmark_pipeline`.

In [ ]:
ws_name = ""                     # Fabric workspace display name; resolved from context if blank
lh_name = "tpcds_bench"          # lakehouse holding the results table and the query file
results_table = "perfresults3"   # Delta table under Tables/dbo/. `perfresults` is the OLD probe +
                                 # one-warm-pass protocol and is deliberately left alone; point a
                                 # rehearsal at e.g. perfresults3_smoke to keep it out of the real
                                 # table
sf = 100                         # scale factor: picks the tpcds_sf{sf}_* schemas AND the schema
                                 # baked into the model definition created below
arms = "cluster"                 # which arms to measure: default, default, defaultf8, cluster,
                                 # clustersn, partition, vorder, vonly, duckdb, ducksort, or a
                                 # comma list.
                                 # Not every arm exists at every sf, so naming one is the normal case
models = ""                      # explicit model names override `arms` entirely; a pipeline
                                 # passes scalars. A pipeline whose `models` parameter is the
                                 # empty string delivers it as None, hence the `or ""` below
iterations = 1                   # passes over the suite INSIDE one thread, inside one load test.
                                 # NOT `runs`, which is separate load tests. Stays 1: the paper's
                                 # Run column is load tests, not repeats within a thread
concurrent_threads = 2
nbr_queries = 24                 # the whole captured suite; never 0
delay_sec = 4
sf_label = 0                     # recorded in loadtest_id only; 0 -> sf. The real SF is measured
                                 # per thread regardless
runs = 3                         # load tests over ONE model -- their Run column: 1, 2, 3

In [ ]:
import time

import notebookutils
from notebookutils.common import configs

configs.tokenCacheEnabled = False

sf = int(sf)
sf_label = int(sf_label) or sf
runs = int(runs)
# Everything scale-factor-shaped is derived from `sf`, never hardcoded: this notebook runs at any
# scale factor without being regenerated. Naming the models explicitly still wins, so a single arm
# can be measured (the V-Order arm does not exist at every sf).
SCHEMA = {"default": f"tpcds_sf{sf}_default",
          # 8M rows per file, one row group each: the same unordered write as `default` at 6M,
          # to test whether segment SIZE is a lever at all with no ordering to eliminate on.
          "defaultf8": f"tpcds_sf{sf}_defaultf8",
          # WITHDRAWN. The two-row-groups-per-file geometry, from the fortnight the recipe carried
          # parquet.block.row.count.limit. It tied `default` and the key left the recipe.
          "default2rg": f"tpcds_sf{sf}_default2rg", "cluster": f"tpcds_sf{sf}_cluster",
          "partition": f"tpcds_sf{sf}_partition", "vorder": f"tpcds_sf{sf}_vorder",
          "vonly": f"tpcds_sf{sf}_vonly",
          # The clustered arm in snappy, with the 128 MB byte target the first one predates.
          "clustersn": f"tpcds_sf{sf}_clustersn",
          # The delta_rs / delta-rs arms (build_duckdb.ipynb). Third writer, two variants:
          # `duckdb` sorts on a key duckrun picked itself, `ducksort` on the date key.
          "duckdb": f"tpcds_sf{sf}_duckdb", "ducksort": f"tpcds_sf{sf}_ducksort"}
_arms = [a.strip() for a in (arms or "default,cluster,vorder").split(",") if a.strip()]
_bad = [a for a in _arms if a not in SCHEMA]
if _bad:
    raise ValueError(f"unknown arm(s) {_bad}; expected any of {list(SCHEMA)}")
model_to_test = ([m.strip() for m in (models or "").split(",") if m.strip()]
                 or [SCHEMA[a] for a in _arms])
# semantic model name -> the Pattern label recorded in loadtest_id. By SUFFIX, one entry per arm:
# a binary "vorder or else the rest" would stamp every Databricks arm with one label and merge
# their rows together.
# Internal only -- these land in loadtest_id and never on a chart, so the existing values stay as
# they are rather than being renamed under rows already in perfresults3.
_SUFFIX = {"_defaultf8": "dbxdefaultf8", "_default": "dbxdefault", "_default2rg": "dbxdefault2rg", "_cluster": "dbxcluster", "_partition": "dbxpartition",
           "_clustersn": "dbxclustersn",         # endswith: does not match "_cluster"
           "_vorder": "fabvorder", "_vonly": "fabvonly",
           "_duckdb": "duckauto", "_ducksort": "ducksort"}
PATTERN = {m: next((p for s, p in _SUFFIX.items() if m.endswith(s)), m) for m in model_to_test}
_unlabelled = [m for m, p in PATTERN.items() if p == m]
if _unlabelled:
    raise ValueError(f"model(s) {_unlabelled} end in none of {list(_SUFFIX)}; the Pattern label would be the model name")
print(f"sf={sf}  models={model_to_test}")
if not ws_name:
    ws_name = notebookutils.runtime.context.get("currentWorkspaceName", "")

workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
lakehouse_id = notebookutils.lakehouse.get(lh_name)["id"]

# GUIDs, not friendly names. This tenant has OneLake friendly-name support DISABLED, so a
# `<workspace>/<lakehouse>.Lakehouse/...` path is refused outright with
# `FriendlyNameSupportDisabled: WorkspaceId and ArtifactId should be either valid Guids or valid
# Names`. The GUID form works everywhere, so it is the one to use regardless of tenant setting.
# perfresults3, NOT perfresults: under this protocol run 1 is a full 24-query load test on a
# fresh model, where in the old table the first pass was a single probe query. The two tables
# answer differently-defined questions and must never be unioned, so the new one starts empty.
delta_path = (f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/"
              f"{lakehouse_id}/Tables/dbo/{results_table}")
# Empty: RunPerfScenario reads the suite embedded in itself. Set an https URL or a path here to
# override it with a file.
queryfile = ""

def _results_rows() -> int:
    """Rows currently in the results table, or 0 before it exists. Used to prove a pass actually
    recorded something rather than trusting that runMultiple raised on failure -- it does not
    always, and a swallowed failure is indistinguishable from success without this.

    A table that CANNOT BE READ is not an empty table. Returning 0 for both once turned an
    unreadable path into "every virtual user failed identically" and sent an hour of debugging
    at the wrong thing, so only the genuine not-created-yet case returns 0.
    """
    from deltalake import DeltaTable
    from deltalake.exceptions import TableNotFoundError
    try:
        return DeltaTable(delta_path).to_pyarrow_dataset().count_rows()
    except TableNotFoundError:
        return 0


print(f"workspace : {ws_name}")
print(f"models    : {model_to_test}")
print(f"threads   : {concurrent_threads}   runs: {runs}   iterations: {iterations}   "
      f"queries: {nbr_queries}")
print(f"results   : {delta_path}")
print(f"queries   : {queryfile or '<embedded in RunPerfScenario>'}")

In [ ]:
# The paper's own TMDL for both arms, inlined by notebooks/build_notebooks.py from
# paper/model/** (lipinght/DB-DQ-Whitepaper, MIT, 99f9904d) via deploy_paper_model.py:parts_for(). Workspace and item
# GUIDs are already substituted, so REBUILD THESE NOTEBOOKS if either changes:
#     python notebooks/build_notebooks.py --workspace <ws>
import base64, json
EMBEDDED_MODELS = json.loads(r'''{"default":{"model":"tpcds_sf__SF___default","parts":[{"path":"definition/database.tmdl","text":"database tpcds_sf__SF___default\n\tcompatibilityLevel: 1702\n\tcompatibilityMode: powerBI\n\tlanguage: 1033\n\n"},{"path":"definition/expressions.tmdl","text":"expression 'DirectLake - tpcds_sf__SF__' =\n\t\tlet\n\t\t    Source = AzureStorage.DataLake(\"https://onelake.dfs.fabric.microsoft.com/51650f82-6bb5-4023-b0ab-db197d32e0be/01b539f3-4a9d-45ef-b1ef-0ba59552eb21\", [HierarchicalNavigation=true])\n\t\tin\n\t\t    Source\n\tlineageTag: c6e3390d-cc84-4e13-ab93-53308a346336\n\n\tannotation PBI_IncludeFutureArtifacts = False\n\n"},{"path":"definition/model.tmdl","text":"model Model\n\tdirectLakeBehavior: directLakeOnly\n\tculture: en-US\n\tdefaultPowerBIDataSourceVersion: powerBI_V3\n\tsourceQueryCulture: en-US\n\tdataAccessOptions\n\t\tlegacyRedirects\n\t\treturnErrorValuesAsNull\n\nannotation PBI_QueryOrder = [\"DirectLake - tpcds_sf__SF__\"]\n\nannotation __PBI_TimeIntelligenceEnabled = 1\n\nannotation __LastRPTime = 134272970581620514\n\nannotation PBI_ProTooling = [\"DirectLakeOnOneLakeInWeb\",\"WebModelingEdit\"]\n\nannotation __TEdtr = 1\n\nannotation TabularEditor_SerializeOptions = {\"IgnoreInferredObjects\":true,\"IgnoreInferredProperties\":true,\"IgnoreTimestamps\":true,\"SplitMultilineStrings\":true,\"PrefixFilenames\":false,\"LocalTranslations\":true,\"LocalPerspectives\":true,\"LocalRelationships\":true,\"Levels\":[\"Data Sources\",\"Perspectives\",\"Relationships\",\"Roles\",\"Shared Expressions\",\"Tables\",\"Tables/Calculation Items\",\"Tables/Columns\",\"Tables/Hierarchies\",\"Tables/Measures\",\"Tables/Partitions\",\"Translations\"]}\n\nref table store\nref table item\nref table date_dim\nref table store_sales\nref table catalog_page\nref table promotion\nref table ship_mode\nref table catalog_sales\nref table customer_address\nref table customer_demographics\nref table 'Measures 1'\nref table 'Time Unit'\n\n"},{"path":"definition/relationships.tmdl","text":"relationship b3bd8d91-ae6b-4489-83cd-34ca2c3213d8\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_bill_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship b727abac-57ca-484a-8183-1b4c10921d84\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_bill_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship 41e21761-05b3-410b-960e-54805fb2c4f2\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_catalog_page_sk\n\ttoColumn: catalog_page.cp_catalog_page_sk\n\nrelationship c7ce7218-d702-40d6-94b4-80009e453fe9\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship 4d02a5f5-e67a-4f65-a203-192e1c2ef166\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_promo_sk\n\ttoColumn: promotion.p_promo_sk\n\nrelationship 545e95f8-e63b-4a27-9c91-3cd725ca10e0\n\tisActive: false\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship 685424a1-f6c7-42d6-8f44-3cccf5f08db4\n\tisActive: false\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship 2e0c0f18-79bf-48f8-be1a-2c53cc33e257\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_mode_sk\n\ttoColumn: ship_mode.sm_ship_mode_sk\n\nrelationship c5edc5f8-f418-44a4-a8aa-caff6e0bef29\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_sold_date_sk\n\ttoColumn: date_dim.d_date_sk_1\n\nrelationship 9f8c8977-b8ac-4985-aed9-7711ce5004af\n\tisActive: false\n\tfromColumn: promotion.p_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship f824425a-7dd8-4a70-9ee5-2042a74ed97d\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship be9c6d5e-0bed-4db1-8558-326d5dfba951\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship f172b575-b3ce-48a4-b264-2928516104a2\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship 95b347fa-cf7e-4885-8124-86eb2ff56351\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_promo_sk\n\ttoColumn: promotion.p_promo_sk\n\nrelationship c782badb-be54-4423-98b0-22eebad6aa29\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_sold_date_sk\n\ttoColumn: date_dim.d_date_sk_1\n\nrelationship 9e5ca2d3-a5aa-4d43-b00a-07e4a857b1e4\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_store_sk\n\ttoColumn: store.s_store_sk\n\n"},{"path":"definition/tables/Measures 1.tmdl","text":"/// Calculated measures table containing key business metrics for revenue, quantity, profit, tax, and performance analysis across store and catalog channels.\ntable 'Measures 1'\n\tlineageTag: ae7cf165-f530-4f98-a4c5-befd958ff771\n\n\t/// Revenue from catalog sales channel only, based on extended sales price\n\tmeasure 'Catalog Revenue' = SUM('catalog_sales'[cs_ext_sales_price])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 01. Revenue\n\t\tlineageTag: bed34121-c406-4e25-b931-6ea438aede2a\n\n\t/// Revenue from store sales channel only, based on extended sales price\n\tmeasure 'Store Revenue' = SUM('store_sales'[ss_ext_sales_price])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 01. Revenue\n\t\tlineageTag: 1755e841-2eeb-4e1e-9a98-c860eb583a79\n\n\t/// Total units sold through catalog sales channel\n\tmeasure 'Catalog Sales Quantity' = SUM('catalog_sales'[cs_quantity])\n\t\tformatString: #,0\n\t\tdisplayFolder: 02. Quantity\n\t\tlineageTag: 6da96e65-1eb2-4e76-afdb-50d9106f56c1\n\n\t/// Net profit from store sales channel only\n\tmeasure 'Store Net Profit' = SUM('store_sales'[ss_net_profit])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 03. Profit\n\t\tlineageTag: a5a04305-c986-4791-925e-b9115939e453\n\n\t/// Number of unique customers who made store purchases\n\tmeasure 'Store Distinct Customers' = DISTINCTCOUNT('store_sales'[ss_customer_sk])\n\t\tformatString: #,0\n\t\tdisplayFolder: 04. Distinct Counts\n\t\tlineageTag: ace0485f-176a-4401-bb53-41c99ce5c356\n\n\t/// Revenue for the same period in the previous year\n\tmeasure 'Store Revenue Same Period LY' =\n\t\t\t\n\t\t\tCALCULATE(\n\t\t\t    [Store Revenue],\n\t\t\t    SAMEPERIODLASTYEAR('tpcds_calendar')\n\t\t\t)\n\t\tformatString: $#,0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 303dbdc1-0484-496c-b14f-a41b0ef53f82\n\n\t/// Revenue for the same period in the previous year\n\tmeasure 'Store Revenue YoY' =\n\t\t\tVAR CurrentYearRev = [Store Revenue]\n\t\t\tVAR PreviousYearRev = [Store Revenue Same Period LY]\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentYearRev - PreviousYearRev, PreviousYearRev, 0)\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 39648465-7aac-4652-8c2f-d6599df15204\n\n\t/// Catalog Sales Same Period LY\n\tmeasure 'Catalog Sales Same Period LY' =\n\t\t\t\n\t\t\tCALCULATE(\n\t\t\t    [Catalog Sales Quantity],\n\t\t\t    SAMEPERIODLASTYEAR('tpcds_calendar')\n\t\t\t)\n\t\tformatString: 0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: d1d5179d-688b-487c-8638-81780d95eb05\n\n\t/// Catalog Sales YoY\n\tmeasure 'Catalog Sales YoY' =\n\t\t\tVAR CurrentYearCatSales = [Catalog Sales Quantity]\n\t\t\tVAR PreviousYearCatSales = [Catalog Sales Same Period LY]\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentYearCatSales - PreviousYearCatSales, PreviousYearCatSales, 0)\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 0bc09d6d-1bdd-4f7e-ac26-a12e87ccf67d\n\n\t/// Store Profit % by Item Category\n\tmeasure 'Store Profit % by Item Category' = ```\n\t\t\t\n\t\t\tVAR CurrentProfit = [Store Net Profit]\n\t\t\tVAR TotalProfitAllCategories = \n\t\t\t    CALCULATE(\n\t\t\t        [Store Net Profit],\n\t\t\t        ALLEXCEPT(\n\t\t\t            'item',\n\t\t\t            'item'[i_brand],\n\t\t\t            'item'[i_manufact]\n\t\t\t        )\n\t\t\t    )\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentProfit, TotalProfitAllCategories, 0)\n\t\t\t```\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 05. Advanced % Share\n\t\tlineageTag: 04f417ef-ba48-4d9e-873d-76d516ec3a54\n\n\t/// Total revenue from beginning of year to current date selection\n\tmeasure 'Store Revenue YTD' = TOTALYTD([Store Revenue],'tpcds_calendar')\n\t\tformatString: $#,0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 0e0dbb36-1e75-4c77-8b7f-9a462a143dcb\n\n\t/// Dummy\n\tcolumn Dummy\n\t\tformatString: 0\n\t\tlineageTag: c1dfb904-84a8-4dfc-a768-d5e10f70f255\n\t\tsummarizeBy: none\n\t\tisNameInferred\n\t\tsourceColumn: [Dummy]\n\n\tpartition 'Measures 1' = calculated\n\t\tmode: import\n\t\tsource = ROW(\"Dummy\", 1)\n\n"},{"path":"definition/tables/Time Unit.tmdl","text":"/// Field parameter table for dynamic time unit selection in reports, supporting Year and Quarter groupings.\ntable 'Time Unit'\n\tlineageTag: c100712f-ebcb-4d0f-a435-e1a4c8ce3229\n\n\t/// Display name for the time unit selection (Year, Quarter).\n\tcolumn 'Time Unit'\n\t\tlineageTag: bd5d5ab0-6269-4df0-bd39-9616f6e2b7d5\n\t\tsummarizeBy: none\n\t\tsourceColumn: [Value1]\n\t\tsortByColumn: 'Time Unit Order'\n\n\t\trelatedColumnDetails\n\t\t\tgroupByColumn: 'Time Unit Fields'\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// DAX column reference for the selected time unit.\n\tcolumn 'Time Unit Fields'\n\t\tisHidden\n\t\tlineageTag: 4be3ced6-f4db-40c4-84b2-5ef9decf7fee\n\t\tsummarizeBy: none\n\t\tsourceColumn: [Value2]\n\t\tsortByColumn: 'Time Unit Order'\n\n\t\textendedProperty ParameterMetadata =\n\t\t\t\t{\n\t\t\t\t  \"version\": 3,\n\t\t\t\t  \"kind\": 2\n\t\t\t\t}\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Sort order for time unit options in field parameter.\n\tcolumn 'Time Unit Order'\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 36566b71-2a4f-4ddb-86c0-b4c7b3d8ab7f\n\t\tsummarizeBy: sum\n\t\tsourceColumn: [Value3]\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition 'Time Unit' = calculated\n\t\tmode: import\n\t\tsource =\n\t\t\t\t{\n\t\t\t\t    (\"Year\", NAMEOF('date_dim'[d_year]), 0),\n\t\t\t\t    (\"Quarter\", NAMEOF('date_dim'[d_quarter_name]), 1)\n\t\t\t\t}\n\n\tannotation PBI_Id = 706957f972f8497896950148d2d5f0af\n\n"},{"path":"definition/tables/catalog_page.tmdl","text":"/// Catalog page dimension containing information about catalog pages used in catalog sales campaigns.\ntable catalog_page\n\tlineageTag: b4e5474e-8b18-42b8-9a56-9c2416950a0a\n\tsourceLineageTag: [dbo].[catalog_page]\n\n\t/// Catalog page surrogate key - unique identifier for each catalog page.\n\tcolumn cp_catalog_page_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 71a9dba7-e582-461e-b8de-f37322137084\n\t\tsourceLineageTag: cp_catalog_page_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: cp_catalog_page_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type or category of the catalog page.\n\tcolumn cp_type\n\t\tdataType: string\n\t\tlineageTag: bdf42608-dc88-489d-90ce-24a87bc40378\n\t\tsourceLineageTag: cp_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: cp_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition catalog_page = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: catalog_page\n\t\t\tschemaName: tpcds_sf__SF___default\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/catalog_sales.tmdl","text":"/// Fact table containing catalog sales transactions with pricing, quantities, profits, and foreign keys to related dimensions.\ntable catalog_sales\n\tlineageTag: c38e1686-b3a0-473b-8604-8efe0410ea48\n\tsourceLineageTag: [dbo].[catalog_sales]\n\n\t/// Foreign key to date dimension for sale date.\n\tcolumn cs_sold_date_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: ac2ce865-caed-4039-bda1-a182c516cfdd\n\t\tsourceLineageTag: cs_sold_date_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_sold_date_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension for billing customer.\n\tcolumn cs_bill_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 310a5931-6249-4ad7-86b4-7aca5fbc39a9\n\t\tsourceLineageTag: cs_bill_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_bill_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension for billing address.\n\tcolumn cs_bill_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 51e6cbcf-e98d-4ea9-865e-a500419bec99\n\t\tsourceLineageTag: cs_bill_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_bill_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension for shipping customer.\n\tcolumn cs_ship_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4a433af2-d889-41d8-9c6f-1f05f94f0070\n\t\tsourceLineageTag: cs_ship_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension for shipping address.\n\tcolumn cs_ship_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 25b41390-6ff3-4f15-b0e6-f26a3da61ef5\n\t\tsourceLineageTag: cs_ship_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to catalog page dimension.\n\tcolumn cs_catalog_page_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: b01fc3a0-0a51-4660-a5d3-56d28ea386cd\n\t\tsourceLineageTag: cs_catalog_page_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_catalog_page_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to ship mode dimension.\n\tcolumn cs_ship_mode_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 0a85820a-e8f9-41f9-b50e-9610890408e8\n\t\tsourceLineageTag: cs_ship_mode_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_mode_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension.\n\tcolumn cs_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9024dff9-9629-4f0e-abae-0b29f00d568d\n\t\tsourceLineageTag: cs_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to promotion dimension.\n\tcolumn cs_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: dbfb7b81-9eef-4b86-aac0-a48cc36d0168\n\t\tsourceLineageTag: cs_promo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Catalog order number for this transaction.\n\tcolumn cs_order_number\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: ccec0342-6144-411f-bc2d-90234bcfb5ac\n\t\tsourceLineageTag: cs_order_number\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_order_number\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quantity of items ordered in this transaction.\n\tcolumn cs_quantity\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 0f5abf99-45b4-495b-bad3-795d0e8e5802\n\t\tsourceLineageTag: cs_quantity\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_quantity\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Wholesale cost per unit for this transaction.\n\tcolumn cs_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 37d760e7-f755-42e9-9dba-3a28cc211ae7\n\t\tsourceLineageTag: cs_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// List price per unit at time of order.\n\tcolumn cs_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 0eda1627-f565-444d-92e5-1a5bb0929e4d\n\t\tsourceLineageTag: cs_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Actual sales price per unit (after discounts).\n\tcolumn cs_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: eebce86e-da48-4799-8f93-1a8e69de6605\n\t\tsourceLineageTag: cs_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended sales price (sales price \u00d7 quantity).\n\tcolumn cs_ext_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: f0c74fde-bf40-47e4-9352-6bc4d4f219a0\n\t\tsourceLineageTag: cs_ext_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_ext_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended list price (list price \u00d7 quantity).\n\tcolumn cs_ext_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 845b8a06-567d-49c0-9455-1a469d81f646\n\t\tsourceLineageTag: cs_ext_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_ext_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Net profit for this transaction (sales price - wholesale cost).\n\tcolumn cs_net_profit\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 341a4a24-e726-4b20-8a50-967f689a5d01\n\t\tsourceLineageTag: cs_net_profit\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_net_profit\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Technical column used for cache invalidation in DirectLake mode.\n\tcolumn cache_buster\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 33947ffc-dc12-42ab-af62-5b00c42adf7a\n\t\tsourceLineageTag: cache_buster\n\t\tsummarizeBy: none\n\t\tsourceColumn: cache_buster\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition catalog_sales = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: catalog_sales\n\t\t\tschemaName: tpcds_sf__SF___default\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/customer_address.tmdl","text":"/// Customer address dimension containing geographic information for billing and shipping addresses.\ntable customer_address\n\tlineageTag: 75246b34-d9f4-46fd-813f-bc5f0e47ab8c\n\tsourceLineageTag: [dbo].[customer_address]\n\n\t/// Customer address surrogate key - unique identifier for each address.\n\tcolumn ca_address_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9084b5d7-d188-4fb9-9c92-211d5b68f397\n\t\tsourceLineageTag: ca_address_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_address_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// City name for the customer address.\n\tcolumn ca_city\n\t\tdataType: string\n\t\tlineageTag: 03b2fb94-a08c-4010-9157-b91edcb60daf\n\t\tsourceLineageTag: ca_city\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_city\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// County name for the customer address.\n\tcolumn ca_county\n\t\tdataType: string\n\t\tlineageTag: 0ef7c0d6-1e17-42c5-a9bb-fb014c91398a\n\t\tsourceLineageTag: ca_county\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_county\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// State or province for the customer address.\n\tcolumn ca_state\n\t\tdataType: string\n\t\tlineageTag: 24c6ee20-e4ab-4ca3-aa30-934ba66a55db\n\t\tsourceLineageTag: ca_state\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_state\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Postal code for the customer address.\n\tcolumn ca_zip\n\t\tdataType: string\n\t\tlineageTag: a57bf4fe-e3de-4848-81bd-e71dd885c99f\n\t\tsourceLineageTag: ca_zip\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_zip\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type of location (e.g., residential, commercial).\n\tcolumn ca_location_type\n\t\tdataType: string\n\t\tlineageTag: 9a90fb17-4314-40df-84ae-071cdd4eb3c7\n\t\tsourceLineageTag: ca_location_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_location_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition customer_address = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: customer_address\n\t\t\tschemaName: tpcds_sf__SF___default\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/customer_demographics.tmdl","text":"/// Customer demographics dimension containing education status and marital status attributes for customer segmentation.\ntable customer_demographics\n\tlineageTag: 52f8f268-90f3-4f40-a527-07a74d9e9baf\n\tsourceLineageTag: [dbo].[customer_demographics]\n\n\t/// Customer demographics surrogate key - unique identifier for each demographic profile.\n\tcolumn cd_demo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: d2cad4d0-4a88-41ab-b7ee-94ab73973694\n\t\tsourceLineageTag: cd_demo_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_demo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Marital status of the customer.\n\tcolumn cd_marital_status\n\t\tdataType: string\n\t\tlineageTag: e29a3681-2c0d-4579-9b35-5112d86c0f46\n\t\tsourceLineageTag: cd_marital_status\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_marital_status\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Education level of the customer.\n\tcolumn cd_education_status\n\t\tdataType: string\n\t\tlineageTag: 43260b21-0847-455f-a853-e2233af2a62e\n\t\tsourceLineageTag: cd_education_status\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_education_status\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition customer_demographics = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: customer_demographics\n\t\t\tschemaName: tpcds_sf__SF___default\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/date_dim.tmdl","text":"/// Date dimension providing calendar hierarchy with year, quarter, month, and day-of-week attributes for time-based analysis.\ntable date_dim\n\tlineageTag: 9e7ec2de-382c-4a3b-9589-1ece445bfa27\n\tsourceLineageTag: [dbo].[date_dim]\n\n\t/// Date surrogate key - unique identifier for each calendar date.\n\tcolumn d_date_sk_1\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 63841738-6970-46ac-8106-970384b1052b\n\t\tsourceLineageTag: d_date_sk_1\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_date_sk_1\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Calendar date value.\n\tcolumn d_date\n\t\tdataType: dateTime\n\t\tformatString: General Date\n\t\tlineageTag: dc08a5e6-1a30-4d5f-9b03-1c98c7bac892\n\t\tsourceLineageTag: d_date\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_date\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Calendar year (e.g., 2023).\n\tcolumn d_year\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: fcf4b9e9-c880-4ee0-907c-3a18951b0809\n\t\tsourceLineageTag: d_year\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_year\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Day of week (1=Sunday, 7=Saturday).\n\tcolumn d_dow\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 339ac74d-ab31-4d4e-93eb-524b9f7e211c\n\t\tsourceLineageTag: d_dow\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_dow\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Month of year (1-12).\n\tcolumn d_moy\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: faa3deb0-9def-4334-bd42-a1b9d60b51b7\n\t\tsourceLineageTag: d_moy\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_moy\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Day of month (1-31).\n\tcolumn d_dom\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 83cd6ebd-7811-4616-a884-9930f66f8bfe\n\t\tsourceLineageTag: d_dom\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_dom\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quarter of year (1-4).\n\tcolumn d_qoy\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 4962ee0a-fcb9-4bf8-a5c7-43a66a94e67f\n\t\tsourceLineageTag: d_qoy\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_qoy\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quarter name (e.g., 'Q1 2023').\n\tcolumn d_quarter_name\n\t\tdataType: string\n\t\tlineageTag: 2426fa03-107c-4f51-a0b4-c59bb4b85d4f\n\t\tsourceLineageTag: d_quarter_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_quarter_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition date_dim = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: date_dim\n\t\t\tschemaName: tpcds_sf__SF___default\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n\tcalendar tpcds_calendar\n\t\tlineageTag: dd529b7a-99fe-4cb5-928c-df1abed499f1\n\n\t\tcalendarColumnGroup = year\n\t\t\tprimaryColumn: d_year\n\n\t\tcalendarColumnGroup = quarter\n\t\t\tprimaryColumn: d_quarter_name\n\n\t\tcalendarColumnGroup = quarterOfYear\n\t\t\tprimaryColumn: d_qoy\n\n\t\tcalendarColumnGroup = date\n\t\t\tprimaryColumn: d_date\n\n"},{"path":"definition/tables/item.tmdl","text":"/// Product dimension containing item details such as brand, category, class, color, pricing, and manufacturing information.\ntable item\n\tlineageTag: 0f958ae1-af31-4b29-87a3-7e9ee589887e\n\tsourceLineageTag: [dbo].[item]\n\n\t/// Item surrogate key - unique identifier for each product.\n\tcolumn i_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 23893369-a589-4284-afa3-b1e291864600\n\t\tsourceLineageTag: i_item_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Current retail price of the product.\n\tcolumn i_current_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 70b27235-eefb-4732-8e0d-64c9576f0776\n\t\tsourceLineageTag: i_current_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_current_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Wholesale cost paid for the product.\n\tcolumn i_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 90cbdd03-df2f-4439-941d-de295ceb5135\n\t\tsourceLineageTag: i_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Brand name of the product.\n\tcolumn i_brand\n\t\tdataType: string\n\t\tlineageTag: 0c701039-61ca-4f5f-872b-aab6677ef45c\n\t\tsourceLineageTag: i_brand\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_brand\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Product class or subcategory within the main category.\n\tcolumn i_class\n\t\tdataType: string\n\t\tlineageTag: 5bdf127c-d558-46b2-98e6-0e294148d466\n\t\tsourceLineageTag: i_class\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_class\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Product category classification.\n\tcolumn i_category\n\t\tdataType: string\n\t\tlineageTag: 138de914-55e9-4a46-b5b9-6707b88a38e9\n\t\tsourceLineageTag: i_category\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_category\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Manufacturer or producer of the product.\n\tcolumn i_manufact\n\t\tdataType: string\n\t\tlineageTag: e6852c2b-2bc8-4fb1-86cb-d7ebe7c18637\n\t\tsourceLineageTag: i_manufact\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_manufact\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Size specification of the product.\n\tcolumn i_size\n\t\tdataType: string\n\t\tlineageTag: 0aeb0963-bc02-4389-a3fd-ab7232caaf9c\n\t\tsourceLineageTag: i_size\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_size\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Primary color of the product.\n\tcolumn i_color\n\t\tdataType: string\n\t\tlineageTag: d6709cb2-852f-485a-a0de-8cdd07941038\n\t\tsourceLineageTag: i_color\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_color\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition item = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: item\n\t\t\tschemaName: tpcds_sf__SF___default\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/promotion.tmdl","text":"/// Promotion dimension containing promotional campaign details including costs and promotional names.\ntable promotion\n\tlineageTag: 5942c68a-d07a-4a23-8864-b2c733be46c8\n\tsourceLineageTag: [dbo].[promotion]\n\n\t/// Promotion surrogate key - unique identifier for each promotional campaign.\n\tcolumn p_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 0299ca0a-f58f-4ebe-b752-ffa6b63f9760\n\t\tsourceLineageTag: p_promo_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension (for item-specific promotions).\n\tcolumn p_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 92f8bc42-df79-4a35-8a37-10a02a937c40\n\t\tsourceLineageTag: p_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: p_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Cost associated with running this promotional campaign.\n\tcolumn p_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: c42871df-1b39-4b1e-83cb-142e6864a1e0\n\t\tsourceLineageTag: p_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Name or title of the promotional campaign.\n\tcolumn p_promo_name\n\t\tdataType: string\n\t\tlineageTag: 6ceed2f6-4e12-4edc-b161-8251a356ec17\n\t\tsourceLineageTag: p_promo_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_promo_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition promotion = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: promotion\n\t\t\tschemaName: tpcds_sf__SF___default\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/ship_mode.tmdl","text":"/// Shipping method dimension containing carrier information and shipping type classifications.\ntable ship_mode\n\tlineageTag: db1b87b5-f584-4f30-b70e-0eb00c8b45b8\n\tsourceLineageTag: [dbo].[ship_mode]\n\n\t/// Ship mode surrogate key - unique identifier for each shipping method.\n\tcolumn sm_ship_mode_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4c485c32-fa22-4df6-b4a8-a8df04e2f730\n\t\tsourceLineageTag: sm_ship_mode_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_ship_mode_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type or category of shipping method.\n\tcolumn sm_type\n\t\tdataType: string\n\t\tlineageTag: c49fc6f8-d030-49a2-8105-6a45d1a81ac2\n\t\tsourceLineageTag: sm_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Internal code for the shipping method.\n\tcolumn sm_code\n\t\tdataType: string\n\t\tlineageTag: f54a092d-e847-4cee-84e6-5b813cbf5280\n\t\tsourceLineageTag: sm_code\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_code\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Shipping carrier or company name.\n\tcolumn sm_carrier\n\t\tdataType: string\n\t\tlineageTag: aca70983-6b51-4291-83f8-77fac2e381fe\n\t\tsourceLineageTag: sm_carrier\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_carrier\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition ship_mode = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: ship_mode\n\t\t\tschemaName: tpcds_sf__SF___default\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/store.tmdl","text":"/// Store dimension containing retail location information including geographic details, management, and tax rates.\ntable store\n\tlineageTag: 6835d4a1-f40a-40aa-8a65-98aa4e06a406\n\tsourceLineageTag: [dbo].[store]\n\n\t/// Store surrogate key - unique identifier for each store location.\n\tcolumn s_store_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 14502476-8801-4068-b7c4-875ce7cac939\n\t\tsourceLineageTag: s_store_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_store_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Store name or identifier for the retail location.\n\tcolumn s_store_name\n\t\tdataType: string\n\t\tlineageTag: 4e78d335-4974-4d41-8402-ea89f63f9005\n\t\tsourceLineageTag: s_store_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_store_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Name of the store manager.\n\tcolumn s_manager\n\t\tdataType: string\n\t\tlineageTag: 3a8ddb5b-2ada-42d0-9ef9-89cd96b435b6\n\t\tsourceLineageTag: s_manager\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_manager\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Name of the market manager overseeing this store.\n\tcolumn s_market_manager\n\t\tdataType: string\n\t\tlineageTag: 37c8562d-3ab6-47ad-9b99-2ad1d72e6cd6\n\t\tsourceLineageTag: s_market_manager\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_market_manager\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// City where the store is located.\n\tcolumn s_city\n\t\tdataType: string\n\t\tlineageTag: a1a9209c-8194-4f96-8e94-69e545e064ba\n\t\tsourceLineageTag: s_city\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_city\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// County where the store is located.\n\tcolumn s_county\n\t\tdataType: string\n\t\tlineageTag: e219113c-fe81-45b0-b6be-f02e7ce75403\n\t\tsourceLineageTag: s_county\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_county\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// State or province where the store is located.\n\tcolumn s_state\n\t\tdataType: string\n\t\tlineageTag: fbaeba14-56ed-496d-a004-83f851771750\n\t\tsourceLineageTag: s_state\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_state\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Postal code for the store location.\n\tcolumn s_zip\n\t\tdataType: string\n\t\tlineageTag: 44e7ea79-56d6-4d20-8243-92c35919dcc3\n\t\tsourceLineageTag: s_zip\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_zip\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Tax rate percentage applied at this store location.\n\tcolumn s_tax_percentage\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: ba6d903d-f426-46b7-90a8-756a18261f2a\n\t\tsourceLineageTag: s_tax_percentage\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_tax_percentage\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\tpartition store = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: store\n\t\t\tschemaName: tpcds_sf__SF___default\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/store_sales.tmdl","text":"/// Fact table containing retail store sales transactions with pricing, quantities, profits, and foreign keys to related dimensions.\ntable store_sales\n\tlineageTag: 6c7a2caf-3c22-4f38-a6aa-895a77e2e1c9\n\tsourceLineageTag: [dbo].[store_sales]\n\n\t/// Foreign key to date dimension for sale date.\n\tcolumn ss_sold_date_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4e30d94a-216f-4eaf-9858-fb22c6e6d57e\n\t\tsourceLineageTag: ss_sold_date_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_sold_date_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension.\n\tcolumn ss_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 2b151014-7df5-45f0-9907-03cbe191952b\n\t\tsourceLineageTag: ss_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Customer surrogate key for the transaction.\n\tcolumn ss_customer_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 558da56d-ea73-4562-b35c-ff94e774cca3\n\t\tsourceLineageTag: ss_customer_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_customer_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension.\n\tcolumn ss_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: e65f473f-bb26-4c7d-90f1-139de12a51ab\n\t\tsourceLineageTag: ss_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension.\n\tcolumn ss_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: cb65f3a1-6240-4baa-bfb2-4b24d39e2e87\n\t\tsourceLineageTag: ss_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to store dimension.\n\tcolumn ss_store_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 809a4a04-3bbf-4067-b54f-ee4d5fbbf6d4\n\t\tsourceLineageTag: ss_store_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_store_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to promotion dimension.\n\tcolumn ss_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9a64bbfb-0ae6-4919-b4e9-73e6d90353b3\n\t\tsourceLineageTag: ss_promo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Transaction ticket or receipt number.\n\tcolumn ss_ticket_number\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: d8ce37f4-a12c-4558-b131-66631546b607\n\t\tsourceLineageTag: ss_ticket_number\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ticket_number\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quantity of items sold in this transaction.\n\tcolumn ss_quantity\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 318e6a7b-88a2-413c-8257-315f66313d79\n\t\tsourceLineageTag: ss_quantity\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_quantity\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Wholesale cost per unit for this transaction.\n\tcolumn ss_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: d02100cc-2804-43d7-b752-8e98d579e202\n\t\tsourceLineageTag: ss_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// List price per unit at time of sale.\n\tcolumn ss_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 4f32f293-c19e-497e-906e-089bc204cde9\n\t\tsourceLineageTag: ss_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Actual sales price per unit (after discounts).\n\tcolumn ss_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: e620a719-f6e0-4965-8326-f4a2d255999b\n\t\tsourceLineageTag: ss_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended sales price (sales price \u00d7 quantity).\n\tcolumn ss_ext_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 8b66819a-a151-4fc7-9ec0-cacf7b6e8e01\n\t\tsourceLineageTag: ss_ext_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ext_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended list price (list price \u00d7 quantity).\n\tcolumn ss_ext_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: bc8e9c57-98e2-4032-a3fd-69808e3cdd8b\n\t\tsourceLineageTag: ss_ext_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ext_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Net profit for this transaction (sales price - wholesale cost).\n\tcolumn ss_net_profit\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: d2d6233f-6944-40b5-97d7-2870da393d1b\n\t\tsourceLineageTag: ss_net_profit\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_net_profit\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Technical column used for cache invalidation in DirectLake mode.\n\tcolumn cache_buster\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 7ae60894-bf63-4fef-a327-bf8796466374\n\t\tsourceLineageTag: cache_buster\n\t\tsummarizeBy: none\n\t\tsourceColumn: cache_buster\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition store_sales = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: store_sales\n\t\t\tschemaName: tpcds_sf__SF___default\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition.pbism","text":"{\"$schema\": \"https://developer.microsoft.com/json-schemas/fabric/item/semanticModel/definitionProperties/1.0.0/schema.json\", \"version\": \"5.0\", \"settings\": {}}"}]},"defaultf8":{"model":"tpcds_sf__SF___defaultf8","parts":[{"path":"definition/database.tmdl","text":"database tpcds_sf__SF___defaultf8\n\tcompatibilityLevel: 1702\n\tcompatibilityMode: powerBI\n\tlanguage: 1033\n\n"},{"path":"definition/expressions.tmdl","text":"expression 'DirectLake - tpcds_sf__SF__' =\n\t\tlet\n\t\t    Source = AzureStorage.DataLake(\"https://onelake.dfs.fabric.microsoft.com/51650f82-6bb5-4023-b0ab-db197d32e0be/01b539f3-4a9d-45ef-b1ef-0ba59552eb21\", [HierarchicalNavigation=true])\n\t\tin\n\t\t    Source\n\tlineageTag: c6e3390d-cc84-4e13-ab93-53308a346336\n\n\tannotation PBI_IncludeFutureArtifacts = False\n\n"},{"path":"definition/model.tmdl","text":"model Model\n\tdirectLakeBehavior: directLakeOnly\n\tculture: en-US\n\tdefaultPowerBIDataSourceVersion: powerBI_V3\n\tsourceQueryCulture: en-US\n\tdataAccessOptions\n\t\tlegacyRedirects\n\t\treturnErrorValuesAsNull\n\nannotation PBI_QueryOrder = [\"DirectLake - tpcds_sf__SF__\"]\n\nannotation __PBI_TimeIntelligenceEnabled = 1\n\nannotation __LastRPTime = 134272970581620514\n\nannotation PBI_ProTooling = [\"DirectLakeOnOneLakeInWeb\",\"WebModelingEdit\"]\n\nannotation __TEdtr = 1\n\nannotation TabularEditor_SerializeOptions = {\"IgnoreInferredObjects\":true,\"IgnoreInferredProperties\":true,\"IgnoreTimestamps\":true,\"SplitMultilineStrings\":true,\"PrefixFilenames\":false,\"LocalTranslations\":true,\"LocalPerspectives\":true,\"LocalRelationships\":true,\"Levels\":[\"Data Sources\",\"Perspectives\",\"Relationships\",\"Roles\",\"Shared Expressions\",\"Tables\",\"Tables/Calculation Items\",\"Tables/Columns\",\"Tables/Hierarchies\",\"Tables/Measures\",\"Tables/Partitions\",\"Translations\"]}\n\nref table store\nref table item\nref table date_dim\nref table store_sales\nref table catalog_page\nref table promotion\nref table ship_mode\nref table catalog_sales\nref table customer_address\nref table customer_demographics\nref table 'Measures 1'\nref table 'Time Unit'\n\n"},{"path":"definition/relationships.tmdl","text":"relationship b3bd8d91-ae6b-4489-83cd-34ca2c3213d8\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_bill_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship b727abac-57ca-484a-8183-1b4c10921d84\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_bill_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship 41e21761-05b3-410b-960e-54805fb2c4f2\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_catalog_page_sk\n\ttoColumn: catalog_page.cp_catalog_page_sk\n\nrelationship c7ce7218-d702-40d6-94b4-80009e453fe9\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship 4d02a5f5-e67a-4f65-a203-192e1c2ef166\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_promo_sk\n\ttoColumn: promotion.p_promo_sk\n\nrelationship 545e95f8-e63b-4a27-9c91-3cd725ca10e0\n\tisActive: false\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship 685424a1-f6c7-42d6-8f44-3cccf5f08db4\n\tisActive: false\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship 2e0c0f18-79bf-48f8-be1a-2c53cc33e257\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_mode_sk\n\ttoColumn: ship_mode.sm_ship_mode_sk\n\nrelationship c5edc5f8-f418-44a4-a8aa-caff6e0bef29\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_sold_date_sk\n\ttoColumn: date_dim.d_date_sk_1\n\nrelationship 9f8c8977-b8ac-4985-aed9-7711ce5004af\n\tisActive: false\n\tfromColumn: promotion.p_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship f824425a-7dd8-4a70-9ee5-2042a74ed97d\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship be9c6d5e-0bed-4db1-8558-326d5dfba951\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship f172b575-b3ce-48a4-b264-2928516104a2\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship 95b347fa-cf7e-4885-8124-86eb2ff56351\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_promo_sk\n\ttoColumn: promotion.p_promo_sk\n\nrelationship c782badb-be54-4423-98b0-22eebad6aa29\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_sold_date_sk\n\ttoColumn: date_dim.d_date_sk_1\n\nrelationship 9e5ca2d3-a5aa-4d43-b00a-07e4a857b1e4\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_store_sk\n\ttoColumn: store.s_store_sk\n\n"},{"path":"definition/tables/Measures 1.tmdl","text":"/// Calculated measures table containing key business metrics for revenue, quantity, profit, tax, and performance analysis across store and catalog channels.\ntable 'Measures 1'\n\tlineageTag: ae7cf165-f530-4f98-a4c5-befd958ff771\n\n\t/// Revenue from catalog sales channel only, based on extended sales price\n\tmeasure 'Catalog Revenue' = SUM('catalog_sales'[cs_ext_sales_price])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 01. Revenue\n\t\tlineageTag: bed34121-c406-4e25-b931-6ea438aede2a\n\n\t/// Revenue from store sales channel only, based on extended sales price\n\tmeasure 'Store Revenue' = SUM('store_sales'[ss_ext_sales_price])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 01. Revenue\n\t\tlineageTag: 1755e841-2eeb-4e1e-9a98-c860eb583a79\n\n\t/// Total units sold through catalog sales channel\n\tmeasure 'Catalog Sales Quantity' = SUM('catalog_sales'[cs_quantity])\n\t\tformatString: #,0\n\t\tdisplayFolder: 02. Quantity\n\t\tlineageTag: 6da96e65-1eb2-4e76-afdb-50d9106f56c1\n\n\t/// Net profit from store sales channel only\n\tmeasure 'Store Net Profit' = SUM('store_sales'[ss_net_profit])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 03. Profit\n\t\tlineageTag: a5a04305-c986-4791-925e-b9115939e453\n\n\t/// Number of unique customers who made store purchases\n\tmeasure 'Store Distinct Customers' = DISTINCTCOUNT('store_sales'[ss_customer_sk])\n\t\tformatString: #,0\n\t\tdisplayFolder: 04. Distinct Counts\n\t\tlineageTag: ace0485f-176a-4401-bb53-41c99ce5c356\n\n\t/// Revenue for the same period in the previous year\n\tmeasure 'Store Revenue Same Period LY' =\n\t\t\t\n\t\t\tCALCULATE(\n\t\t\t    [Store Revenue],\n\t\t\t    SAMEPERIODLASTYEAR('tpcds_calendar')\n\t\t\t)\n\t\tformatString: $#,0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 303dbdc1-0484-496c-b14f-a41b0ef53f82\n\n\t/// Revenue for the same period in the previous year\n\tmeasure 'Store Revenue YoY' =\n\t\t\tVAR CurrentYearRev = [Store Revenue]\n\t\t\tVAR PreviousYearRev = [Store Revenue Same Period LY]\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentYearRev - PreviousYearRev, PreviousYearRev, 0)\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 39648465-7aac-4652-8c2f-d6599df15204\n\n\t/// Catalog Sales Same Period LY\n\tmeasure 'Catalog Sales Same Period LY' =\n\t\t\t\n\t\t\tCALCULATE(\n\t\t\t    [Catalog Sales Quantity],\n\t\t\t    SAMEPERIODLASTYEAR('tpcds_calendar')\n\t\t\t)\n\t\tformatString: 0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: d1d5179d-688b-487c-8638-81780d95eb05\n\n\t/// Catalog Sales YoY\n\tmeasure 'Catalog Sales YoY' =\n\t\t\tVAR CurrentYearCatSales = [Catalog Sales Quantity]\n\t\t\tVAR PreviousYearCatSales = [Catalog Sales Same Period LY]\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentYearCatSales - PreviousYearCatSales, PreviousYearCatSales, 0)\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 0bc09d6d-1bdd-4f7e-ac26-a12e87ccf67d\n\n\t/// Store Profit % by Item Category\n\tmeasure 'Store Profit % by Item Category' = ```\n\t\t\t\n\t\t\tVAR CurrentProfit = [Store Net Profit]\n\t\t\tVAR TotalProfitAllCategories = \n\t\t\t    CALCULATE(\n\t\t\t        [Store Net Profit],\n\t\t\t        ALLEXCEPT(\n\t\t\t            'item',\n\t\t\t            'item'[i_brand],\n\t\t\t            'item'[i_manufact]\n\t\t\t        )\n\t\t\t    )\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentProfit, TotalProfitAllCategories, 0)\n\t\t\t```\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 05. Advanced % Share\n\t\tlineageTag: 04f417ef-ba48-4d9e-873d-76d516ec3a54\n\n\t/// Total revenue from beginning of year to current date selection\n\tmeasure 'Store Revenue YTD' = TOTALYTD([Store Revenue],'tpcds_calendar')\n\t\tformatString: $#,0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 0e0dbb36-1e75-4c77-8b7f-9a462a143dcb\n\n\t/// Dummy\n\tcolumn Dummy\n\t\tformatString: 0\n\t\tlineageTag: c1dfb904-84a8-4dfc-a768-d5e10f70f255\n\t\tsummarizeBy: none\n\t\tisNameInferred\n\t\tsourceColumn: [Dummy]\n\n\tpartition 'Measures 1' = calculated\n\t\tmode: import\n\t\tsource = ROW(\"Dummy\", 1)\n\n"},{"path":"definition/tables/Time Unit.tmdl","text":"/// Field parameter table for dynamic time unit selection in reports, supporting Year and Quarter groupings.\ntable 'Time Unit'\n\tlineageTag: c100712f-ebcb-4d0f-a435-e1a4c8ce3229\n\n\t/// Display name for the time unit selection (Year, Quarter).\n\tcolumn 'Time Unit'\n\t\tlineageTag: bd5d5ab0-6269-4df0-bd39-9616f6e2b7d5\n\t\tsummarizeBy: none\n\t\tsourceColumn: [Value1]\n\t\tsortByColumn: 'Time Unit Order'\n\n\t\trelatedColumnDetails\n\t\t\tgroupByColumn: 'Time Unit Fields'\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// DAX column reference for the selected time unit.\n\tcolumn 'Time Unit Fields'\n\t\tisHidden\n\t\tlineageTag: 4be3ced6-f4db-40c4-84b2-5ef9decf7fee\n\t\tsummarizeBy: none\n\t\tsourceColumn: [Value2]\n\t\tsortByColumn: 'Time Unit Order'\n\n\t\textendedProperty ParameterMetadata =\n\t\t\t\t{\n\t\t\t\t  \"version\": 3,\n\t\t\t\t  \"kind\": 2\n\t\t\t\t}\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Sort order for time unit options in field parameter.\n\tcolumn 'Time Unit Order'\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 36566b71-2a4f-4ddb-86c0-b4c7b3d8ab7f\n\t\tsummarizeBy: sum\n\t\tsourceColumn: [Value3]\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition 'Time Unit' = calculated\n\t\tmode: import\n\t\tsource =\n\t\t\t\t{\n\t\t\t\t    (\"Year\", NAMEOF('date_dim'[d_year]), 0),\n\t\t\t\t    (\"Quarter\", NAMEOF('date_dim'[d_quarter_name]), 1)\n\t\t\t\t}\n\n\tannotation PBI_Id = 706957f972f8497896950148d2d5f0af\n\n"},{"path":"definition/tables/catalog_page.tmdl","text":"/// Catalog page dimension containing information about catalog pages used in catalog sales campaigns.\ntable catalog_page\n\tlineageTag: b4e5474e-8b18-42b8-9a56-9c2416950a0a\n\tsourceLineageTag: [dbo].[catalog_page]\n\n\t/// Catalog page surrogate key - unique identifier for each catalog page.\n\tcolumn cp_catalog_page_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 71a9dba7-e582-461e-b8de-f37322137084\n\t\tsourceLineageTag: cp_catalog_page_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: cp_catalog_page_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type or category of the catalog page.\n\tcolumn cp_type\n\t\tdataType: string\n\t\tlineageTag: bdf42608-dc88-489d-90ce-24a87bc40378\n\t\tsourceLineageTag: cp_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: cp_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition catalog_page = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: catalog_page\n\t\t\tschemaName: tpcds_sf__SF___defaultf8\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/catalog_sales.tmdl","text":"/// Fact table containing catalog sales transactions with pricing, quantities, profits, and foreign keys to related dimensions.\ntable catalog_sales\n\tlineageTag: c38e1686-b3a0-473b-8604-8efe0410ea48\n\tsourceLineageTag: [dbo].[catalog_sales]\n\n\t/// Foreign key to date dimension for sale date.\n\tcolumn cs_sold_date_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: ac2ce865-caed-4039-bda1-a182c516cfdd\n\t\tsourceLineageTag: cs_sold_date_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_sold_date_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension for billing customer.\n\tcolumn cs_bill_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 310a5931-6249-4ad7-86b4-7aca5fbc39a9\n\t\tsourceLineageTag: cs_bill_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_bill_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension for billing address.\n\tcolumn cs_bill_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 51e6cbcf-e98d-4ea9-865e-a500419bec99\n\t\tsourceLineageTag: cs_bill_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_bill_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension for shipping customer.\n\tcolumn cs_ship_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4a433af2-d889-41d8-9c6f-1f05f94f0070\n\t\tsourceLineageTag: cs_ship_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension for shipping address.\n\tcolumn cs_ship_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 25b41390-6ff3-4f15-b0e6-f26a3da61ef5\n\t\tsourceLineageTag: cs_ship_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to catalog page dimension.\n\tcolumn cs_catalog_page_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: b01fc3a0-0a51-4660-a5d3-56d28ea386cd\n\t\tsourceLineageTag: cs_catalog_page_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_catalog_page_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to ship mode dimension.\n\tcolumn cs_ship_mode_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 0a85820a-e8f9-41f9-b50e-9610890408e8\n\t\tsourceLineageTag: cs_ship_mode_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_mode_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension.\n\tcolumn cs_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9024dff9-9629-4f0e-abae-0b29f00d568d\n\t\tsourceLineageTag: cs_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to promotion dimension.\n\tcolumn cs_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: dbfb7b81-9eef-4b86-aac0-a48cc36d0168\n\t\tsourceLineageTag: cs_promo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Catalog order number for this transaction.\n\tcolumn cs_order_number\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: ccec0342-6144-411f-bc2d-90234bcfb5ac\n\t\tsourceLineageTag: cs_order_number\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_order_number\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quantity of items ordered in this transaction.\n\tcolumn cs_quantity\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 0f5abf99-45b4-495b-bad3-795d0e8e5802\n\t\tsourceLineageTag: cs_quantity\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_quantity\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Wholesale cost per unit for this transaction.\n\tcolumn cs_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 37d760e7-f755-42e9-9dba-3a28cc211ae7\n\t\tsourceLineageTag: cs_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// List price per unit at time of order.\n\tcolumn cs_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 0eda1627-f565-444d-92e5-1a5bb0929e4d\n\t\tsourceLineageTag: cs_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Actual sales price per unit (after discounts).\n\tcolumn cs_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: eebce86e-da48-4799-8f93-1a8e69de6605\n\t\tsourceLineageTag: cs_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended sales price (sales price \u00d7 quantity).\n\tcolumn cs_ext_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: f0c74fde-bf40-47e4-9352-6bc4d4f219a0\n\t\tsourceLineageTag: cs_ext_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_ext_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended list price (list price \u00d7 quantity).\n\tcolumn cs_ext_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 845b8a06-567d-49c0-9455-1a469d81f646\n\t\tsourceLineageTag: cs_ext_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_ext_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Net profit for this transaction (sales price - wholesale cost).\n\tcolumn cs_net_profit\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 341a4a24-e726-4b20-8a50-967f689a5d01\n\t\tsourceLineageTag: cs_net_profit\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_net_profit\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Technical column used for cache invalidation in DirectLake mode.\n\tcolumn cache_buster\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 33947ffc-dc12-42ab-af62-5b00c42adf7a\n\t\tsourceLineageTag: cache_buster\n\t\tsummarizeBy: none\n\t\tsourceColumn: cache_buster\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition catalog_sales = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: catalog_sales\n\t\t\tschemaName: tpcds_sf__SF___defaultf8\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/customer_address.tmdl","text":"/// Customer address dimension containing geographic information for billing and shipping addresses.\ntable customer_address\n\tlineageTag: 75246b34-d9f4-46fd-813f-bc5f0e47ab8c\n\tsourceLineageTag: [dbo].[customer_address]\n\n\t/// Customer address surrogate key - unique identifier for each address.\n\tcolumn ca_address_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9084b5d7-d188-4fb9-9c92-211d5b68f397\n\t\tsourceLineageTag: ca_address_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_address_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// City name for the customer address.\n\tcolumn ca_city\n\t\tdataType: string\n\t\tlineageTag: 03b2fb94-a08c-4010-9157-b91edcb60daf\n\t\tsourceLineageTag: ca_city\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_city\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// County name for the customer address.\n\tcolumn ca_county\n\t\tdataType: string\n\t\tlineageTag: 0ef7c0d6-1e17-42c5-a9bb-fb014c91398a\n\t\tsourceLineageTag: ca_county\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_county\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// State or province for the customer address.\n\tcolumn ca_state\n\t\tdataType: string\n\t\tlineageTag: 24c6ee20-e4ab-4ca3-aa30-934ba66a55db\n\t\tsourceLineageTag: ca_state\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_state\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Postal code for the customer address.\n\tcolumn ca_zip\n\t\tdataType: string\n\t\tlineageTag: a57bf4fe-e3de-4848-81bd-e71dd885c99f\n\t\tsourceLineageTag: ca_zip\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_zip\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type of location (e.g., residential, commercial).\n\tcolumn ca_location_type\n\t\tdataType: string\n\t\tlineageTag: 9a90fb17-4314-40df-84ae-071cdd4eb3c7\n\t\tsourceLineageTag: ca_location_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_location_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition customer_address = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: customer_address\n\t\t\tschemaName: tpcds_sf__SF___defaultf8\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/customer_demographics.tmdl","text":"/// Customer demographics dimension containing education status and marital status attributes for customer segmentation.\ntable customer_demographics\n\tlineageTag: 52f8f268-90f3-4f40-a527-07a74d9e9baf\n\tsourceLineageTag: [dbo].[customer_demographics]\n\n\t/// Customer demographics surrogate key - unique identifier for each demographic profile.\n\tcolumn cd_demo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: d2cad4d0-4a88-41ab-b7ee-94ab73973694\n\t\tsourceLineageTag: cd_demo_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_demo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Marital status of the customer.\n\tcolumn cd_marital_status\n\t\tdataType: string\n\t\tlineageTag: e29a3681-2c0d-4579-9b35-5112d86c0f46\n\t\tsourceLineageTag: cd_marital_status\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_marital_status\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Education level of the customer.\n\tcolumn cd_education_status\n\t\tdataType: string\n\t\tlineageTag: 43260b21-0847-455f-a853-e2233af2a62e\n\t\tsourceLineageTag: cd_education_status\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_education_status\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition customer_demographics = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: customer_demographics\n\t\t\tschemaName: tpcds_sf__SF___defaultf8\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/date_dim.tmdl","text":"/// Date dimension providing calendar hierarchy with year, quarter, month, and day-of-week attributes for time-based analysis.\ntable date_dim\n\tlineageTag: 9e7ec2de-382c-4a3b-9589-1ece445bfa27\n\tsourceLineageTag: [dbo].[date_dim]\n\n\t/// Date surrogate key - unique identifier for each calendar date.\n\tcolumn d_date_sk_1\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 63841738-6970-46ac-8106-970384b1052b\n\t\tsourceLineageTag: d_date_sk_1\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_date_sk_1\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Calendar date value.\n\tcolumn d_date\n\t\tdataType: dateTime\n\t\tformatString: General Date\n\t\tlineageTag: dc08a5e6-1a30-4d5f-9b03-1c98c7bac892\n\t\tsourceLineageTag: d_date\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_date\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Calendar year (e.g., 2023).\n\tcolumn d_year\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: fcf4b9e9-c880-4ee0-907c-3a18951b0809\n\t\tsourceLineageTag: d_year\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_year\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Day of week (1=Sunday, 7=Saturday).\n\tcolumn d_dow\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 339ac74d-ab31-4d4e-93eb-524b9f7e211c\n\t\tsourceLineageTag: d_dow\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_dow\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Month of year (1-12).\n\tcolumn d_moy\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: faa3deb0-9def-4334-bd42-a1b9d60b51b7\n\t\tsourceLineageTag: d_moy\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_moy\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Day of month (1-31).\n\tcolumn d_dom\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 83cd6ebd-7811-4616-a884-9930f66f8bfe\n\t\tsourceLineageTag: d_dom\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_dom\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quarter of year (1-4).\n\tcolumn d_qoy\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 4962ee0a-fcb9-4bf8-a5c7-43a66a94e67f\n\t\tsourceLineageTag: d_qoy\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_qoy\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quarter name (e.g., 'Q1 2023').\n\tcolumn d_quarter_name\n\t\tdataType: string\n\t\tlineageTag: 2426fa03-107c-4f51-a0b4-c59bb4b85d4f\n\t\tsourceLineageTag: d_quarter_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_quarter_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition date_dim = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: date_dim\n\t\t\tschemaName: tpcds_sf__SF___defaultf8\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n\tcalendar tpcds_calendar\n\t\tlineageTag: dd529b7a-99fe-4cb5-928c-df1abed499f1\n\n\t\tcalendarColumnGroup = year\n\t\t\tprimaryColumn: d_year\n\n\t\tcalendarColumnGroup = quarter\n\t\t\tprimaryColumn: d_quarter_name\n\n\t\tcalendarColumnGroup = quarterOfYear\n\t\t\tprimaryColumn: d_qoy\n\n\t\tcalendarColumnGroup = date\n\t\t\tprimaryColumn: d_date\n\n"},{"path":"definition/tables/item.tmdl","text":"/// Product dimension containing item details such as brand, category, class, color, pricing, and manufacturing information.\ntable item\n\tlineageTag: 0f958ae1-af31-4b29-87a3-7e9ee589887e\n\tsourceLineageTag: [dbo].[item]\n\n\t/// Item surrogate key - unique identifier for each product.\n\tcolumn i_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 23893369-a589-4284-afa3-b1e291864600\n\t\tsourceLineageTag: i_item_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Current retail price of the product.\n\tcolumn i_current_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 70b27235-eefb-4732-8e0d-64c9576f0776\n\t\tsourceLineageTag: i_current_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_current_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Wholesale cost paid for the product.\n\tcolumn i_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 90cbdd03-df2f-4439-941d-de295ceb5135\n\t\tsourceLineageTag: i_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Brand name of the product.\n\tcolumn i_brand\n\t\tdataType: string\n\t\tlineageTag: 0c701039-61ca-4f5f-872b-aab6677ef45c\n\t\tsourceLineageTag: i_brand\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_brand\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Product class or subcategory within the main category.\n\tcolumn i_class\n\t\tdataType: string\n\t\tlineageTag: 5bdf127c-d558-46b2-98e6-0e294148d466\n\t\tsourceLineageTag: i_class\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_class\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Product category classification.\n\tcolumn i_category\n\t\tdataType: string\n\t\tlineageTag: 138de914-55e9-4a46-b5b9-6707b88a38e9\n\t\tsourceLineageTag: i_category\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_category\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Manufacturer or producer of the product.\n\tcolumn i_manufact\n\t\tdataType: string\n\t\tlineageTag: e6852c2b-2bc8-4fb1-86cb-d7ebe7c18637\n\t\tsourceLineageTag: i_manufact\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_manufact\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Size specification of the product.\n\tcolumn i_size\n\t\tdataType: string\n\t\tlineageTag: 0aeb0963-bc02-4389-a3fd-ab7232caaf9c\n\t\tsourceLineageTag: i_size\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_size\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Primary color of the product.\n\tcolumn i_color\n\t\tdataType: string\n\t\tlineageTag: d6709cb2-852f-485a-a0de-8cdd07941038\n\t\tsourceLineageTag: i_color\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_color\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition item = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: item\n\t\t\tschemaName: tpcds_sf__SF___defaultf8\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/promotion.tmdl","text":"/// Promotion dimension containing promotional campaign details including costs and promotional names.\ntable promotion\n\tlineageTag: 5942c68a-d07a-4a23-8864-b2c733be46c8\n\tsourceLineageTag: [dbo].[promotion]\n\n\t/// Promotion surrogate key - unique identifier for each promotional campaign.\n\tcolumn p_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 0299ca0a-f58f-4ebe-b752-ffa6b63f9760\n\t\tsourceLineageTag: p_promo_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension (for item-specific promotions).\n\tcolumn p_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 92f8bc42-df79-4a35-8a37-10a02a937c40\n\t\tsourceLineageTag: p_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: p_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Cost associated with running this promotional campaign.\n\tcolumn p_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: c42871df-1b39-4b1e-83cb-142e6864a1e0\n\t\tsourceLineageTag: p_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Name or title of the promotional campaign.\n\tcolumn p_promo_name\n\t\tdataType: string\n\t\tlineageTag: 6ceed2f6-4e12-4edc-b161-8251a356ec17\n\t\tsourceLineageTag: p_promo_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_promo_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition promotion = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: promotion\n\t\t\tschemaName: tpcds_sf__SF___defaultf8\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/ship_mode.tmdl","text":"/// Shipping method dimension containing carrier information and shipping type classifications.\ntable ship_mode\n\tlineageTag: db1b87b5-f584-4f30-b70e-0eb00c8b45b8\n\tsourceLineageTag: [dbo].[ship_mode]\n\n\t/// Ship mode surrogate key - unique identifier for each shipping method.\n\tcolumn sm_ship_mode_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4c485c32-fa22-4df6-b4a8-a8df04e2f730\n\t\tsourceLineageTag: sm_ship_mode_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_ship_mode_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type or category of shipping method.\n\tcolumn sm_type\n\t\tdataType: string\n\t\tlineageTag: c49fc6f8-d030-49a2-8105-6a45d1a81ac2\n\t\tsourceLineageTag: sm_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Internal code for the shipping method.\n\tcolumn sm_code\n\t\tdataType: string\n\t\tlineageTag: f54a092d-e847-4cee-84e6-5b813cbf5280\n\t\tsourceLineageTag: sm_code\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_code\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Shipping carrier or company name.\n\tcolumn sm_carrier\n\t\tdataType: string\n\t\tlineageTag: aca70983-6b51-4291-83f8-77fac2e381fe\n\t\tsourceLineageTag: sm_carrier\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_carrier\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition ship_mode = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: ship_mode\n\t\t\tschemaName: tpcds_sf__SF___defaultf8\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/store.tmdl","text":"/// Store dimension containing retail location information including geographic details, management, and tax rates.\ntable store\n\tlineageTag: 6835d4a1-f40a-40aa-8a65-98aa4e06a406\n\tsourceLineageTag: [dbo].[store]\n\n\t/// Store surrogate key - unique identifier for each store location.\n\tcolumn s_store_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 14502476-8801-4068-b7c4-875ce7cac939\n\t\tsourceLineageTag: s_store_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_store_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Store name or identifier for the retail location.\n\tcolumn s_store_name\n\t\tdataType: string\n\t\tlineageTag: 4e78d335-4974-4d41-8402-ea89f63f9005\n\t\tsourceLineageTag: s_store_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_store_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Name of the store manager.\n\tcolumn s_manager\n\t\tdataType: string\n\t\tlineageTag: 3a8ddb5b-2ada-42d0-9ef9-89cd96b435b6\n\t\tsourceLineageTag: s_manager\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_manager\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Name of the market manager overseeing this store.\n\tcolumn s_market_manager\n\t\tdataType: string\n\t\tlineageTag: 37c8562d-3ab6-47ad-9b99-2ad1d72e6cd6\n\t\tsourceLineageTag: s_market_manager\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_market_manager\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// City where the store is located.\n\tcolumn s_city\n\t\tdataType: string\n\t\tlineageTag: a1a9209c-8194-4f96-8e94-69e545e064ba\n\t\tsourceLineageTag: s_city\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_city\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// County where the store is located.\n\tcolumn s_county\n\t\tdataType: string\n\t\tlineageTag: e219113c-fe81-45b0-b6be-f02e7ce75403\n\t\tsourceLineageTag: s_county\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_county\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// State or province where the store is located.\n\tcolumn s_state\n\t\tdataType: string\n\t\tlineageTag: fbaeba14-56ed-496d-a004-83f851771750\n\t\tsourceLineageTag: s_state\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_state\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Postal code for the store location.\n\tcolumn s_zip\n\t\tdataType: string\n\t\tlineageTag: 44e7ea79-56d6-4d20-8243-92c35919dcc3\n\t\tsourceLineageTag: s_zip\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_zip\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Tax rate percentage applied at this store location.\n\tcolumn s_tax_percentage\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: ba6d903d-f426-46b7-90a8-756a18261f2a\n\t\tsourceLineageTag: s_tax_percentage\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_tax_percentage\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\tpartition store = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: store\n\t\t\tschemaName: tpcds_sf__SF___defaultf8\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/store_sales.tmdl","text":"/// Fact table containing retail store sales transactions with pricing, quantities, profits, and foreign keys to related dimensions.\ntable store_sales\n\tlineageTag: 6c7a2caf-3c22-4f38-a6aa-895a77e2e1c9\n\tsourceLineageTag: [dbo].[store_sales]\n\n\t/// Foreign key to date dimension for sale date.\n\tcolumn ss_sold_date_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4e30d94a-216f-4eaf-9858-fb22c6e6d57e\n\t\tsourceLineageTag: ss_sold_date_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_sold_date_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension.\n\tcolumn ss_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 2b151014-7df5-45f0-9907-03cbe191952b\n\t\tsourceLineageTag: ss_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Customer surrogate key for the transaction.\n\tcolumn ss_customer_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 558da56d-ea73-4562-b35c-ff94e774cca3\n\t\tsourceLineageTag: ss_customer_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_customer_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension.\n\tcolumn ss_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: e65f473f-bb26-4c7d-90f1-139de12a51ab\n\t\tsourceLineageTag: ss_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension.\n\tcolumn ss_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: cb65f3a1-6240-4baa-bfb2-4b24d39e2e87\n\t\tsourceLineageTag: ss_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to store dimension.\n\tcolumn ss_store_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 809a4a04-3bbf-4067-b54f-ee4d5fbbf6d4\n\t\tsourceLineageTag: ss_store_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_store_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to promotion dimension.\n\tcolumn ss_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9a64bbfb-0ae6-4919-b4e9-73e6d90353b3\n\t\tsourceLineageTag: ss_promo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Transaction ticket or receipt number.\n\tcolumn ss_ticket_number\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: d8ce37f4-a12c-4558-b131-66631546b607\n\t\tsourceLineageTag: ss_ticket_number\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ticket_number\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quantity of items sold in this transaction.\n\tcolumn ss_quantity\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 318e6a7b-88a2-413c-8257-315f66313d79\n\t\tsourceLineageTag: ss_quantity\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_quantity\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Wholesale cost per unit for this transaction.\n\tcolumn ss_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: d02100cc-2804-43d7-b752-8e98d579e202\n\t\tsourceLineageTag: ss_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// List price per unit at time of sale.\n\tcolumn ss_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 4f32f293-c19e-497e-906e-089bc204cde9\n\t\tsourceLineageTag: ss_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Actual sales price per unit (after discounts).\n\tcolumn ss_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: e620a719-f6e0-4965-8326-f4a2d255999b\n\t\tsourceLineageTag: ss_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended sales price (sales price \u00d7 quantity).\n\tcolumn ss_ext_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 8b66819a-a151-4fc7-9ec0-cacf7b6e8e01\n\t\tsourceLineageTag: ss_ext_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ext_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended list price (list price \u00d7 quantity).\n\tcolumn ss_ext_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: bc8e9c57-98e2-4032-a3fd-69808e3cdd8b\n\t\tsourceLineageTag: ss_ext_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ext_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Net profit for this transaction (sales price - wholesale cost).\n\tcolumn ss_net_profit\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: d2d6233f-6944-40b5-97d7-2870da393d1b\n\t\tsourceLineageTag: ss_net_profit\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_net_profit\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Technical column used for cache invalidation in DirectLake mode.\n\tcolumn cache_buster\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 7ae60894-bf63-4fef-a327-bf8796466374\n\t\tsourceLineageTag: cache_buster\n\t\tsummarizeBy: none\n\t\tsourceColumn: cache_buster\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition store_sales = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: store_sales\n\t\t\tschemaName: tpcds_sf__SF___defaultf8\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition.pbism","text":"{\"$schema\": \"https://developer.microsoft.com/json-schemas/fabric/item/semanticModel/definitionProperties/1.0.0/schema.json\", \"version\": \"5.0\", \"settings\": {}}"}]},"default2rg":{"model":"tpcds_sf__SF___default2rg","parts":[{"path":"definition/database.tmdl","text":"database tpcds_sf__SF___default2rg\n\tcompatibilityLevel: 1702\n\tcompatibilityMode: powerBI\n\tlanguage: 1033\n\n"},{"path":"definition/expressions.tmdl","text":"expression 'DirectLake - tpcds_sf__SF__' =\n\t\tlet\n\t\t    Source = AzureStorage.DataLake(\"https://onelake.dfs.fabric.microsoft.com/51650f82-6bb5-4023-b0ab-db197d32e0be/01b539f3-4a9d-45ef-b1ef-0ba59552eb21\", [HierarchicalNavigation=true])\n\t\tin\n\t\t    Source\n\tlineageTag: c6e3390d-cc84-4e13-ab93-53308a346336\n\n\tannotation PBI_IncludeFutureArtifacts = False\n\n"},{"path":"definition/model.tmdl","text":"model Model\n\tdirectLakeBehavior: directLakeOnly\n\tculture: en-US\n\tdefaultPowerBIDataSourceVersion: powerBI_V3\n\tsourceQueryCulture: en-US\n\tdataAccessOptions\n\t\tlegacyRedirects\n\t\treturnErrorValuesAsNull\n\nannotation PBI_QueryOrder = [\"DirectLake - tpcds_sf__SF__\"]\n\nannotation __PBI_TimeIntelligenceEnabled = 1\n\nannotation __LastRPTime = 134272970581620514\n\nannotation PBI_ProTooling = [\"DirectLakeOnOneLakeInWeb\",\"WebModelingEdit\"]\n\nannotation __TEdtr = 1\n\nannotation TabularEditor_SerializeOptions = {\"IgnoreInferredObjects\":true,\"IgnoreInferredProperties\":true,\"IgnoreTimestamps\":true,\"SplitMultilineStrings\":true,\"PrefixFilenames\":false,\"LocalTranslations\":true,\"LocalPerspectives\":true,\"LocalRelationships\":true,\"Levels\":[\"Data Sources\",\"Perspectives\",\"Relationships\",\"Roles\",\"Shared Expressions\",\"Tables\",\"Tables/Calculation Items\",\"Tables/Columns\",\"Tables/Hierarchies\",\"Tables/Measures\",\"Tables/Partitions\",\"Translations\"]}\n\nref table store\nref table item\nref table date_dim\nref table store_sales\nref table catalog_page\nref table promotion\nref table ship_mode\nref table catalog_sales\nref table customer_address\nref table customer_demographics\nref table 'Measures 1'\nref table 'Time Unit'\n\n"},{"path":"definition/relationships.tmdl","text":"relationship b3bd8d91-ae6b-4489-83cd-34ca2c3213d8\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_bill_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship b727abac-57ca-484a-8183-1b4c10921d84\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_bill_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship 41e21761-05b3-410b-960e-54805fb2c4f2\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_catalog_page_sk\n\ttoColumn: catalog_page.cp_catalog_page_sk\n\nrelationship c7ce7218-d702-40d6-94b4-80009e453fe9\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship 4d02a5f5-e67a-4f65-a203-192e1c2ef166\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_promo_sk\n\ttoColumn: promotion.p_promo_sk\n\nrelationship 545e95f8-e63b-4a27-9c91-3cd725ca10e0\n\tisActive: false\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship 685424a1-f6c7-42d6-8f44-3cccf5f08db4\n\tisActive: false\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship 2e0c0f18-79bf-48f8-be1a-2c53cc33e257\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_mode_sk\n\ttoColumn: ship_mode.sm_ship_mode_sk\n\nrelationship c5edc5f8-f418-44a4-a8aa-caff6e0bef29\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_sold_date_sk\n\ttoColumn: date_dim.d_date_sk_1\n\nrelationship 9f8c8977-b8ac-4985-aed9-7711ce5004af\n\tisActive: false\n\tfromColumn: promotion.p_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship f824425a-7dd8-4a70-9ee5-2042a74ed97d\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship be9c6d5e-0bed-4db1-8558-326d5dfba951\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship f172b575-b3ce-48a4-b264-2928516104a2\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship 95b347fa-cf7e-4885-8124-86eb2ff56351\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_promo_sk\n\ttoColumn: promotion.p_promo_sk\n\nrelationship c782badb-be54-4423-98b0-22eebad6aa29\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_sold_date_sk\n\ttoColumn: date_dim.d_date_sk_1\n\nrelationship 9e5ca2d3-a5aa-4d43-b00a-07e4a857b1e4\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_store_sk\n\ttoColumn: store.s_store_sk\n\n"},{"path":"definition/tables/Measures 1.tmdl","text":"/// Calculated measures table containing key business metrics for revenue, quantity, profit, tax, and performance analysis across store and catalog channels.\ntable 'Measures 1'\n\tlineageTag: ae7cf165-f530-4f98-a4c5-befd958ff771\n\n\t/// Revenue from catalog sales channel only, based on extended sales price\n\tmeasure 'Catalog Revenue' = SUM('catalog_sales'[cs_ext_sales_price])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 01. Revenue\n\t\tlineageTag: bed34121-c406-4e25-b931-6ea438aede2a\n\n\t/// Revenue from store sales channel only, based on extended sales price\n\tmeasure 'Store Revenue' = SUM('store_sales'[ss_ext_sales_price])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 01. Revenue\n\t\tlineageTag: 1755e841-2eeb-4e1e-9a98-c860eb583a79\n\n\t/// Total units sold through catalog sales channel\n\tmeasure 'Catalog Sales Quantity' = SUM('catalog_sales'[cs_quantity])\n\t\tformatString: #,0\n\t\tdisplayFolder: 02. Quantity\n\t\tlineageTag: 6da96e65-1eb2-4e76-afdb-50d9106f56c1\n\n\t/// Net profit from store sales channel only\n\tmeasure 'Store Net Profit' = SUM('store_sales'[ss_net_profit])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 03. Profit\n\t\tlineageTag: a5a04305-c986-4791-925e-b9115939e453\n\n\t/// Number of unique customers who made store purchases\n\tmeasure 'Store Distinct Customers' = DISTINCTCOUNT('store_sales'[ss_customer_sk])\n\t\tformatString: #,0\n\t\tdisplayFolder: 04. Distinct Counts\n\t\tlineageTag: ace0485f-176a-4401-bb53-41c99ce5c356\n\n\t/// Revenue for the same period in the previous year\n\tmeasure 'Store Revenue Same Period LY' =\n\t\t\t\n\t\t\tCALCULATE(\n\t\t\t    [Store Revenue],\n\t\t\t    SAMEPERIODLASTYEAR('tpcds_calendar')\n\t\t\t)\n\t\tformatString: $#,0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 303dbdc1-0484-496c-b14f-a41b0ef53f82\n\n\t/// Revenue for the same period in the previous year\n\tmeasure 'Store Revenue YoY' =\n\t\t\tVAR CurrentYearRev = [Store Revenue]\n\t\t\tVAR PreviousYearRev = [Store Revenue Same Period LY]\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentYearRev - PreviousYearRev, PreviousYearRev, 0)\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 39648465-7aac-4652-8c2f-d6599df15204\n\n\t/// Catalog Sales Same Period LY\n\tmeasure 'Catalog Sales Same Period LY' =\n\t\t\t\n\t\t\tCALCULATE(\n\t\t\t    [Catalog Sales Quantity],\n\t\t\t    SAMEPERIODLASTYEAR('tpcds_calendar')\n\t\t\t)\n\t\tformatString: 0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: d1d5179d-688b-487c-8638-81780d95eb05\n\n\t/// Catalog Sales YoY\n\tmeasure 'Catalog Sales YoY' =\n\t\t\tVAR CurrentYearCatSales = [Catalog Sales Quantity]\n\t\t\tVAR PreviousYearCatSales = [Catalog Sales Same Period LY]\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentYearCatSales - PreviousYearCatSales, PreviousYearCatSales, 0)\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 0bc09d6d-1bdd-4f7e-ac26-a12e87ccf67d\n\n\t/// Store Profit % by Item Category\n\tmeasure 'Store Profit % by Item Category' = ```\n\t\t\t\n\t\t\tVAR CurrentProfit = [Store Net Profit]\n\t\t\tVAR TotalProfitAllCategories = \n\t\t\t    CALCULATE(\n\t\t\t        [Store Net Profit],\n\t\t\t        ALLEXCEPT(\n\t\t\t            'item',\n\t\t\t            'item'[i_brand],\n\t\t\t            'item'[i_manufact]\n\t\t\t        )\n\t\t\t    )\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentProfit, TotalProfitAllCategories, 0)\n\t\t\t```\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 05. Advanced % Share\n\t\tlineageTag: 04f417ef-ba48-4d9e-873d-76d516ec3a54\n\n\t/// Total revenue from beginning of year to current date selection\n\tmeasure 'Store Revenue YTD' = TOTALYTD([Store Revenue],'tpcds_calendar')\n\t\tformatString: $#,0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 0e0dbb36-1e75-4c77-8b7f-9a462a143dcb\n\n\t/// Dummy\n\tcolumn Dummy\n\t\tformatString: 0\n\t\tlineageTag: c1dfb904-84a8-4dfc-a768-d5e10f70f255\n\t\tsummarizeBy: none\n\t\tisNameInferred\n\t\tsourceColumn: [Dummy]\n\n\tpartition 'Measures 1' = calculated\n\t\tmode: import\n\t\tsource = ROW(\"Dummy\", 1)\n\n"},{"path":"definition/tables/Time Unit.tmdl","text":"/// Field parameter table for dynamic time unit selection in reports, supporting Year and Quarter groupings.\ntable 'Time Unit'\n\tlineageTag: c100712f-ebcb-4d0f-a435-e1a4c8ce3229\n\n\t/// Display name for the time unit selection (Year, Quarter).\n\tcolumn 'Time Unit'\n\t\tlineageTag: bd5d5ab0-6269-4df0-bd39-9616f6e2b7d5\n\t\tsummarizeBy: none\n\t\tsourceColumn: [Value1]\n\t\tsortByColumn: 'Time Unit Order'\n\n\t\trelatedColumnDetails\n\t\t\tgroupByColumn: 'Time Unit Fields'\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// DAX column reference for the selected time unit.\n\tcolumn 'Time Unit Fields'\n\t\tisHidden\n\t\tlineageTag: 4be3ced6-f4db-40c4-84b2-5ef9decf7fee\n\t\tsummarizeBy: none\n\t\tsourceColumn: [Value2]\n\t\tsortByColumn: 'Time Unit Order'\n\n\t\textendedProperty ParameterMetadata =\n\t\t\t\t{\n\t\t\t\t  \"version\": 3,\n\t\t\t\t  \"kind\": 2\n\t\t\t\t}\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Sort order for time unit options in field parameter.\n\tcolumn 'Time Unit Order'\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 36566b71-2a4f-4ddb-86c0-b4c7b3d8ab7f\n\t\tsummarizeBy: sum\n\t\tsourceColumn: [Value3]\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition 'Time Unit' = calculated\n\t\tmode: import\n\t\tsource =\n\t\t\t\t{\n\t\t\t\t    (\"Year\", NAMEOF('date_dim'[d_year]), 0),\n\t\t\t\t    (\"Quarter\", NAMEOF('date_dim'[d_quarter_name]), 1)\n\t\t\t\t}\n\n\tannotation PBI_Id = 706957f972f8497896950148d2d5f0af\n\n"},{"path":"definition/tables/catalog_page.tmdl","text":"/// Catalog page dimension containing information about catalog pages used in catalog sales campaigns.\ntable catalog_page\n\tlineageTag: b4e5474e-8b18-42b8-9a56-9c2416950a0a\n\tsourceLineageTag: [dbo].[catalog_page]\n\n\t/// Catalog page surrogate key - unique identifier for each catalog page.\n\tcolumn cp_catalog_page_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 71a9dba7-e582-461e-b8de-f37322137084\n\t\tsourceLineageTag: cp_catalog_page_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: cp_catalog_page_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type or category of the catalog page.\n\tcolumn cp_type\n\t\tdataType: string\n\t\tlineageTag: bdf42608-dc88-489d-90ce-24a87bc40378\n\t\tsourceLineageTag: cp_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: cp_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition catalog_page = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: catalog_page\n\t\t\tschemaName: tpcds_sf__SF___default2rg\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/catalog_sales.tmdl","text":"/// Fact table containing catalog sales transactions with pricing, quantities, profits, and foreign keys to related dimensions.\ntable catalog_sales\n\tlineageTag: c38e1686-b3a0-473b-8604-8efe0410ea48\n\tsourceLineageTag: [dbo].[catalog_sales]\n\n\t/// Foreign key to date dimension for sale date.\n\tcolumn cs_sold_date_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: ac2ce865-caed-4039-bda1-a182c516cfdd\n\t\tsourceLineageTag: cs_sold_date_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_sold_date_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension for billing customer.\n\tcolumn cs_bill_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 310a5931-6249-4ad7-86b4-7aca5fbc39a9\n\t\tsourceLineageTag: cs_bill_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_bill_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension for billing address.\n\tcolumn cs_bill_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 51e6cbcf-e98d-4ea9-865e-a500419bec99\n\t\tsourceLineageTag: cs_bill_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_bill_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension for shipping customer.\n\tcolumn cs_ship_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4a433af2-d889-41d8-9c6f-1f05f94f0070\n\t\tsourceLineageTag: cs_ship_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension for shipping address.\n\tcolumn cs_ship_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 25b41390-6ff3-4f15-b0e6-f26a3da61ef5\n\t\tsourceLineageTag: cs_ship_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to catalog page dimension.\n\tcolumn cs_catalog_page_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: b01fc3a0-0a51-4660-a5d3-56d28ea386cd\n\t\tsourceLineageTag: cs_catalog_page_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_catalog_page_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to ship mode dimension.\n\tcolumn cs_ship_mode_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 0a85820a-e8f9-41f9-b50e-9610890408e8\n\t\tsourceLineageTag: cs_ship_mode_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_mode_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension.\n\tcolumn cs_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9024dff9-9629-4f0e-abae-0b29f00d568d\n\t\tsourceLineageTag: cs_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to promotion dimension.\n\tcolumn cs_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: dbfb7b81-9eef-4b86-aac0-a48cc36d0168\n\t\tsourceLineageTag: cs_promo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Catalog order number for this transaction.\n\tcolumn cs_order_number\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: ccec0342-6144-411f-bc2d-90234bcfb5ac\n\t\tsourceLineageTag: cs_order_number\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_order_number\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quantity of items ordered in this transaction.\n\tcolumn cs_quantity\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 0f5abf99-45b4-495b-bad3-795d0e8e5802\n\t\tsourceLineageTag: cs_quantity\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_quantity\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Wholesale cost per unit for this transaction.\n\tcolumn cs_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 37d760e7-f755-42e9-9dba-3a28cc211ae7\n\t\tsourceLineageTag: cs_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// List price per unit at time of order.\n\tcolumn cs_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 0eda1627-f565-444d-92e5-1a5bb0929e4d\n\t\tsourceLineageTag: cs_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Actual sales price per unit (after discounts).\n\tcolumn cs_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: eebce86e-da48-4799-8f93-1a8e69de6605\n\t\tsourceLineageTag: cs_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended sales price (sales price \u00d7 quantity).\n\tcolumn cs_ext_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: f0c74fde-bf40-47e4-9352-6bc4d4f219a0\n\t\tsourceLineageTag: cs_ext_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_ext_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended list price (list price \u00d7 quantity).\n\tcolumn cs_ext_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 845b8a06-567d-49c0-9455-1a469d81f646\n\t\tsourceLineageTag: cs_ext_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_ext_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Net profit for this transaction (sales price - wholesale cost).\n\tcolumn cs_net_profit\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 341a4a24-e726-4b20-8a50-967f689a5d01\n\t\tsourceLineageTag: cs_net_profit\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_net_profit\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Technical column used for cache invalidation in DirectLake mode.\n\tcolumn cache_buster\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 33947ffc-dc12-42ab-af62-5b00c42adf7a\n\t\tsourceLineageTag: cache_buster\n\t\tsummarizeBy: none\n\t\tsourceColumn: cache_buster\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition catalog_sales = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: catalog_sales\n\t\t\tschemaName: tpcds_sf__SF___default2rg\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/customer_address.tmdl","text":"/// Customer address dimension containing geographic information for billing and shipping addresses.\ntable customer_address\n\tlineageTag: 75246b34-d9f4-46fd-813f-bc5f0e47ab8c\n\tsourceLineageTag: [dbo].[customer_address]\n\n\t/// Customer address surrogate key - unique identifier for each address.\n\tcolumn ca_address_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9084b5d7-d188-4fb9-9c92-211d5b68f397\n\t\tsourceLineageTag: ca_address_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_address_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// City name for the customer address.\n\tcolumn ca_city\n\t\tdataType: string\n\t\tlineageTag: 03b2fb94-a08c-4010-9157-b91edcb60daf\n\t\tsourceLineageTag: ca_city\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_city\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// County name for the customer address.\n\tcolumn ca_county\n\t\tdataType: string\n\t\tlineageTag: 0ef7c0d6-1e17-42c5-a9bb-fb014c91398a\n\t\tsourceLineageTag: ca_county\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_county\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// State or province for the customer address.\n\tcolumn ca_state\n\t\tdataType: string\n\t\tlineageTag: 24c6ee20-e4ab-4ca3-aa30-934ba66a55db\n\t\tsourceLineageTag: ca_state\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_state\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Postal code for the customer address.\n\tcolumn ca_zip\n\t\tdataType: string\n\t\tlineageTag: a57bf4fe-e3de-4848-81bd-e71dd885c99f\n\t\tsourceLineageTag: ca_zip\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_zip\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type of location (e.g., residential, commercial).\n\tcolumn ca_location_type\n\t\tdataType: string\n\t\tlineageTag: 9a90fb17-4314-40df-84ae-071cdd4eb3c7\n\t\tsourceLineageTag: ca_location_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_location_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition customer_address = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: customer_address\n\t\t\tschemaName: tpcds_sf__SF___default2rg\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/customer_demographics.tmdl","text":"/// Customer demographics dimension containing education status and marital status attributes for customer segmentation.\ntable customer_demographics\n\tlineageTag: 52f8f268-90f3-4f40-a527-07a74d9e9baf\n\tsourceLineageTag: [dbo].[customer_demographics]\n\n\t/// Customer demographics surrogate key - unique identifier for each demographic profile.\n\tcolumn cd_demo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: d2cad4d0-4a88-41ab-b7ee-94ab73973694\n\t\tsourceLineageTag: cd_demo_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_demo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Marital status of the customer.\n\tcolumn cd_marital_status\n\t\tdataType: string\n\t\tlineageTag: e29a3681-2c0d-4579-9b35-5112d86c0f46\n\t\tsourceLineageTag: cd_marital_status\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_marital_status\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Education level of the customer.\n\tcolumn cd_education_status\n\t\tdataType: string\n\t\tlineageTag: 43260b21-0847-455f-a853-e2233af2a62e\n\t\tsourceLineageTag: cd_education_status\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_education_status\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition customer_demographics = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: customer_demographics\n\t\t\tschemaName: tpcds_sf__SF___default2rg\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/date_dim.tmdl","text":"/// Date dimension providing calendar hierarchy with year, quarter, month, and day-of-week attributes for time-based analysis.\ntable date_dim\n\tlineageTag: 9e7ec2de-382c-4a3b-9589-1ece445bfa27\n\tsourceLineageTag: [dbo].[date_dim]\n\n\t/// Date surrogate key - unique identifier for each calendar date.\n\tcolumn d_date_sk_1\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 63841738-6970-46ac-8106-970384b1052b\n\t\tsourceLineageTag: d_date_sk_1\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_date_sk_1\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Calendar date value.\n\tcolumn d_date\n\t\tdataType: dateTime\n\t\tformatString: General Date\n\t\tlineageTag: dc08a5e6-1a30-4d5f-9b03-1c98c7bac892\n\t\tsourceLineageTag: d_date\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_date\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Calendar year (e.g., 2023).\n\tcolumn d_year\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: fcf4b9e9-c880-4ee0-907c-3a18951b0809\n\t\tsourceLineageTag: d_year\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_year\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Day of week (1=Sunday, 7=Saturday).\n\tcolumn d_dow\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 339ac74d-ab31-4d4e-93eb-524b9f7e211c\n\t\tsourceLineageTag: d_dow\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_dow\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Month of year (1-12).\n\tcolumn d_moy\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: faa3deb0-9def-4334-bd42-a1b9d60b51b7\n\t\tsourceLineageTag: d_moy\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_moy\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Day of month (1-31).\n\tcolumn d_dom\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 83cd6ebd-7811-4616-a884-9930f66f8bfe\n\t\tsourceLineageTag: d_dom\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_dom\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quarter of year (1-4).\n\tcolumn d_qoy\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 4962ee0a-fcb9-4bf8-a5c7-43a66a94e67f\n\t\tsourceLineageTag: d_qoy\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_qoy\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quarter name (e.g., 'Q1 2023').\n\tcolumn d_quarter_name\n\t\tdataType: string\n\t\tlineageTag: 2426fa03-107c-4f51-a0b4-c59bb4b85d4f\n\t\tsourceLineageTag: d_quarter_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_quarter_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition date_dim = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: date_dim\n\t\t\tschemaName: tpcds_sf__SF___default2rg\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n\tcalendar tpcds_calendar\n\t\tlineageTag: dd529b7a-99fe-4cb5-928c-df1abed499f1\n\n\t\tcalendarColumnGroup = year\n\t\t\tprimaryColumn: d_year\n\n\t\tcalendarColumnGroup = quarter\n\t\t\tprimaryColumn: d_quarter_name\n\n\t\tcalendarColumnGroup = quarterOfYear\n\t\t\tprimaryColumn: d_qoy\n\n\t\tcalendarColumnGroup = date\n\t\t\tprimaryColumn: d_date\n\n"},{"path":"definition/tables/item.tmdl","text":"/// Product dimension containing item details such as brand, category, class, color, pricing, and manufacturing information.\ntable item\n\tlineageTag: 0f958ae1-af31-4b29-87a3-7e9ee589887e\n\tsourceLineageTag: [dbo].[item]\n\n\t/// Item surrogate key - unique identifier for each product.\n\tcolumn i_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 23893369-a589-4284-afa3-b1e291864600\n\t\tsourceLineageTag: i_item_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Current retail price of the product.\n\tcolumn i_current_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 70b27235-eefb-4732-8e0d-64c9576f0776\n\t\tsourceLineageTag: i_current_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_current_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Wholesale cost paid for the product.\n\tcolumn i_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 90cbdd03-df2f-4439-941d-de295ceb5135\n\t\tsourceLineageTag: i_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Brand name of the product.\n\tcolumn i_brand\n\t\tdataType: string\n\t\tlineageTag: 0c701039-61ca-4f5f-872b-aab6677ef45c\n\t\tsourceLineageTag: i_brand\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_brand\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Product class or subcategory within the main category.\n\tcolumn i_class\n\t\tdataType: string\n\t\tlineageTag: 5bdf127c-d558-46b2-98e6-0e294148d466\n\t\tsourceLineageTag: i_class\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_class\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Product category classification.\n\tcolumn i_category\n\t\tdataType: string\n\t\tlineageTag: 138de914-55e9-4a46-b5b9-6707b88a38e9\n\t\tsourceLineageTag: i_category\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_category\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Manufacturer or producer of the product.\n\tcolumn i_manufact\n\t\tdataType: string\n\t\tlineageTag: e6852c2b-2bc8-4fb1-86cb-d7ebe7c18637\n\t\tsourceLineageTag: i_manufact\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_manufact\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Size specification of the product.\n\tcolumn i_size\n\t\tdataType: string\n\t\tlineageTag: 0aeb0963-bc02-4389-a3fd-ab7232caaf9c\n\t\tsourceLineageTag: i_size\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_size\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Primary color of the product.\n\tcolumn i_color\n\t\tdataType: string\n\t\tlineageTag: d6709cb2-852f-485a-a0de-8cdd07941038\n\t\tsourceLineageTag: i_color\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_color\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition item = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: item\n\t\t\tschemaName: tpcds_sf__SF___default2rg\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/promotion.tmdl","text":"/// Promotion dimension containing promotional campaign details including costs and promotional names.\ntable promotion\n\tlineageTag: 5942c68a-d07a-4a23-8864-b2c733be46c8\n\tsourceLineageTag: [dbo].[promotion]\n\n\t/// Promotion surrogate key - unique identifier for each promotional campaign.\n\tcolumn p_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 0299ca0a-f58f-4ebe-b752-ffa6b63f9760\n\t\tsourceLineageTag: p_promo_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension (for item-specific promotions).\n\tcolumn p_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 92f8bc42-df79-4a35-8a37-10a02a937c40\n\t\tsourceLineageTag: p_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: p_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Cost associated with running this promotional campaign.\n\tcolumn p_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: c42871df-1b39-4b1e-83cb-142e6864a1e0\n\t\tsourceLineageTag: p_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Name or title of the promotional campaign.\n\tcolumn p_promo_name\n\t\tdataType: string\n\t\tlineageTag: 6ceed2f6-4e12-4edc-b161-8251a356ec17\n\t\tsourceLineageTag: p_promo_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_promo_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition promotion = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: promotion\n\t\t\tschemaName: tpcds_sf__SF___default2rg\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/ship_mode.tmdl","text":"/// Shipping method dimension containing carrier information and shipping type classifications.\ntable ship_mode\n\tlineageTag: db1b87b5-f584-4f30-b70e-0eb00c8b45b8\n\tsourceLineageTag: [dbo].[ship_mode]\n\n\t/// Ship mode surrogate key - unique identifier for each shipping method.\n\tcolumn sm_ship_mode_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4c485c32-fa22-4df6-b4a8-a8df04e2f730\n\t\tsourceLineageTag: sm_ship_mode_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_ship_mode_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type or category of shipping method.\n\tcolumn sm_type\n\t\tdataType: string\n\t\tlineageTag: c49fc6f8-d030-49a2-8105-6a45d1a81ac2\n\t\tsourceLineageTag: sm_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Internal code for the shipping method.\n\tcolumn sm_code\n\t\tdataType: string\n\t\tlineageTag: f54a092d-e847-4cee-84e6-5b813cbf5280\n\t\tsourceLineageTag: sm_code\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_code\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Shipping carrier or company name.\n\tcolumn sm_carrier\n\t\tdataType: string\n\t\tlineageTag: aca70983-6b51-4291-83f8-77fac2e381fe\n\t\tsourceLineageTag: sm_carrier\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_carrier\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition ship_mode = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: ship_mode\n\t\t\tschemaName: tpcds_sf__SF___default2rg\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/store.tmdl","text":"/// Store dimension containing retail location information including geographic details, management, and tax rates.\ntable store\n\tlineageTag: 6835d4a1-f40a-40aa-8a65-98aa4e06a406\n\tsourceLineageTag: [dbo].[store]\n\n\t/// Store surrogate key - unique identifier for each store location.\n\tcolumn s_store_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 14502476-8801-4068-b7c4-875ce7cac939\n\t\tsourceLineageTag: s_store_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_store_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Store name or identifier for the retail location.\n\tcolumn s_store_name\n\t\tdataType: string\n\t\tlineageTag: 4e78d335-4974-4d41-8402-ea89f63f9005\n\t\tsourceLineageTag: s_store_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_store_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Name of the store manager.\n\tcolumn s_manager\n\t\tdataType: string\n\t\tlineageTag: 3a8ddb5b-2ada-42d0-9ef9-89cd96b435b6\n\t\tsourceLineageTag: s_manager\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_manager\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Name of the market manager overseeing this store.\n\tcolumn s_market_manager\n\t\tdataType: string\n\t\tlineageTag: 37c8562d-3ab6-47ad-9b99-2ad1d72e6cd6\n\t\tsourceLineageTag: s_market_manager\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_market_manager\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// City where the store is located.\n\tcolumn s_city\n\t\tdataType: string\n\t\tlineageTag: a1a9209c-8194-4f96-8e94-69e545e064ba\n\t\tsourceLineageTag: s_city\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_city\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// County where the store is located.\n\tcolumn s_county\n\t\tdataType: string\n\t\tlineageTag: e219113c-fe81-45b0-b6be-f02e7ce75403\n\t\tsourceLineageTag: s_county\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_county\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// State or province where the store is located.\n\tcolumn s_state\n\t\tdataType: string\n\t\tlineageTag: fbaeba14-56ed-496d-a004-83f851771750\n\t\tsourceLineageTag: s_state\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_state\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Postal code for the store location.\n\tcolumn s_zip\n\t\tdataType: string\n\t\tlineageTag: 44e7ea79-56d6-4d20-8243-92c35919dcc3\n\t\tsourceLineageTag: s_zip\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_zip\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Tax rate percentage applied at this store location.\n\tcolumn s_tax_percentage\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: ba6d903d-f426-46b7-90a8-756a18261f2a\n\t\tsourceLineageTag: s_tax_percentage\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_tax_percentage\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\tpartition store = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: store\n\t\t\tschemaName: tpcds_sf__SF___default2rg\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/store_sales.tmdl","text":"/// Fact table containing retail store sales transactions with pricing, quantities, profits, and foreign keys to related dimensions.\ntable store_sales\n\tlineageTag: 6c7a2caf-3c22-4f38-a6aa-895a77e2e1c9\n\tsourceLineageTag: [dbo].[store_sales]\n\n\t/// Foreign key to date dimension for sale date.\n\tcolumn ss_sold_date_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4e30d94a-216f-4eaf-9858-fb22c6e6d57e\n\t\tsourceLineageTag: ss_sold_date_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_sold_date_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension.\n\tcolumn ss_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 2b151014-7df5-45f0-9907-03cbe191952b\n\t\tsourceLineageTag: ss_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Customer surrogate key for the transaction.\n\tcolumn ss_customer_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 558da56d-ea73-4562-b35c-ff94e774cca3\n\t\tsourceLineageTag: ss_customer_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_customer_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension.\n\tcolumn ss_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: e65f473f-bb26-4c7d-90f1-139de12a51ab\n\t\tsourceLineageTag: ss_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension.\n\tcolumn ss_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: cb65f3a1-6240-4baa-bfb2-4b24d39e2e87\n\t\tsourceLineageTag: ss_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to store dimension.\n\tcolumn ss_store_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 809a4a04-3bbf-4067-b54f-ee4d5fbbf6d4\n\t\tsourceLineageTag: ss_store_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_store_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to promotion dimension.\n\tcolumn ss_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9a64bbfb-0ae6-4919-b4e9-73e6d90353b3\n\t\tsourceLineageTag: ss_promo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Transaction ticket or receipt number.\n\tcolumn ss_ticket_number\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: d8ce37f4-a12c-4558-b131-66631546b607\n\t\tsourceLineageTag: ss_ticket_number\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ticket_number\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quantity of items sold in this transaction.\n\tcolumn ss_quantity\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 318e6a7b-88a2-413c-8257-315f66313d79\n\t\tsourceLineageTag: ss_quantity\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_quantity\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Wholesale cost per unit for this transaction.\n\tcolumn ss_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: d02100cc-2804-43d7-b752-8e98d579e202\n\t\tsourceLineageTag: ss_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// List price per unit at time of sale.\n\tcolumn ss_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 4f32f293-c19e-497e-906e-089bc204cde9\n\t\tsourceLineageTag: ss_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Actual sales price per unit (after discounts).\n\tcolumn ss_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: e620a719-f6e0-4965-8326-f4a2d255999b\n\t\tsourceLineageTag: ss_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended sales price (sales price \u00d7 quantity).\n\tcolumn ss_ext_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 8b66819a-a151-4fc7-9ec0-cacf7b6e8e01\n\t\tsourceLineageTag: ss_ext_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ext_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended list price (list price \u00d7 quantity).\n\tcolumn ss_ext_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: bc8e9c57-98e2-4032-a3fd-69808e3cdd8b\n\t\tsourceLineageTag: ss_ext_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ext_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Net profit for this transaction (sales price - wholesale cost).\n\tcolumn ss_net_profit\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: d2d6233f-6944-40b5-97d7-2870da393d1b\n\t\tsourceLineageTag: ss_net_profit\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_net_profit\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Technical column used for cache invalidation in DirectLake mode.\n\tcolumn cache_buster\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 7ae60894-bf63-4fef-a327-bf8796466374\n\t\tsourceLineageTag: cache_buster\n\t\tsummarizeBy: none\n\t\tsourceColumn: cache_buster\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition store_sales = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: store_sales\n\t\t\tschemaName: tpcds_sf__SF___default2rg\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition.pbism","text":"{\"$schema\": \"https://developer.microsoft.com/json-schemas/fabric/item/semanticModel/definitionProperties/1.0.0/schema.json\", \"version\": \"5.0\", \"settings\": {}}"}]},"cluster":{"model":"tpcds_sf__SF___cluster","parts":[{"path":"definition/database.tmdl","text":"database tpcds_sf__SF___cluster\n\tcompatibilityLevel: 1702\n\tcompatibilityMode: powerBI\n\tlanguage: 1033\n\n"},{"path":"definition/expressions.tmdl","text":"expression 'DirectLake - tpcds_sf__SF__' =\n\t\tlet\n\t\t    Source = AzureStorage.DataLake(\"https://onelake.dfs.fabric.microsoft.com/51650f82-6bb5-4023-b0ab-db197d32e0be/01b539f3-4a9d-45ef-b1ef-0ba59552eb21\", [HierarchicalNavigation=true])\n\t\tin\n\t\t    Source\n\tlineageTag: c6e3390d-cc84-4e13-ab93-53308a346336\n\n\tannotation PBI_IncludeFutureArtifacts = False\n\n"},{"path":"definition/model.tmdl","text":"model Model\n\tdirectLakeBehavior: directLakeOnly\n\tculture: en-US\n\tdefaultPowerBIDataSourceVersion: powerBI_V3\n\tsourceQueryCulture: en-US\n\tdataAccessOptions\n\t\tlegacyRedirects\n\t\treturnErrorValuesAsNull\n\nannotation PBI_QueryOrder = [\"DirectLake - tpcds_sf__SF__\"]\n\nannotation __PBI_TimeIntelligenceEnabled = 1\n\nannotation __LastRPTime = 134272970581620514\n\nannotation PBI_ProTooling = [\"DirectLakeOnOneLakeInWeb\",\"WebModelingEdit\"]\n\nannotation __TEdtr = 1\n\nannotation TabularEditor_SerializeOptions = {\"IgnoreInferredObjects\":true,\"IgnoreInferredProperties\":true,\"IgnoreTimestamps\":true,\"SplitMultilineStrings\":true,\"PrefixFilenames\":false,\"LocalTranslations\":true,\"LocalPerspectives\":true,\"LocalRelationships\":true,\"Levels\":[\"Data Sources\",\"Perspectives\",\"Relationships\",\"Roles\",\"Shared Expressions\",\"Tables\",\"Tables/Calculation Items\",\"Tables/Columns\",\"Tables/Hierarchies\",\"Tables/Measures\",\"Tables/Partitions\",\"Translations\"]}\n\nref table store\nref table item\nref table date_dim\nref table store_sales\nref table catalog_page\nref table promotion\nref table ship_mode\nref table catalog_sales\nref table customer_address\nref table customer_demographics\nref table 'Measures 1'\nref table 'Time Unit'\n\n"},{"path":"definition/relationships.tmdl","text":"relationship b3bd8d91-ae6b-4489-83cd-34ca2c3213d8\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_bill_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship b727abac-57ca-484a-8183-1b4c10921d84\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_bill_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship 41e21761-05b3-410b-960e-54805fb2c4f2\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_catalog_page_sk\n\ttoColumn: catalog_page.cp_catalog_page_sk\n\nrelationship c7ce7218-d702-40d6-94b4-80009e453fe9\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship 4d02a5f5-e67a-4f65-a203-192e1c2ef166\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_promo_sk\n\ttoColumn: promotion.p_promo_sk\n\nrelationship 545e95f8-e63b-4a27-9c91-3cd725ca10e0\n\tisActive: false\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship 685424a1-f6c7-42d6-8f44-3cccf5f08db4\n\tisActive: false\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship 2e0c0f18-79bf-48f8-be1a-2c53cc33e257\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_mode_sk\n\ttoColumn: ship_mode.sm_ship_mode_sk\n\nrelationship c5edc5f8-f418-44a4-a8aa-caff6e0bef29\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_sold_date_sk\n\ttoColumn: date_dim.d_date_sk_1\n\nrelationship 9f8c8977-b8ac-4985-aed9-7711ce5004af\n\tisActive: false\n\tfromColumn: promotion.p_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship f824425a-7dd8-4a70-9ee5-2042a74ed97d\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship be9c6d5e-0bed-4db1-8558-326d5dfba951\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship f172b575-b3ce-48a4-b264-2928516104a2\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship 95b347fa-cf7e-4885-8124-86eb2ff56351\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_promo_sk\n\ttoColumn: promotion.p_promo_sk\n\nrelationship c782badb-be54-4423-98b0-22eebad6aa29\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_sold_date_sk\n\ttoColumn: date_dim.d_date_sk_1\n\nrelationship 9e5ca2d3-a5aa-4d43-b00a-07e4a857b1e4\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_store_sk\n\ttoColumn: store.s_store_sk\n\n"},{"path":"definition/tables/Measures 1.tmdl","text":"/// Calculated measures table containing key business metrics for revenue, quantity, profit, tax, and performance analysis across store and catalog channels.\ntable 'Measures 1'\n\tlineageTag: ae7cf165-f530-4f98-a4c5-befd958ff771\n\n\t/// Revenue from catalog sales channel only, based on extended sales price\n\tmeasure 'Catalog Revenue' = SUM('catalog_sales'[cs_ext_sales_price])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 01. Revenue\n\t\tlineageTag: bed34121-c406-4e25-b931-6ea438aede2a\n\n\t/// Revenue from store sales channel only, based on extended sales price\n\tmeasure 'Store Revenue' = SUM('store_sales'[ss_ext_sales_price])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 01. Revenue\n\t\tlineageTag: 1755e841-2eeb-4e1e-9a98-c860eb583a79\n\n\t/// Total units sold through catalog sales channel\n\tmeasure 'Catalog Sales Quantity' = SUM('catalog_sales'[cs_quantity])\n\t\tformatString: #,0\n\t\tdisplayFolder: 02. Quantity\n\t\tlineageTag: 6da96e65-1eb2-4e76-afdb-50d9106f56c1\n\n\t/// Net profit from store sales channel only\n\tmeasure 'Store Net Profit' = SUM('store_sales'[ss_net_profit])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 03. Profit\n\t\tlineageTag: a5a04305-c986-4791-925e-b9115939e453\n\n\t/// Number of unique customers who made store purchases\n\tmeasure 'Store Distinct Customers' = DISTINCTCOUNT('store_sales'[ss_customer_sk])\n\t\tformatString: #,0\n\t\tdisplayFolder: 04. Distinct Counts\n\t\tlineageTag: ace0485f-176a-4401-bb53-41c99ce5c356\n\n\t/// Revenue for the same period in the previous year\n\tmeasure 'Store Revenue Same Period LY' =\n\t\t\t\n\t\t\tCALCULATE(\n\t\t\t    [Store Revenue],\n\t\t\t    SAMEPERIODLASTYEAR('tpcds_calendar')\n\t\t\t)\n\t\tformatString: $#,0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 303dbdc1-0484-496c-b14f-a41b0ef53f82\n\n\t/// Revenue for the same period in the previous year\n\tmeasure 'Store Revenue YoY' =\n\t\t\tVAR CurrentYearRev = [Store Revenue]\n\t\t\tVAR PreviousYearRev = [Store Revenue Same Period LY]\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentYearRev - PreviousYearRev, PreviousYearRev, 0)\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 39648465-7aac-4652-8c2f-d6599df15204\n\n\t/// Catalog Sales Same Period LY\n\tmeasure 'Catalog Sales Same Period LY' =\n\t\t\t\n\t\t\tCALCULATE(\n\t\t\t    [Catalog Sales Quantity],\n\t\t\t    SAMEPERIODLASTYEAR('tpcds_calendar')\n\t\t\t)\n\t\tformatString: 0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: d1d5179d-688b-487c-8638-81780d95eb05\n\n\t/// Catalog Sales YoY\n\tmeasure 'Catalog Sales YoY' =\n\t\t\tVAR CurrentYearCatSales = [Catalog Sales Quantity]\n\t\t\tVAR PreviousYearCatSales = [Catalog Sales Same Period LY]\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentYearCatSales - PreviousYearCatSales, PreviousYearCatSales, 0)\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 0bc09d6d-1bdd-4f7e-ac26-a12e87ccf67d\n\n\t/// Store Profit % by Item Category\n\tmeasure 'Store Profit % by Item Category' = ```\n\t\t\t\n\t\t\tVAR CurrentProfit = [Store Net Profit]\n\t\t\tVAR TotalProfitAllCategories = \n\t\t\t    CALCULATE(\n\t\t\t        [Store Net Profit],\n\t\t\t        ALLEXCEPT(\n\t\t\t            'item',\n\t\t\t            'item'[i_brand],\n\t\t\t            'item'[i_manufact]\n\t\t\t        )\n\t\t\t    )\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentProfit, TotalProfitAllCategories, 0)\n\t\t\t```\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 05. Advanced % Share\n\t\tlineageTag: 04f417ef-ba48-4d9e-873d-76d516ec3a54\n\n\t/// Total revenue from beginning of year to current date selection\n\tmeasure 'Store Revenue YTD' = TOTALYTD([Store Revenue],'tpcds_calendar')\n\t\tformatString: $#,0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 0e0dbb36-1e75-4c77-8b7f-9a462a143dcb\n\n\t/// Dummy\n\tcolumn Dummy\n\t\tformatString: 0\n\t\tlineageTag: c1dfb904-84a8-4dfc-a768-d5e10f70f255\n\t\tsummarizeBy: none\n\t\tisNameInferred\n\t\tsourceColumn: [Dummy]\n\n\tpartition 'Measures 1' = calculated\n\t\tmode: import\n\t\tsource = ROW(\"Dummy\", 1)\n\n"},{"path":"definition/tables/Time Unit.tmdl","text":"/// Field parameter table for dynamic time unit selection in reports, supporting Year and Quarter groupings.\ntable 'Time Unit'\n\tlineageTag: c100712f-ebcb-4d0f-a435-e1a4c8ce3229\n\n\t/// Display name for the time unit selection (Year, Quarter).\n\tcolumn 'Time Unit'\n\t\tlineageTag: bd5d5ab0-6269-4df0-bd39-9616f6e2b7d5\n\t\tsummarizeBy: none\n\t\tsourceColumn: [Value1]\n\t\tsortByColumn: 'Time Unit Order'\n\n\t\trelatedColumnDetails\n\t\t\tgroupByColumn: 'Time Unit Fields'\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// DAX column reference for the selected time unit.\n\tcolumn 'Time Unit Fields'\n\t\tisHidden\n\t\tlineageTag: 4be3ced6-f4db-40c4-84b2-5ef9decf7fee\n\t\tsummarizeBy: none\n\t\tsourceColumn: [Value2]\n\t\tsortByColumn: 'Time Unit Order'\n\n\t\textendedProperty ParameterMetadata =\n\t\t\t\t{\n\t\t\t\t  \"version\": 3,\n\t\t\t\t  \"kind\": 2\n\t\t\t\t}\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Sort order for time unit options in field parameter.\n\tcolumn 'Time Unit Order'\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 36566b71-2a4f-4ddb-86c0-b4c7b3d8ab7f\n\t\tsummarizeBy: sum\n\t\tsourceColumn: [Value3]\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition 'Time Unit' = calculated\n\t\tmode: import\n\t\tsource =\n\t\t\t\t{\n\t\t\t\t    (\"Year\", NAMEOF('date_dim'[d_year]), 0),\n\t\t\t\t    (\"Quarter\", NAMEOF('date_dim'[d_quarter_name]), 1)\n\t\t\t\t}\n\n\tannotation PBI_Id = 706957f972f8497896950148d2d5f0af\n\n"},{"path":"definition/tables/catalog_page.tmdl","text":"/// Catalog page dimension containing information about catalog pages used in catalog sales campaigns.\ntable catalog_page\n\tlineageTag: b4e5474e-8b18-42b8-9a56-9c2416950a0a\n\tsourceLineageTag: [dbo].[catalog_page]\n\n\t/// Catalog page surrogate key - unique identifier for each catalog page.\n\tcolumn cp_catalog_page_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 71a9dba7-e582-461e-b8de-f37322137084\n\t\tsourceLineageTag: cp_catalog_page_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: cp_catalog_page_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type or category of the catalog page.\n\tcolumn cp_type\n\t\tdataType: string\n\t\tlineageTag: bdf42608-dc88-489d-90ce-24a87bc40378\n\t\tsourceLineageTag: cp_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: cp_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition catalog_page = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: catalog_page\n\t\t\tschemaName: tpcds_sf__SF___cluster\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/catalog_sales.tmdl","text":"/// Fact table containing catalog sales transactions with pricing, quantities, profits, and foreign keys to related dimensions.\ntable catalog_sales\n\tlineageTag: c38e1686-b3a0-473b-8604-8efe0410ea48\n\tsourceLineageTag: [dbo].[catalog_sales]\n\n\t/// Foreign key to date dimension for sale date.\n\tcolumn cs_sold_date_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: ac2ce865-caed-4039-bda1-a182c516cfdd\n\t\tsourceLineageTag: cs_sold_date_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_sold_date_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension for billing customer.\n\tcolumn cs_bill_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 310a5931-6249-4ad7-86b4-7aca5fbc39a9\n\t\tsourceLineageTag: cs_bill_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_bill_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension for billing address.\n\tcolumn cs_bill_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 51e6cbcf-e98d-4ea9-865e-a500419bec99\n\t\tsourceLineageTag: cs_bill_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_bill_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension for shipping customer.\n\tcolumn cs_ship_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4a433af2-d889-41d8-9c6f-1f05f94f0070\n\t\tsourceLineageTag: cs_ship_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension for shipping address.\n\tcolumn cs_ship_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 25b41390-6ff3-4f15-b0e6-f26a3da61ef5\n\t\tsourceLineageTag: cs_ship_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to catalog page dimension.\n\tcolumn cs_catalog_page_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: b01fc3a0-0a51-4660-a5d3-56d28ea386cd\n\t\tsourceLineageTag: cs_catalog_page_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_catalog_page_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to ship mode dimension.\n\tcolumn cs_ship_mode_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 0a85820a-e8f9-41f9-b50e-9610890408e8\n\t\tsourceLineageTag: cs_ship_mode_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_mode_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension.\n\tcolumn cs_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9024dff9-9629-4f0e-abae-0b29f00d568d\n\t\tsourceLineageTag: cs_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to promotion dimension.\n\tcolumn cs_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: dbfb7b81-9eef-4b86-aac0-a48cc36d0168\n\t\tsourceLineageTag: cs_promo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Catalog order number for this transaction.\n\tcolumn cs_order_number\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: ccec0342-6144-411f-bc2d-90234bcfb5ac\n\t\tsourceLineageTag: cs_order_number\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_order_number\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quantity of items ordered in this transaction.\n\tcolumn cs_quantity\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 0f5abf99-45b4-495b-bad3-795d0e8e5802\n\t\tsourceLineageTag: cs_quantity\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_quantity\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Wholesale cost per unit for this transaction.\n\tcolumn cs_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 37d760e7-f755-42e9-9dba-3a28cc211ae7\n\t\tsourceLineageTag: cs_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// List price per unit at time of order.\n\tcolumn cs_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 0eda1627-f565-444d-92e5-1a5bb0929e4d\n\t\tsourceLineageTag: cs_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Actual sales price per unit (after discounts).\n\tcolumn cs_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: eebce86e-da48-4799-8f93-1a8e69de6605\n\t\tsourceLineageTag: cs_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended sales price (sales price \u00d7 quantity).\n\tcolumn cs_ext_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: f0c74fde-bf40-47e4-9352-6bc4d4f219a0\n\t\tsourceLineageTag: cs_ext_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_ext_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended list price (list price \u00d7 quantity).\n\tcolumn cs_ext_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 845b8a06-567d-49c0-9455-1a469d81f646\n\t\tsourceLineageTag: cs_ext_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_ext_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Net profit for this transaction (sales price - wholesale cost).\n\tcolumn cs_net_profit\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 341a4a24-e726-4b20-8a50-967f689a5d01\n\t\tsourceLineageTag: cs_net_profit\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_net_profit\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Technical column used for cache invalidation in DirectLake mode.\n\tcolumn cache_buster\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 33947ffc-dc12-42ab-af62-5b00c42adf7a\n\t\tsourceLineageTag: cache_buster\n\t\tsummarizeBy: none\n\t\tsourceColumn: cache_buster\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition catalog_sales = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: catalog_sales\n\t\t\tschemaName: tpcds_sf__SF___cluster\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/customer_address.tmdl","text":"/// Customer address dimension containing geographic information for billing and shipping addresses.\ntable customer_address\n\tlineageTag: 75246b34-d9f4-46fd-813f-bc5f0e47ab8c\n\tsourceLineageTag: [dbo].[customer_address]\n\n\t/// Customer address surrogate key - unique identifier for each address.\n\tcolumn ca_address_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9084b5d7-d188-4fb9-9c92-211d5b68f397\n\t\tsourceLineageTag: ca_address_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_address_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// City name for the customer address.\n\tcolumn ca_city\n\t\tdataType: string\n\t\tlineageTag: 03b2fb94-a08c-4010-9157-b91edcb60daf\n\t\tsourceLineageTag: ca_city\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_city\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// County name for the customer address.\n\tcolumn ca_county\n\t\tdataType: string\n\t\tlineageTag: 0ef7c0d6-1e17-42c5-a9bb-fb014c91398a\n\t\tsourceLineageTag: ca_county\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_county\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// State or province for the customer address.\n\tcolumn ca_state\n\t\tdataType: string\n\t\tlineageTag: 24c6ee20-e4ab-4ca3-aa30-934ba66a55db\n\t\tsourceLineageTag: ca_state\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_state\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Postal code for the customer address.\n\tcolumn ca_zip\n\t\tdataType: string\n\t\tlineageTag: a57bf4fe-e3de-4848-81bd-e71dd885c99f\n\t\tsourceLineageTag: ca_zip\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_zip\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type of location (e.g., residential, commercial).\n\tcolumn ca_location_type\n\t\tdataType: string\n\t\tlineageTag: 9a90fb17-4314-40df-84ae-071cdd4eb3c7\n\t\tsourceLineageTag: ca_location_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_location_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition customer_address = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: customer_address\n\t\t\tschemaName: tpcds_sf__SF___cluster\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/customer_demographics.tmdl","text":"/// Customer demographics dimension containing education status and marital status attributes for customer segmentation.\ntable customer_demographics\n\tlineageTag: 52f8f268-90f3-4f40-a527-07a74d9e9baf\n\tsourceLineageTag: [dbo].[customer_demographics]\n\n\t/// Customer demographics surrogate key - unique identifier for each demographic profile.\n\tcolumn cd_demo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: d2cad4d0-4a88-41ab-b7ee-94ab73973694\n\t\tsourceLineageTag: cd_demo_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_demo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Marital status of the customer.\n\tcolumn cd_marital_status\n\t\tdataType: string\n\t\tlineageTag: e29a3681-2c0d-4579-9b35-5112d86c0f46\n\t\tsourceLineageTag: cd_marital_status\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_marital_status\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Education level of the customer.\n\tcolumn cd_education_status\n\t\tdataType: string\n\t\tlineageTag: 43260b21-0847-455f-a853-e2233af2a62e\n\t\tsourceLineageTag: cd_education_status\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_education_status\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition customer_demographics = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: customer_demographics\n\t\t\tschemaName: tpcds_sf__SF___cluster\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/date_dim.tmdl","text":"/// Date dimension providing calendar hierarchy with year, quarter, month, and day-of-week attributes for time-based analysis.\ntable date_dim\n\tlineageTag: 9e7ec2de-382c-4a3b-9589-1ece445bfa27\n\tsourceLineageTag: [dbo].[date_dim]\n\n\t/// Date surrogate key - unique identifier for each calendar date.\n\tcolumn d_date_sk_1\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 63841738-6970-46ac-8106-970384b1052b\n\t\tsourceLineageTag: d_date_sk_1\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_date_sk_1\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Calendar date value.\n\tcolumn d_date\n\t\tdataType: dateTime\n\t\tformatString: General Date\n\t\tlineageTag: dc08a5e6-1a30-4d5f-9b03-1c98c7bac892\n\t\tsourceLineageTag: d_date\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_date\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Calendar year (e.g., 2023).\n\tcolumn d_year\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: fcf4b9e9-c880-4ee0-907c-3a18951b0809\n\t\tsourceLineageTag: d_year\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_year\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Day of week (1=Sunday, 7=Saturday).\n\tcolumn d_dow\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 339ac74d-ab31-4d4e-93eb-524b9f7e211c\n\t\tsourceLineageTag: d_dow\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_dow\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Month of year (1-12).\n\tcolumn d_moy\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: faa3deb0-9def-4334-bd42-a1b9d60b51b7\n\t\tsourceLineageTag: d_moy\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_moy\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Day of month (1-31).\n\tcolumn d_dom\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 83cd6ebd-7811-4616-a884-9930f66f8bfe\n\t\tsourceLineageTag: d_dom\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_dom\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quarter of year (1-4).\n\tcolumn d_qoy\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 4962ee0a-fcb9-4bf8-a5c7-43a66a94e67f\n\t\tsourceLineageTag: d_qoy\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_qoy\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quarter name (e.g., 'Q1 2023').\n\tcolumn d_quarter_name\n\t\tdataType: string\n\t\tlineageTag: 2426fa03-107c-4f51-a0b4-c59bb4b85d4f\n\t\tsourceLineageTag: d_quarter_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_quarter_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition date_dim = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: date_dim\n\t\t\tschemaName: tpcds_sf__SF___cluster\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n\tcalendar tpcds_calendar\n\t\tlineageTag: dd529b7a-99fe-4cb5-928c-df1abed499f1\n\n\t\tcalendarColumnGroup = year\n\t\t\tprimaryColumn: d_year\n\n\t\tcalendarColumnGroup = quarter\n\t\t\tprimaryColumn: d_quarter_name\n\n\t\tcalendarColumnGroup = quarterOfYear\n\t\t\tprimaryColumn: d_qoy\n\n\t\tcalendarColumnGroup = date\n\t\t\tprimaryColumn: d_date\n\n"},{"path":"definition/tables/item.tmdl","text":"/// Product dimension containing item details such as brand, category, class, color, pricing, and manufacturing information.\ntable item\n\tlineageTag: 0f958ae1-af31-4b29-87a3-7e9ee589887e\n\tsourceLineageTag: [dbo].[item]\n\n\t/// Item surrogate key - unique identifier for each product.\n\tcolumn i_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 23893369-a589-4284-afa3-b1e291864600\n\t\tsourceLineageTag: i_item_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Current retail price of the product.\n\tcolumn i_current_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 70b27235-eefb-4732-8e0d-64c9576f0776\n\t\tsourceLineageTag: i_current_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_current_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Wholesale cost paid for the product.\n\tcolumn i_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 90cbdd03-df2f-4439-941d-de295ceb5135\n\t\tsourceLineageTag: i_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Brand name of the product.\n\tcolumn i_brand\n\t\tdataType: string\n\t\tlineageTag: 0c701039-61ca-4f5f-872b-aab6677ef45c\n\t\tsourceLineageTag: i_brand\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_brand\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Product class or subcategory within the main category.\n\tcolumn i_class\n\t\tdataType: string\n\t\tlineageTag: 5bdf127c-d558-46b2-98e6-0e294148d466\n\t\tsourceLineageTag: i_class\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_class\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Product category classification.\n\tcolumn i_category\n\t\tdataType: string\n\t\tlineageTag: 138de914-55e9-4a46-b5b9-6707b88a38e9\n\t\tsourceLineageTag: i_category\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_category\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Manufacturer or producer of the product.\n\tcolumn i_manufact\n\t\tdataType: string\n\t\tlineageTag: e6852c2b-2bc8-4fb1-86cb-d7ebe7c18637\n\t\tsourceLineageTag: i_manufact\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_manufact\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Size specification of the product.\n\tcolumn i_size\n\t\tdataType: string\n\t\tlineageTag: 0aeb0963-bc02-4389-a3fd-ab7232caaf9c\n\t\tsourceLineageTag: i_size\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_size\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Primary color of the product.\n\tcolumn i_color\n\t\tdataType: string\n\t\tlineageTag: d6709cb2-852f-485a-a0de-8cdd07941038\n\t\tsourceLineageTag: i_color\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_color\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition item = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: item\n\t\t\tschemaName: tpcds_sf__SF___cluster\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/promotion.tmdl","text":"/// Promotion dimension containing promotional campaign details including costs and promotional names.\ntable promotion\n\tlineageTag: 5942c68a-d07a-4a23-8864-b2c733be46c8\n\tsourceLineageTag: [dbo].[promotion]\n\n\t/// Promotion surrogate key - unique identifier for each promotional campaign.\n\tcolumn p_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 0299ca0a-f58f-4ebe-b752-ffa6b63f9760\n\t\tsourceLineageTag: p_promo_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension (for item-specific promotions).\n\tcolumn p_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 92f8bc42-df79-4a35-8a37-10a02a937c40\n\t\tsourceLineageTag: p_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: p_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Cost associated with running this promotional campaign.\n\tcolumn p_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: c42871df-1b39-4b1e-83cb-142e6864a1e0\n\t\tsourceLineageTag: p_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Name or title of the promotional campaign.\n\tcolumn p_promo_name\n\t\tdataType: string\n\t\tlineageTag: 6ceed2f6-4e12-4edc-b161-8251a356ec17\n\t\tsourceLineageTag: p_promo_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_promo_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition promotion = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: promotion\n\t\t\tschemaName: tpcds_sf__SF___cluster\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/ship_mode.tmdl","text":"/// Shipping method dimension containing carrier information and shipping type classifications.\ntable ship_mode\n\tlineageTag: db1b87b5-f584-4f30-b70e-0eb00c8b45b8\n\tsourceLineageTag: [dbo].[ship_mode]\n\n\t/// Ship mode surrogate key - unique identifier for each shipping method.\n\tcolumn sm_ship_mode_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4c485c32-fa22-4df6-b4a8-a8df04e2f730\n\t\tsourceLineageTag: sm_ship_mode_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_ship_mode_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type or category of shipping method.\n\tcolumn sm_type\n\t\tdataType: string\n\t\tlineageTag: c49fc6f8-d030-49a2-8105-6a45d1a81ac2\n\t\tsourceLineageTag: sm_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Internal code for the shipping method.\n\tcolumn sm_code\n\t\tdataType: string\n\t\tlineageTag: f54a092d-e847-4cee-84e6-5b813cbf5280\n\t\tsourceLineageTag: sm_code\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_code\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Shipping carrier or company name.\n\tcolumn sm_carrier\n\t\tdataType: string\n\t\tlineageTag: aca70983-6b51-4291-83f8-77fac2e381fe\n\t\tsourceLineageTag: sm_carrier\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_carrier\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition ship_mode = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: ship_mode\n\t\t\tschemaName: tpcds_sf__SF___cluster\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/store.tmdl","text":"/// Store dimension containing retail location information including geographic details, management, and tax rates.\ntable store\n\tlineageTag: 6835d4a1-f40a-40aa-8a65-98aa4e06a406\n\tsourceLineageTag: [dbo].[store]\n\n\t/// Store surrogate key - unique identifier for each store location.\n\tcolumn s_store_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 14502476-8801-4068-b7c4-875ce7cac939\n\t\tsourceLineageTag: s_store_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_store_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Store name or identifier for the retail location.\n\tcolumn s_store_name\n\t\tdataType: string\n\t\tlineageTag: 4e78d335-4974-4d41-8402-ea89f63f9005\n\t\tsourceLineageTag: s_store_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_store_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Name of the store manager.\n\tcolumn s_manager\n\t\tdataType: string\n\t\tlineageTag: 3a8ddb5b-2ada-42d0-9ef9-89cd96b435b6\n\t\tsourceLineageTag: s_manager\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_manager\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Name of the market manager overseeing this store.\n\tcolumn s_market_manager\n\t\tdataType: string\n\t\tlineageTag: 37c8562d-3ab6-47ad-9b99-2ad1d72e6cd6\n\t\tsourceLineageTag: s_market_manager\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_market_manager\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// City where the store is located.\n\tcolumn s_city\n\t\tdataType: string\n\t\tlineageTag: a1a9209c-8194-4f96-8e94-69e545e064ba\n\t\tsourceLineageTag: s_city\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_city\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// County where the store is located.\n\tcolumn s_county\n\t\tdataType: string\n\t\tlineageTag: e219113c-fe81-45b0-b6be-f02e7ce75403\n\t\tsourceLineageTag: s_county\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_county\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// State or province where the store is located.\n\tcolumn s_state\n\t\tdataType: string\n\t\tlineageTag: fbaeba14-56ed-496d-a004-83f851771750\n\t\tsourceLineageTag: s_state\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_state\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Postal code for the store location.\n\tcolumn s_zip\n\t\tdataType: string\n\t\tlineageTag: 44e7ea79-56d6-4d20-8243-92c35919dcc3\n\t\tsourceLineageTag: s_zip\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_zip\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Tax rate percentage applied at this store location.\n\tcolumn s_tax_percentage\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: ba6d903d-f426-46b7-90a8-756a18261f2a\n\t\tsourceLineageTag: s_tax_percentage\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_tax_percentage\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\tpartition store = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: store\n\t\t\tschemaName: tpcds_sf__SF___cluster\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/store_sales.tmdl","text":"/// Fact table containing retail store sales transactions with pricing, quantities, profits, and foreign keys to related dimensions.\ntable store_sales\n\tlineageTag: 6c7a2caf-3c22-4f38-a6aa-895a77e2e1c9\n\tsourceLineageTag: [dbo].[store_sales]\n\n\t/// Foreign key to date dimension for sale date.\n\tcolumn ss_sold_date_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4e30d94a-216f-4eaf-9858-fb22c6e6d57e\n\t\tsourceLineageTag: ss_sold_date_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_sold_date_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension.\n\tcolumn ss_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 2b151014-7df5-45f0-9907-03cbe191952b\n\t\tsourceLineageTag: ss_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Customer surrogate key for the transaction.\n\tcolumn ss_customer_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 558da56d-ea73-4562-b35c-ff94e774cca3\n\t\tsourceLineageTag: ss_customer_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_customer_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension.\n\tcolumn ss_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: e65f473f-bb26-4c7d-90f1-139de12a51ab\n\t\tsourceLineageTag: ss_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension.\n\tcolumn ss_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: cb65f3a1-6240-4baa-bfb2-4b24d39e2e87\n\t\tsourceLineageTag: ss_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to store dimension.\n\tcolumn ss_store_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 809a4a04-3bbf-4067-b54f-ee4d5fbbf6d4\n\t\tsourceLineageTag: ss_store_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_store_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to promotion dimension.\n\tcolumn ss_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9a64bbfb-0ae6-4919-b4e9-73e6d90353b3\n\t\tsourceLineageTag: ss_promo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Transaction ticket or receipt number.\n\tcolumn ss_ticket_number\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: d8ce37f4-a12c-4558-b131-66631546b607\n\t\tsourceLineageTag: ss_ticket_number\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ticket_number\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quantity of items sold in this transaction.\n\tcolumn ss_quantity\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 318e6a7b-88a2-413c-8257-315f66313d79\n\t\tsourceLineageTag: ss_quantity\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_quantity\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Wholesale cost per unit for this transaction.\n\tcolumn ss_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: d02100cc-2804-43d7-b752-8e98d579e202\n\t\tsourceLineageTag: ss_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// List price per unit at time of sale.\n\tcolumn ss_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 4f32f293-c19e-497e-906e-089bc204cde9\n\t\tsourceLineageTag: ss_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Actual sales price per unit (after discounts).\n\tcolumn ss_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: e620a719-f6e0-4965-8326-f4a2d255999b\n\t\tsourceLineageTag: ss_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended sales price (sales price \u00d7 quantity).\n\tcolumn ss_ext_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 8b66819a-a151-4fc7-9ec0-cacf7b6e8e01\n\t\tsourceLineageTag: ss_ext_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ext_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended list price (list price \u00d7 quantity).\n\tcolumn ss_ext_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: bc8e9c57-98e2-4032-a3fd-69808e3cdd8b\n\t\tsourceLineageTag: ss_ext_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ext_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Net profit for this transaction (sales price - wholesale cost).\n\tcolumn ss_net_profit\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: d2d6233f-6944-40b5-97d7-2870da393d1b\n\t\tsourceLineageTag: ss_net_profit\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_net_profit\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Technical column used for cache invalidation in DirectLake mode.\n\tcolumn cache_buster\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 7ae60894-bf63-4fef-a327-bf8796466374\n\t\tsourceLineageTag: cache_buster\n\t\tsummarizeBy: none\n\t\tsourceColumn: cache_buster\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition store_sales = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: store_sales\n\t\t\tschemaName: tpcds_sf__SF___cluster\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition.pbism","text":"{\"$schema\": \"https://developer.microsoft.com/json-schemas/fabric/item/semanticModel/definitionProperties/1.0.0/schema.json\", \"version\": \"5.0\", \"settings\": {}}"}]},"clustersn":{"model":"tpcds_sf__SF___clustersn","parts":[{"path":"definition/database.tmdl","text":"database tpcds_sf__SF___clustersn\n\tcompatibilityLevel: 1702\n\tcompatibilityMode: powerBI\n\tlanguage: 1033\n\n"},{"path":"definition/expressions.tmdl","text":"expression 'DirectLake - tpcds_sf__SF__' =\n\t\tlet\n\t\t    Source = AzureStorage.DataLake(\"https://onelake.dfs.fabric.microsoft.com/51650f82-6bb5-4023-b0ab-db197d32e0be/01b539f3-4a9d-45ef-b1ef-0ba59552eb21\", [HierarchicalNavigation=true])\n\t\tin\n\t\t    Source\n\tlineageTag: c6e3390d-cc84-4e13-ab93-53308a346336\n\n\tannotation PBI_IncludeFutureArtifacts = False\n\n"},{"path":"definition/model.tmdl","text":"model Model\n\tdirectLakeBehavior: directLakeOnly\n\tculture: en-US\n\tdefaultPowerBIDataSourceVersion: powerBI_V3\n\tsourceQueryCulture: en-US\n\tdataAccessOptions\n\t\tlegacyRedirects\n\t\treturnErrorValuesAsNull\n\nannotation PBI_QueryOrder = [\"DirectLake - tpcds_sf__SF__\"]\n\nannotation __PBI_TimeIntelligenceEnabled = 1\n\nannotation __LastRPTime = 134272970581620514\n\nannotation PBI_ProTooling = [\"DirectLakeOnOneLakeInWeb\",\"WebModelingEdit\"]\n\nannotation __TEdtr = 1\n\nannotation TabularEditor_SerializeOptions = {\"IgnoreInferredObjects\":true,\"IgnoreInferredProperties\":true,\"IgnoreTimestamps\":true,\"SplitMultilineStrings\":true,\"PrefixFilenames\":false,\"LocalTranslations\":true,\"LocalPerspectives\":true,\"LocalRelationships\":true,\"Levels\":[\"Data Sources\",\"Perspectives\",\"Relationships\",\"Roles\",\"Shared Expressions\",\"Tables\",\"Tables/Calculation Items\",\"Tables/Columns\",\"Tables/Hierarchies\",\"Tables/Measures\",\"Tables/Partitions\",\"Translations\"]}\n\nref table store\nref table item\nref table date_dim\nref table store_sales\nref table catalog_page\nref table promotion\nref table ship_mode\nref table catalog_sales\nref table customer_address\nref table customer_demographics\nref table 'Measures 1'\nref table 'Time Unit'\n\n"},{"path":"definition/relationships.tmdl","text":"relationship b3bd8d91-ae6b-4489-83cd-34ca2c3213d8\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_bill_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship b727abac-57ca-484a-8183-1b4c10921d84\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_bill_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship 41e21761-05b3-410b-960e-54805fb2c4f2\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_catalog_page_sk\n\ttoColumn: catalog_page.cp_catalog_page_sk\n\nrelationship c7ce7218-d702-40d6-94b4-80009e453fe9\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship 4d02a5f5-e67a-4f65-a203-192e1c2ef166\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_promo_sk\n\ttoColumn: promotion.p_promo_sk\n\nrelationship 545e95f8-e63b-4a27-9c91-3cd725ca10e0\n\tisActive: false\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship 685424a1-f6c7-42d6-8f44-3cccf5f08db4\n\tisActive: false\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship 2e0c0f18-79bf-48f8-be1a-2c53cc33e257\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_mode_sk\n\ttoColumn: ship_mode.sm_ship_mode_sk\n\nrelationship c5edc5f8-f418-44a4-a8aa-caff6e0bef29\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_sold_date_sk\n\ttoColumn: date_dim.d_date_sk_1\n\nrelationship 9f8c8977-b8ac-4985-aed9-7711ce5004af\n\tisActive: false\n\tfromColumn: promotion.p_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship f824425a-7dd8-4a70-9ee5-2042a74ed97d\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship be9c6d5e-0bed-4db1-8558-326d5dfba951\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship f172b575-b3ce-48a4-b264-2928516104a2\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship 95b347fa-cf7e-4885-8124-86eb2ff56351\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_promo_sk\n\ttoColumn: promotion.p_promo_sk\n\nrelationship c782badb-be54-4423-98b0-22eebad6aa29\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_sold_date_sk\n\ttoColumn: date_dim.d_date_sk_1\n\nrelationship 9e5ca2d3-a5aa-4d43-b00a-07e4a857b1e4\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_store_sk\n\ttoColumn: store.s_store_sk\n\n"},{"path":"definition/tables/Measures 1.tmdl","text":"/// Calculated measures table containing key business metrics for revenue, quantity, profit, tax, and performance analysis across store and catalog channels.\ntable 'Measures 1'\n\tlineageTag: ae7cf165-f530-4f98-a4c5-befd958ff771\n\n\t/// Revenue from catalog sales channel only, based on extended sales price\n\tmeasure 'Catalog Revenue' = SUM('catalog_sales'[cs_ext_sales_price])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 01. Revenue\n\t\tlineageTag: bed34121-c406-4e25-b931-6ea438aede2a\n\n\t/// Revenue from store sales channel only, based on extended sales price\n\tmeasure 'Store Revenue' = SUM('store_sales'[ss_ext_sales_price])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 01. Revenue\n\t\tlineageTag: 1755e841-2eeb-4e1e-9a98-c860eb583a79\n\n\t/// Total units sold through catalog sales channel\n\tmeasure 'Catalog Sales Quantity' = SUM('catalog_sales'[cs_quantity])\n\t\tformatString: #,0\n\t\tdisplayFolder: 02. Quantity\n\t\tlineageTag: 6da96e65-1eb2-4e76-afdb-50d9106f56c1\n\n\t/// Net profit from store sales channel only\n\tmeasure 'Store Net Profit' = SUM('store_sales'[ss_net_profit])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 03. Profit\n\t\tlineageTag: a5a04305-c986-4791-925e-b9115939e453\n\n\t/// Number of unique customers who made store purchases\n\tmeasure 'Store Distinct Customers' = DISTINCTCOUNT('store_sales'[ss_customer_sk])\n\t\tformatString: #,0\n\t\tdisplayFolder: 04. Distinct Counts\n\t\tlineageTag: ace0485f-176a-4401-bb53-41c99ce5c356\n\n\t/// Revenue for the same period in the previous year\n\tmeasure 'Store Revenue Same Period LY' =\n\t\t\t\n\t\t\tCALCULATE(\n\t\t\t    [Store Revenue],\n\t\t\t    SAMEPERIODLASTYEAR('tpcds_calendar')\n\t\t\t)\n\t\tformatString: $#,0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 303dbdc1-0484-496c-b14f-a41b0ef53f82\n\n\t/// Revenue for the same period in the previous year\n\tmeasure 'Store Revenue YoY' =\n\t\t\tVAR CurrentYearRev = [Store Revenue]\n\t\t\tVAR PreviousYearRev = [Store Revenue Same Period LY]\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentYearRev - PreviousYearRev, PreviousYearRev, 0)\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 39648465-7aac-4652-8c2f-d6599df15204\n\n\t/// Catalog Sales Same Period LY\n\tmeasure 'Catalog Sales Same Period LY' =\n\t\t\t\n\t\t\tCALCULATE(\n\t\t\t    [Catalog Sales Quantity],\n\t\t\t    SAMEPERIODLASTYEAR('tpcds_calendar')\n\t\t\t)\n\t\tformatString: 0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: d1d5179d-688b-487c-8638-81780d95eb05\n\n\t/// Catalog Sales YoY\n\tmeasure 'Catalog Sales YoY' =\n\t\t\tVAR CurrentYearCatSales = [Catalog Sales Quantity]\n\t\t\tVAR PreviousYearCatSales = [Catalog Sales Same Period LY]\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentYearCatSales - PreviousYearCatSales, PreviousYearCatSales, 0)\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 0bc09d6d-1bdd-4f7e-ac26-a12e87ccf67d\n\n\t/// Store Profit % by Item Category\n\tmeasure 'Store Profit % by Item Category' = ```\n\t\t\t\n\t\t\tVAR CurrentProfit = [Store Net Profit]\n\t\t\tVAR TotalProfitAllCategories = \n\t\t\t    CALCULATE(\n\t\t\t        [Store Net Profit],\n\t\t\t        ALLEXCEPT(\n\t\t\t            'item',\n\t\t\t            'item'[i_brand],\n\t\t\t            'item'[i_manufact]\n\t\t\t        )\n\t\t\t    )\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentProfit, TotalProfitAllCategories, 0)\n\t\t\t```\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 05. Advanced % Share\n\t\tlineageTag: 04f417ef-ba48-4d9e-873d-76d516ec3a54\n\n\t/// Total revenue from beginning of year to current date selection\n\tmeasure 'Store Revenue YTD' = TOTALYTD([Store Revenue],'tpcds_calendar')\n\t\tformatString: $#,0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 0e0dbb36-1e75-4c77-8b7f-9a462a143dcb\n\n\t/// Dummy\n\tcolumn Dummy\n\t\tformatString: 0\n\t\tlineageTag: c1dfb904-84a8-4dfc-a768-d5e10f70f255\n\t\tsummarizeBy: none\n\t\tisNameInferred\n\t\tsourceColumn: [Dummy]\n\n\tpartition 'Measures 1' = calculated\n\t\tmode: import\n\t\tsource = ROW(\"Dummy\", 1)\n\n"},{"path":"definition/tables/Time Unit.tmdl","text":"/// Field parameter table for dynamic time unit selection in reports, supporting Year and Quarter groupings.\ntable 'Time Unit'\n\tlineageTag: c100712f-ebcb-4d0f-a435-e1a4c8ce3229\n\n\t/// Display name for the time unit selection (Year, Quarter).\n\tcolumn 'Time Unit'\n\t\tlineageTag: bd5d5ab0-6269-4df0-bd39-9616f6e2b7d5\n\t\tsummarizeBy: none\n\t\tsourceColumn: [Value1]\n\t\tsortByColumn: 'Time Unit Order'\n\n\t\trelatedColumnDetails\n\t\t\tgroupByColumn: 'Time Unit Fields'\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// DAX column reference for the selected time unit.\n\tcolumn 'Time Unit Fields'\n\t\tisHidden\n\t\tlineageTag: 4be3ced6-f4db-40c4-84b2-5ef9decf7fee\n\t\tsummarizeBy: none\n\t\tsourceColumn: [Value2]\n\t\tsortByColumn: 'Time Unit Order'\n\n\t\textendedProperty ParameterMetadata =\n\t\t\t\t{\n\t\t\t\t  \"version\": 3,\n\t\t\t\t  \"kind\": 2\n\t\t\t\t}\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Sort order for time unit options in field parameter.\n\tcolumn 'Time Unit Order'\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 36566b71-2a4f-4ddb-86c0-b4c7b3d8ab7f\n\t\tsummarizeBy: sum\n\t\tsourceColumn: [Value3]\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition 'Time Unit' = calculated\n\t\tmode: import\n\t\tsource =\n\t\t\t\t{\n\t\t\t\t    (\"Year\", NAMEOF('date_dim'[d_year]), 0),\n\t\t\t\t    (\"Quarter\", NAMEOF('date_dim'[d_quarter_name]), 1)\n\t\t\t\t}\n\n\tannotation PBI_Id = 706957f972f8497896950148d2d5f0af\n\n"},{"path":"definition/tables/catalog_page.tmdl","text":"/// Catalog page dimension containing information about catalog pages used in catalog sales campaigns.\ntable catalog_page\n\tlineageTag: b4e5474e-8b18-42b8-9a56-9c2416950a0a\n\tsourceLineageTag: [dbo].[catalog_page]\n\n\t/// Catalog page surrogate key - unique identifier for each catalog page.\n\tcolumn cp_catalog_page_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 71a9dba7-e582-461e-b8de-f37322137084\n\t\tsourceLineageTag: cp_catalog_page_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: cp_catalog_page_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type or category of the catalog page.\n\tcolumn cp_type\n\t\tdataType: string\n\t\tlineageTag: bdf42608-dc88-489d-90ce-24a87bc40378\n\t\tsourceLineageTag: cp_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: cp_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition catalog_page = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: catalog_page\n\t\t\tschemaName: tpcds_sf__SF___clustersn\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/catalog_sales.tmdl","text":"/// Fact table containing catalog sales transactions with pricing, quantities, profits, and foreign keys to related dimensions.\ntable catalog_sales\n\tlineageTag: c38e1686-b3a0-473b-8604-8efe0410ea48\n\tsourceLineageTag: [dbo].[catalog_sales]\n\n\t/// Foreign key to date dimension for sale date.\n\tcolumn cs_sold_date_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: ac2ce865-caed-4039-bda1-a182c516cfdd\n\t\tsourceLineageTag: cs_sold_date_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_sold_date_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension for billing customer.\n\tcolumn cs_bill_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 310a5931-6249-4ad7-86b4-7aca5fbc39a9\n\t\tsourceLineageTag: cs_bill_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_bill_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension for billing address.\n\tcolumn cs_bill_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 51e6cbcf-e98d-4ea9-865e-a500419bec99\n\t\tsourceLineageTag: cs_bill_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_bill_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension for shipping customer.\n\tcolumn cs_ship_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4a433af2-d889-41d8-9c6f-1f05f94f0070\n\t\tsourceLineageTag: cs_ship_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension for shipping address.\n\tcolumn cs_ship_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 25b41390-6ff3-4f15-b0e6-f26a3da61ef5\n\t\tsourceLineageTag: cs_ship_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to catalog page dimension.\n\tcolumn cs_catalog_page_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: b01fc3a0-0a51-4660-a5d3-56d28ea386cd\n\t\tsourceLineageTag: cs_catalog_page_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_catalog_page_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to ship mode dimension.\n\tcolumn cs_ship_mode_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 0a85820a-e8f9-41f9-b50e-9610890408e8\n\t\tsourceLineageTag: cs_ship_mode_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_mode_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension.\n\tcolumn cs_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9024dff9-9629-4f0e-abae-0b29f00d568d\n\t\tsourceLineageTag: cs_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to promotion dimension.\n\tcolumn cs_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: dbfb7b81-9eef-4b86-aac0-a48cc36d0168\n\t\tsourceLineageTag: cs_promo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Catalog order number for this transaction.\n\tcolumn cs_order_number\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: ccec0342-6144-411f-bc2d-90234bcfb5ac\n\t\tsourceLineageTag: cs_order_number\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_order_number\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quantity of items ordered in this transaction.\n\tcolumn cs_quantity\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 0f5abf99-45b4-495b-bad3-795d0e8e5802\n\t\tsourceLineageTag: cs_quantity\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_quantity\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Wholesale cost per unit for this transaction.\n\tcolumn cs_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 37d760e7-f755-42e9-9dba-3a28cc211ae7\n\t\tsourceLineageTag: cs_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// List price per unit at time of order.\n\tcolumn cs_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 0eda1627-f565-444d-92e5-1a5bb0929e4d\n\t\tsourceLineageTag: cs_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Actual sales price per unit (after discounts).\n\tcolumn cs_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: eebce86e-da48-4799-8f93-1a8e69de6605\n\t\tsourceLineageTag: cs_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended sales price (sales price \u00d7 quantity).\n\tcolumn cs_ext_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: f0c74fde-bf40-47e4-9352-6bc4d4f219a0\n\t\tsourceLineageTag: cs_ext_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_ext_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended list price (list price \u00d7 quantity).\n\tcolumn cs_ext_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 845b8a06-567d-49c0-9455-1a469d81f646\n\t\tsourceLineageTag: cs_ext_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_ext_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Net profit for this transaction (sales price - wholesale cost).\n\tcolumn cs_net_profit\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 341a4a24-e726-4b20-8a50-967f689a5d01\n\t\tsourceLineageTag: cs_net_profit\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_net_profit\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Technical column used for cache invalidation in DirectLake mode.\n\tcolumn cache_buster\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 33947ffc-dc12-42ab-af62-5b00c42adf7a\n\t\tsourceLineageTag: cache_buster\n\t\tsummarizeBy: none\n\t\tsourceColumn: cache_buster\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition catalog_sales = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: catalog_sales\n\t\t\tschemaName: tpcds_sf__SF___clustersn\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/customer_address.tmdl","text":"/// Customer address dimension containing geographic information for billing and shipping addresses.\ntable customer_address\n\tlineageTag: 75246b34-d9f4-46fd-813f-bc5f0e47ab8c\n\tsourceLineageTag: [dbo].[customer_address]\n\n\t/// Customer address surrogate key - unique identifier for each address.\n\tcolumn ca_address_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9084b5d7-d188-4fb9-9c92-211d5b68f397\n\t\tsourceLineageTag: ca_address_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_address_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// City name for the customer address.\n\tcolumn ca_city\n\t\tdataType: string\n\t\tlineageTag: 03b2fb94-a08c-4010-9157-b91edcb60daf\n\t\tsourceLineageTag: ca_city\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_city\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// County name for the customer address.\n\tcolumn ca_county\n\t\tdataType: string\n\t\tlineageTag: 0ef7c0d6-1e17-42c5-a9bb-fb014c91398a\n\t\tsourceLineageTag: ca_county\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_county\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// State or province for the customer address.\n\tcolumn ca_state\n\t\tdataType: string\n\t\tlineageTag: 24c6ee20-e4ab-4ca3-aa30-934ba66a55db\n\t\tsourceLineageTag: ca_state\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_state\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Postal code for the customer address.\n\tcolumn ca_zip\n\t\tdataType: string\n\t\tlineageTag: a57bf4fe-e3de-4848-81bd-e71dd885c99f\n\t\tsourceLineageTag: ca_zip\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_zip\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type of location (e.g., residential, commercial).\n\tcolumn ca_location_type\n\t\tdataType: string\n\t\tlineageTag: 9a90fb17-4314-40df-84ae-071cdd4eb3c7\n\t\tsourceLineageTag: ca_location_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_location_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition customer_address = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: customer_address\n\t\t\tschemaName: tpcds_sf__SF___clustersn\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/customer_demographics.tmdl","text":"/// Customer demographics dimension containing education status and marital status attributes for customer segmentation.\ntable customer_demographics\n\tlineageTag: 52f8f268-90f3-4f40-a527-07a74d9e9baf\n\tsourceLineageTag: [dbo].[customer_demographics]\n\n\t/// Customer demographics surrogate key - unique identifier for each demographic profile.\n\tcolumn cd_demo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: d2cad4d0-4a88-41ab-b7ee-94ab73973694\n\t\tsourceLineageTag: cd_demo_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_demo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Marital status of the customer.\n\tcolumn cd_marital_status\n\t\tdataType: string\n\t\tlineageTag: e29a3681-2c0d-4579-9b35-5112d86c0f46\n\t\tsourceLineageTag: cd_marital_status\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_marital_status\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Education level of the customer.\n\tcolumn cd_education_status\n\t\tdataType: string\n\t\tlineageTag: 43260b21-0847-455f-a853-e2233af2a62e\n\t\tsourceLineageTag: cd_education_status\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_education_status\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition customer_demographics = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: customer_demographics\n\t\t\tschemaName: tpcds_sf__SF___clustersn\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/date_dim.tmdl","text":"/// Date dimension providing calendar hierarchy with year, quarter, month, and day-of-week attributes for time-based analysis.\ntable date_dim\n\tlineageTag: 9e7ec2de-382c-4a3b-9589-1ece445bfa27\n\tsourceLineageTag: [dbo].[date_dim]\n\n\t/// Date surrogate key - unique identifier for each calendar date.\n\tcolumn d_date_sk_1\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 63841738-6970-46ac-8106-970384b1052b\n\t\tsourceLineageTag: d_date_sk_1\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_date_sk_1\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Calendar date value.\n\tcolumn d_date\n\t\tdataType: dateTime\n\t\tformatString: General Date\n\t\tlineageTag: dc08a5e6-1a30-4d5f-9b03-1c98c7bac892\n\t\tsourceLineageTag: d_date\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_date\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Calendar year (e.g., 2023).\n\tcolumn d_year\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: fcf4b9e9-c880-4ee0-907c-3a18951b0809\n\t\tsourceLineageTag: d_year\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_year\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Day of week (1=Sunday, 7=Saturday).\n\tcolumn d_dow\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 339ac74d-ab31-4d4e-93eb-524b9f7e211c\n\t\tsourceLineageTag: d_dow\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_dow\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Month of year (1-12).\n\tcolumn d_moy\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: faa3deb0-9def-4334-bd42-a1b9d60b51b7\n\t\tsourceLineageTag: d_moy\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_moy\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Day of month (1-31).\n\tcolumn d_dom\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 83cd6ebd-7811-4616-a884-9930f66f8bfe\n\t\tsourceLineageTag: d_dom\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_dom\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quarter of year (1-4).\n\tcolumn d_qoy\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 4962ee0a-fcb9-4bf8-a5c7-43a66a94e67f\n\t\tsourceLineageTag: d_qoy\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_qoy\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quarter name (e.g., 'Q1 2023').\n\tcolumn d_quarter_name\n\t\tdataType: string\n\t\tlineageTag: 2426fa03-107c-4f51-a0b4-c59bb4b85d4f\n\t\tsourceLineageTag: d_quarter_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_quarter_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition date_dim = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: date_dim\n\t\t\tschemaName: tpcds_sf__SF___clustersn\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n\tcalendar tpcds_calendar\n\t\tlineageTag: dd529b7a-99fe-4cb5-928c-df1abed499f1\n\n\t\tcalendarColumnGroup = year\n\t\t\tprimaryColumn: d_year\n\n\t\tcalendarColumnGroup = quarter\n\t\t\tprimaryColumn: d_quarter_name\n\n\t\tcalendarColumnGroup = quarterOfYear\n\t\t\tprimaryColumn: d_qoy\n\n\t\tcalendarColumnGroup = date\n\t\t\tprimaryColumn: d_date\n\n"},{"path":"definition/tables/item.tmdl","text":"/// Product dimension containing item details such as brand, category, class, color, pricing, and manufacturing information.\ntable item\n\tlineageTag: 0f958ae1-af31-4b29-87a3-7e9ee589887e\n\tsourceLineageTag: [dbo].[item]\n\n\t/// Item surrogate key - unique identifier for each product.\n\tcolumn i_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 23893369-a589-4284-afa3-b1e291864600\n\t\tsourceLineageTag: i_item_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Current retail price of the product.\n\tcolumn i_current_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 70b27235-eefb-4732-8e0d-64c9576f0776\n\t\tsourceLineageTag: i_current_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_current_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Wholesale cost paid for the product.\n\tcolumn i_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 90cbdd03-df2f-4439-941d-de295ceb5135\n\t\tsourceLineageTag: i_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Brand name of the product.\n\tcolumn i_brand\n\t\tdataType: string\n\t\tlineageTag: 0c701039-61ca-4f5f-872b-aab6677ef45c\n\t\tsourceLineageTag: i_brand\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_brand\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Product class or subcategory within the main category.\n\tcolumn i_class\n\t\tdataType: string\n\t\tlineageTag: 5bdf127c-d558-46b2-98e6-0e294148d466\n\t\tsourceLineageTag: i_class\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_class\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Product category classification.\n\tcolumn i_category\n\t\tdataType: string\n\t\tlineageTag: 138de914-55e9-4a46-b5b9-6707b88a38e9\n\t\tsourceLineageTag: i_category\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_category\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Manufacturer or producer of the product.\n\tcolumn i_manufact\n\t\tdataType: string\n\t\tlineageTag: e6852c2b-2bc8-4fb1-86cb-d7ebe7c18637\n\t\tsourceLineageTag: i_manufact\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_manufact\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Size specification of the product.\n\tcolumn i_size\n\t\tdataType: string\n\t\tlineageTag: 0aeb0963-bc02-4389-a3fd-ab7232caaf9c\n\t\tsourceLineageTag: i_size\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_size\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Primary color of the product.\n\tcolumn i_color\n\t\tdataType: string\n\t\tlineageTag: d6709cb2-852f-485a-a0de-8cdd07941038\n\t\tsourceLineageTag: i_color\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_color\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition item = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: item\n\t\t\tschemaName: tpcds_sf__SF___clustersn\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/promotion.tmdl","text":"/// Promotion dimension containing promotional campaign details including costs and promotional names.\ntable promotion\n\tlineageTag: 5942c68a-d07a-4a23-8864-b2c733be46c8\n\tsourceLineageTag: [dbo].[promotion]\n\n\t/// Promotion surrogate key - unique identifier for each promotional campaign.\n\tcolumn p_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 0299ca0a-f58f-4ebe-b752-ffa6b63f9760\n\t\tsourceLineageTag: p_promo_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension (for item-specific promotions).\n\tcolumn p_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 92f8bc42-df79-4a35-8a37-10a02a937c40\n\t\tsourceLineageTag: p_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: p_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Cost associated with running this promotional campaign.\n\tcolumn p_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: c42871df-1b39-4b1e-83cb-142e6864a1e0\n\t\tsourceLineageTag: p_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Name or title of the promotional campaign.\n\tcolumn p_promo_name\n\t\tdataType: string\n\t\tlineageTag: 6ceed2f6-4e12-4edc-b161-8251a356ec17\n\t\tsourceLineageTag: p_promo_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_promo_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition promotion = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: promotion\n\t\t\tschemaName: tpcds_sf__SF___clustersn\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/ship_mode.tmdl","text":"/// Shipping method dimension containing carrier information and shipping type classifications.\ntable ship_mode\n\tlineageTag: db1b87b5-f584-4f30-b70e-0eb00c8b45b8\n\tsourceLineageTag: [dbo].[ship_mode]\n\n\t/// Ship mode surrogate key - unique identifier for each shipping method.\n\tcolumn sm_ship_mode_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4c485c32-fa22-4df6-b4a8-a8df04e2f730\n\t\tsourceLineageTag: sm_ship_mode_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_ship_mode_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type or category of shipping method.\n\tcolumn sm_type\n\t\tdataType: string\n\t\tlineageTag: c49fc6f8-d030-49a2-8105-6a45d1a81ac2\n\t\tsourceLineageTag: sm_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Internal code for the shipping method.\n\tcolumn sm_code\n\t\tdataType: string\n\t\tlineageTag: f54a092d-e847-4cee-84e6-5b813cbf5280\n\t\tsourceLineageTag: sm_code\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_code\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Shipping carrier or company name.\n\tcolumn sm_carrier\n\t\tdataType: string\n\t\tlineageTag: aca70983-6b51-4291-83f8-77fac2e381fe\n\t\tsourceLineageTag: sm_carrier\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_carrier\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition ship_mode = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: ship_mode\n\t\t\tschemaName: tpcds_sf__SF___clustersn\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/store.tmdl","text":"/// Store dimension containing retail location information including geographic details, management, and tax rates.\ntable store\n\tlineageTag: 6835d4a1-f40a-40aa-8a65-98aa4e06a406\n\tsourceLineageTag: [dbo].[store]\n\n\t/// Store surrogate key - unique identifier for each store location.\n\tcolumn s_store_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 14502476-8801-4068-b7c4-875ce7cac939\n\t\tsourceLineageTag: s_store_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_store_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Store name or identifier for the retail location.\n\tcolumn s_store_name\n\t\tdataType: string\n\t\tlineageTag: 4e78d335-4974-4d41-8402-ea89f63f9005\n\t\tsourceLineageTag: s_store_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_store_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Name of the store manager.\n\tcolumn s_manager\n\t\tdataType: string\n\t\tlineageTag: 3a8ddb5b-2ada-42d0-9ef9-89cd96b435b6\n\t\tsourceLineageTag: s_manager\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_manager\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Name of the market manager overseeing this store.\n\tcolumn s_market_manager\n\t\tdataType: string\n\t\tlineageTag: 37c8562d-3ab6-47ad-9b99-2ad1d72e6cd6\n\t\tsourceLineageTag: s_market_manager\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_market_manager\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// City where the store is located.\n\tcolumn s_city\n\t\tdataType: string\n\t\tlineageTag: a1a9209c-8194-4f96-8e94-69e545e064ba\n\t\tsourceLineageTag: s_city\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_city\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// County where the store is located.\n\tcolumn s_county\n\t\tdataType: string\n\t\tlineageTag: e219113c-fe81-45b0-b6be-f02e7ce75403\n\t\tsourceLineageTag: s_county\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_county\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// State or province where the store is located.\n\tcolumn s_state\n\t\tdataType: string\n\t\tlineageTag: fbaeba14-56ed-496d-a004-83f851771750\n\t\tsourceLineageTag: s_state\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_state\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Postal code for the store location.\n\tcolumn s_zip\n\t\tdataType: string\n\t\tlineageTag: 44e7ea79-56d6-4d20-8243-92c35919dcc3\n\t\tsourceLineageTag: s_zip\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_zip\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Tax rate percentage applied at this store location.\n\tcolumn s_tax_percentage\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: ba6d903d-f426-46b7-90a8-756a18261f2a\n\t\tsourceLineageTag: s_tax_percentage\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_tax_percentage\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\tpartition store = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: store\n\t\t\tschemaName: tpcds_sf__SF___clustersn\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/store_sales.tmdl","text":"/// Fact table containing retail store sales transactions with pricing, quantities, profits, and foreign keys to related dimensions.\ntable store_sales\n\tlineageTag: 6c7a2caf-3c22-4f38-a6aa-895a77e2e1c9\n\tsourceLineageTag: [dbo].[store_sales]\n\n\t/// Foreign key to date dimension for sale date.\n\tcolumn ss_sold_date_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4e30d94a-216f-4eaf-9858-fb22c6e6d57e\n\t\tsourceLineageTag: ss_sold_date_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_sold_date_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension.\n\tcolumn ss_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 2b151014-7df5-45f0-9907-03cbe191952b\n\t\tsourceLineageTag: ss_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Customer surrogate key for the transaction.\n\tcolumn ss_customer_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 558da56d-ea73-4562-b35c-ff94e774cca3\n\t\tsourceLineageTag: ss_customer_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_customer_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension.\n\tcolumn ss_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: e65f473f-bb26-4c7d-90f1-139de12a51ab\n\t\tsourceLineageTag: ss_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension.\n\tcolumn ss_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: cb65f3a1-6240-4baa-bfb2-4b24d39e2e87\n\t\tsourceLineageTag: ss_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to store dimension.\n\tcolumn ss_store_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 809a4a04-3bbf-4067-b54f-ee4d5fbbf6d4\n\t\tsourceLineageTag: ss_store_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_store_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to promotion dimension.\n\tcolumn ss_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9a64bbfb-0ae6-4919-b4e9-73e6d90353b3\n\t\tsourceLineageTag: ss_promo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Transaction ticket or receipt number.\n\tcolumn ss_ticket_number\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: d8ce37f4-a12c-4558-b131-66631546b607\n\t\tsourceLineageTag: ss_ticket_number\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ticket_number\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quantity of items sold in this transaction.\n\tcolumn ss_quantity\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 318e6a7b-88a2-413c-8257-315f66313d79\n\t\tsourceLineageTag: ss_quantity\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_quantity\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Wholesale cost per unit for this transaction.\n\tcolumn ss_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: d02100cc-2804-43d7-b752-8e98d579e202\n\t\tsourceLineageTag: ss_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// List price per unit at time of sale.\n\tcolumn ss_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 4f32f293-c19e-497e-906e-089bc204cde9\n\t\tsourceLineageTag: ss_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Actual sales price per unit (after discounts).\n\tcolumn ss_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: e620a719-f6e0-4965-8326-f4a2d255999b\n\t\tsourceLineageTag: ss_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended sales price (sales price \u00d7 quantity).\n\tcolumn ss_ext_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 8b66819a-a151-4fc7-9ec0-cacf7b6e8e01\n\t\tsourceLineageTag: ss_ext_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ext_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended list price (list price \u00d7 quantity).\n\tcolumn ss_ext_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: bc8e9c57-98e2-4032-a3fd-69808e3cdd8b\n\t\tsourceLineageTag: ss_ext_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ext_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Net profit for this transaction (sales price - wholesale cost).\n\tcolumn ss_net_profit\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: d2d6233f-6944-40b5-97d7-2870da393d1b\n\t\tsourceLineageTag: ss_net_profit\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_net_profit\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Technical column used for cache invalidation in DirectLake mode.\n\tcolumn cache_buster\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 7ae60894-bf63-4fef-a327-bf8796466374\n\t\tsourceLineageTag: cache_buster\n\t\tsummarizeBy: none\n\t\tsourceColumn: cache_buster\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition store_sales = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: store_sales\n\t\t\tschemaName: tpcds_sf__SF___clustersn\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition.pbism","text":"{\"$schema\": \"https://developer.microsoft.com/json-schemas/fabric/item/semanticModel/definitionProperties/1.0.0/schema.json\", \"version\": \"5.0\", \"settings\": {}}"}]},"partition":{"model":"tpcds_sf__SF___partition","parts":[{"path":"definition/database.tmdl","text":"database tpcds_sf__SF___partition\n\tcompatibilityLevel: 1702\n\tcompatibilityMode: powerBI\n\tlanguage: 1033\n\n"},{"path":"definition/expressions.tmdl","text":"expression 'DirectLake - tpcds_sf__SF__' =\n\t\tlet\n\t\t    Source = AzureStorage.DataLake(\"https://onelake.dfs.fabric.microsoft.com/51650f82-6bb5-4023-b0ab-db197d32e0be/01b539f3-4a9d-45ef-b1ef-0ba59552eb21\", [HierarchicalNavigation=true])\n\t\tin\n\t\t    Source\n\tlineageTag: c6e3390d-cc84-4e13-ab93-53308a346336\n\n\tannotation PBI_IncludeFutureArtifacts = False\n\n"},{"path":"definition/model.tmdl","text":"model Model\n\tdirectLakeBehavior: directLakeOnly\n\tculture: en-US\n\tdefaultPowerBIDataSourceVersion: powerBI_V3\n\tsourceQueryCulture: en-US\n\tdataAccessOptions\n\t\tlegacyRedirects\n\t\treturnErrorValuesAsNull\n\nannotation PBI_QueryOrder = [\"DirectLake - tpcds_sf__SF__\"]\n\nannotation __PBI_TimeIntelligenceEnabled = 1\n\nannotation __LastRPTime = 134272970581620514\n\nannotation PBI_ProTooling = [\"DirectLakeOnOneLakeInWeb\",\"WebModelingEdit\"]\n\nannotation __TEdtr = 1\n\nannotation TabularEditor_SerializeOptions = {\"IgnoreInferredObjects\":true,\"IgnoreInferredProperties\":true,\"IgnoreTimestamps\":true,\"SplitMultilineStrings\":true,\"PrefixFilenames\":false,\"LocalTranslations\":true,\"LocalPerspectives\":true,\"LocalRelationships\":true,\"Levels\":[\"Data Sources\",\"Perspectives\",\"Relationships\",\"Roles\",\"Shared Expressions\",\"Tables\",\"Tables/Calculation Items\",\"Tables/Columns\",\"Tables/Hierarchies\",\"Tables/Measures\",\"Tables/Partitions\",\"Translations\"]}\n\nref table store\nref table item\nref table date_dim\nref table store_sales\nref table catalog_page\nref table promotion\nref table ship_mode\nref table catalog_sales\nref table customer_address\nref table customer_demographics\nref table 'Measures 1'\nref table 'Time Unit'\n\n"},{"path":"definition/relationships.tmdl","text":"relationship b3bd8d91-ae6b-4489-83cd-34ca2c3213d8\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_bill_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship b727abac-57ca-484a-8183-1b4c10921d84\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_bill_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship 41e21761-05b3-410b-960e-54805fb2c4f2\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_catalog_page_sk\n\ttoColumn: catalog_page.cp_catalog_page_sk\n\nrelationship c7ce7218-d702-40d6-94b4-80009e453fe9\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship 4d02a5f5-e67a-4f65-a203-192e1c2ef166\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_promo_sk\n\ttoColumn: promotion.p_promo_sk\n\nrelationship 545e95f8-e63b-4a27-9c91-3cd725ca10e0\n\tisActive: false\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship 685424a1-f6c7-42d6-8f44-3cccf5f08db4\n\tisActive: false\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship 2e0c0f18-79bf-48f8-be1a-2c53cc33e257\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_mode_sk\n\ttoColumn: ship_mode.sm_ship_mode_sk\n\nrelationship c5edc5f8-f418-44a4-a8aa-caff6e0bef29\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_sold_date_sk\n\ttoColumn: date_dim.d_date_sk_1\n\nrelationship 9f8c8977-b8ac-4985-aed9-7711ce5004af\n\tisActive: false\n\tfromColumn: promotion.p_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship f824425a-7dd8-4a70-9ee5-2042a74ed97d\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship be9c6d5e-0bed-4db1-8558-326d5dfba951\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship f172b575-b3ce-48a4-b264-2928516104a2\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship 95b347fa-cf7e-4885-8124-86eb2ff56351\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_promo_sk\n\ttoColumn: promotion.p_promo_sk\n\nrelationship c782badb-be54-4423-98b0-22eebad6aa29\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_sold_date_sk\n\ttoColumn: date_dim.d_date_sk_1\n\nrelationship 9e5ca2d3-a5aa-4d43-b00a-07e4a857b1e4\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_store_sk\n\ttoColumn: store.s_store_sk\n\n"},{"path":"definition/tables/Measures 1.tmdl","text":"/// Calculated measures table containing key business metrics for revenue, quantity, profit, tax, and performance analysis across store and catalog channels.\ntable 'Measures 1'\n\tlineageTag: ae7cf165-f530-4f98-a4c5-befd958ff771\n\n\t/// Revenue from catalog sales channel only, based on extended sales price\n\tmeasure 'Catalog Revenue' = SUM('catalog_sales'[cs_ext_sales_price])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 01. Revenue\n\t\tlineageTag: bed34121-c406-4e25-b931-6ea438aede2a\n\n\t/// Revenue from store sales channel only, based on extended sales price\n\tmeasure 'Store Revenue' = SUM('store_sales'[ss_ext_sales_price])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 01. Revenue\n\t\tlineageTag: 1755e841-2eeb-4e1e-9a98-c860eb583a79\n\n\t/// Total units sold through catalog sales channel\n\tmeasure 'Catalog Sales Quantity' = SUM('catalog_sales'[cs_quantity])\n\t\tformatString: #,0\n\t\tdisplayFolder: 02. Quantity\n\t\tlineageTag: 6da96e65-1eb2-4e76-afdb-50d9106f56c1\n\n\t/// Net profit from store sales channel only\n\tmeasure 'Store Net Profit' = SUM('store_sales'[ss_net_profit])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 03. Profit\n\t\tlineageTag: a5a04305-c986-4791-925e-b9115939e453\n\n\t/// Number of unique customers who made store purchases\n\tmeasure 'Store Distinct Customers' = DISTINCTCOUNT('store_sales'[ss_customer_sk])\n\t\tformatString: #,0\n\t\tdisplayFolder: 04. Distinct Counts\n\t\tlineageTag: ace0485f-176a-4401-bb53-41c99ce5c356\n\n\t/// Revenue for the same period in the previous year\n\tmeasure 'Store Revenue Same Period LY' =\n\t\t\t\n\t\t\tCALCULATE(\n\t\t\t    [Store Revenue],\n\t\t\t    SAMEPERIODLASTYEAR('tpcds_calendar')\n\t\t\t)\n\t\tformatString: $#,0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 303dbdc1-0484-496c-b14f-a41b0ef53f82\n\n\t/// Revenue for the same period in the previous year\n\tmeasure 'Store Revenue YoY' =\n\t\t\tVAR CurrentYearRev = [Store Revenue]\n\t\t\tVAR PreviousYearRev = [Store Revenue Same Period LY]\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentYearRev - PreviousYearRev, PreviousYearRev, 0)\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 39648465-7aac-4652-8c2f-d6599df15204\n\n\t/// Catalog Sales Same Period LY\n\tmeasure 'Catalog Sales Same Period LY' =\n\t\t\t\n\t\t\tCALCULATE(\n\t\t\t    [Catalog Sales Quantity],\n\t\t\t    SAMEPERIODLASTYEAR('tpcds_calendar')\n\t\t\t)\n\t\tformatString: 0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: d1d5179d-688b-487c-8638-81780d95eb05\n\n\t/// Catalog Sales YoY\n\tmeasure 'Catalog Sales YoY' =\n\t\t\tVAR CurrentYearCatSales = [Catalog Sales Quantity]\n\t\t\tVAR PreviousYearCatSales = [Catalog Sales Same Period LY]\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentYearCatSales - PreviousYearCatSales, PreviousYearCatSales, 0)\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 0bc09d6d-1bdd-4f7e-ac26-a12e87ccf67d\n\n\t/// Store Profit % by Item Category\n\tmeasure 'Store Profit % by Item Category' = ```\n\t\t\t\n\t\t\tVAR CurrentProfit = [Store Net Profit]\n\t\t\tVAR TotalProfitAllCategories = \n\t\t\t    CALCULATE(\n\t\t\t        [Store Net Profit],\n\t\t\t        ALLEXCEPT(\n\t\t\t            'item',\n\t\t\t            'item'[i_brand],\n\t\t\t            'item'[i_manufact]\n\t\t\t        )\n\t\t\t    )\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentProfit, TotalProfitAllCategories, 0)\n\t\t\t```\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 05. Advanced % Share\n\t\tlineageTag: 04f417ef-ba48-4d9e-873d-76d516ec3a54\n\n\t/// Total revenue from beginning of year to current date selection\n\tmeasure 'Store Revenue YTD' = TOTALYTD([Store Revenue],'tpcds_calendar')\n\t\tformatString: $#,0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 0e0dbb36-1e75-4c77-8b7f-9a462a143dcb\n\n\t/// Dummy\n\tcolumn Dummy\n\t\tformatString: 0\n\t\tlineageTag: c1dfb904-84a8-4dfc-a768-d5e10f70f255\n\t\tsummarizeBy: none\n\t\tisNameInferred\n\t\tsourceColumn: [Dummy]\n\n\tpartition 'Measures 1' = calculated\n\t\tmode: import\n\t\tsource = ROW(\"Dummy\", 1)\n\n"},{"path":"definition/tables/Time Unit.tmdl","text":"/// Field parameter table for dynamic time unit selection in reports, supporting Year and Quarter groupings.\ntable 'Time Unit'\n\tlineageTag: c100712f-ebcb-4d0f-a435-e1a4c8ce3229\n\n\t/// Display name for the time unit selection (Year, Quarter).\n\tcolumn 'Time Unit'\n\t\tlineageTag: bd5d5ab0-6269-4df0-bd39-9616f6e2b7d5\n\t\tsummarizeBy: none\n\t\tsourceColumn: [Value1]\n\t\tsortByColumn: 'Time Unit Order'\n\n\t\trelatedColumnDetails\n\t\t\tgroupByColumn: 'Time Unit Fields'\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// DAX column reference for the selected time unit.\n\tcolumn 'Time Unit Fields'\n\t\tisHidden\n\t\tlineageTag: 4be3ced6-f4db-40c4-84b2-5ef9decf7fee\n\t\tsummarizeBy: none\n\t\tsourceColumn: [Value2]\n\t\tsortByColumn: 'Time Unit Order'\n\n\t\textendedProperty ParameterMetadata =\n\t\t\t\t{\n\t\t\t\t  \"version\": 3,\n\t\t\t\t  \"kind\": 2\n\t\t\t\t}\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Sort order for time unit options in field parameter.\n\tcolumn 'Time Unit Order'\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 36566b71-2a4f-4ddb-86c0-b4c7b3d8ab7f\n\t\tsummarizeBy: sum\n\t\tsourceColumn: [Value3]\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition 'Time Unit' = calculated\n\t\tmode: import\n\t\tsource =\n\t\t\t\t{\n\t\t\t\t    (\"Year\", NAMEOF('date_dim'[d_year]), 0),\n\t\t\t\t    (\"Quarter\", NAMEOF('date_dim'[d_quarter_name]), 1)\n\t\t\t\t}\n\n\tannotation PBI_Id = 706957f972f8497896950148d2d5f0af\n\n"},{"path":"definition/tables/catalog_page.tmdl","text":"/// Catalog page dimension containing information about catalog pages used in catalog sales campaigns.\ntable catalog_page\n\tlineageTag: b4e5474e-8b18-42b8-9a56-9c2416950a0a\n\tsourceLineageTag: [dbo].[catalog_page]\n\n\t/// Catalog page surrogate key - unique identifier for each catalog page.\n\tcolumn cp_catalog_page_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 71a9dba7-e582-461e-b8de-f37322137084\n\t\tsourceLineageTag: cp_catalog_page_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: cp_catalog_page_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type or category of the catalog page.\n\tcolumn cp_type\n\t\tdataType: string\n\t\tlineageTag: bdf42608-dc88-489d-90ce-24a87bc40378\n\t\tsourceLineageTag: cp_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: cp_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition catalog_page = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: catalog_page\n\t\t\tschemaName: tpcds_sf__SF___partition\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/catalog_sales.tmdl","text":"/// Fact table containing catalog sales transactions with pricing, quantities, profits, and foreign keys to related dimensions.\ntable catalog_sales\n\tlineageTag: c38e1686-b3a0-473b-8604-8efe0410ea48\n\tsourceLineageTag: [dbo].[catalog_sales]\n\n\t/// Foreign key to date dimension for sale date.\n\tcolumn cs_sold_date_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: ac2ce865-caed-4039-bda1-a182c516cfdd\n\t\tsourceLineageTag: cs_sold_date_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_sold_date_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension for billing customer.\n\tcolumn cs_bill_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 310a5931-6249-4ad7-86b4-7aca5fbc39a9\n\t\tsourceLineageTag: cs_bill_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_bill_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension for billing address.\n\tcolumn cs_bill_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 51e6cbcf-e98d-4ea9-865e-a500419bec99\n\t\tsourceLineageTag: cs_bill_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_bill_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension for shipping customer.\n\tcolumn cs_ship_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4a433af2-d889-41d8-9c6f-1f05f94f0070\n\t\tsourceLineageTag: cs_ship_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension for shipping address.\n\tcolumn cs_ship_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 25b41390-6ff3-4f15-b0e6-f26a3da61ef5\n\t\tsourceLineageTag: cs_ship_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to catalog page dimension.\n\tcolumn cs_catalog_page_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: b01fc3a0-0a51-4660-a5d3-56d28ea386cd\n\t\tsourceLineageTag: cs_catalog_page_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_catalog_page_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to ship mode dimension.\n\tcolumn cs_ship_mode_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 0a85820a-e8f9-41f9-b50e-9610890408e8\n\t\tsourceLineageTag: cs_ship_mode_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_mode_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension.\n\tcolumn cs_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9024dff9-9629-4f0e-abae-0b29f00d568d\n\t\tsourceLineageTag: cs_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to promotion dimension.\n\tcolumn cs_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: dbfb7b81-9eef-4b86-aac0-a48cc36d0168\n\t\tsourceLineageTag: cs_promo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Catalog order number for this transaction.\n\tcolumn cs_order_number\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: ccec0342-6144-411f-bc2d-90234bcfb5ac\n\t\tsourceLineageTag: cs_order_number\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_order_number\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quantity of items ordered in this transaction.\n\tcolumn cs_quantity\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 0f5abf99-45b4-495b-bad3-795d0e8e5802\n\t\tsourceLineageTag: cs_quantity\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_quantity\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Wholesale cost per unit for this transaction.\n\tcolumn cs_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 37d760e7-f755-42e9-9dba-3a28cc211ae7\n\t\tsourceLineageTag: cs_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// List price per unit at time of order.\n\tcolumn cs_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 0eda1627-f565-444d-92e5-1a5bb0929e4d\n\t\tsourceLineageTag: cs_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Actual sales price per unit (after discounts).\n\tcolumn cs_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: eebce86e-da48-4799-8f93-1a8e69de6605\n\t\tsourceLineageTag: cs_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended sales price (sales price \u00d7 quantity).\n\tcolumn cs_ext_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: f0c74fde-bf40-47e4-9352-6bc4d4f219a0\n\t\tsourceLineageTag: cs_ext_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_ext_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended list price (list price \u00d7 quantity).\n\tcolumn cs_ext_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 845b8a06-567d-49c0-9455-1a469d81f646\n\t\tsourceLineageTag: cs_ext_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_ext_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Net profit for this transaction (sales price - wholesale cost).\n\tcolumn cs_net_profit\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 341a4a24-e726-4b20-8a50-967f689a5d01\n\t\tsourceLineageTag: cs_net_profit\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_net_profit\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Technical column used for cache invalidation in DirectLake mode.\n\tcolumn cache_buster\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 33947ffc-dc12-42ab-af62-5b00c42adf7a\n\t\tsourceLineageTag: cache_buster\n\t\tsummarizeBy: none\n\t\tsourceColumn: cache_buster\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition catalog_sales = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: catalog_sales\n\t\t\tschemaName: tpcds_sf__SF___partition\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/customer_address.tmdl","text":"/// Customer address dimension containing geographic information for billing and shipping addresses.\ntable customer_address\n\tlineageTag: 75246b34-d9f4-46fd-813f-bc5f0e47ab8c\n\tsourceLineageTag: [dbo].[customer_address]\n\n\t/// Customer address surrogate key - unique identifier for each address.\n\tcolumn ca_address_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9084b5d7-d188-4fb9-9c92-211d5b68f397\n\t\tsourceLineageTag: ca_address_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_address_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// City name for the customer address.\n\tcolumn ca_city\n\t\tdataType: string\n\t\tlineageTag: 03b2fb94-a08c-4010-9157-b91edcb60daf\n\t\tsourceLineageTag: ca_city\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_city\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// County name for the customer address.\n\tcolumn ca_county\n\t\tdataType: string\n\t\tlineageTag: 0ef7c0d6-1e17-42c5-a9bb-fb014c91398a\n\t\tsourceLineageTag: ca_county\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_county\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// State or province for the customer address.\n\tcolumn ca_state\n\t\tdataType: string\n\t\tlineageTag: 24c6ee20-e4ab-4ca3-aa30-934ba66a55db\n\t\tsourceLineageTag: ca_state\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_state\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Postal code for the customer address.\n\tcolumn ca_zip\n\t\tdataType: string\n\t\tlineageTag: a57bf4fe-e3de-4848-81bd-e71dd885c99f\n\t\tsourceLineageTag: ca_zip\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_zip\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type of location (e.g., residential, commercial).\n\tcolumn ca_location_type\n\t\tdataType: string\n\t\tlineageTag: 9a90fb17-4314-40df-84ae-071cdd4eb3c7\n\t\tsourceLineageTag: ca_location_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_location_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition customer_address = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: customer_address\n\t\t\tschemaName: tpcds_sf__SF___partition\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/customer_demographics.tmdl","text":"/// Customer demographics dimension containing education status and marital status attributes for customer segmentation.\ntable customer_demographics\n\tlineageTag: 52f8f268-90f3-4f40-a527-07a74d9e9baf\n\tsourceLineageTag: [dbo].[customer_demographics]\n\n\t/// Customer demographics surrogate key - unique identifier for each demographic profile.\n\tcolumn cd_demo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: d2cad4d0-4a88-41ab-b7ee-94ab73973694\n\t\tsourceLineageTag: cd_demo_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_demo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Marital status of the customer.\n\tcolumn cd_marital_status\n\t\tdataType: string\n\t\tlineageTag: e29a3681-2c0d-4579-9b35-5112d86c0f46\n\t\tsourceLineageTag: cd_marital_status\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_marital_status\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Education level of the customer.\n\tcolumn cd_education_status\n\t\tdataType: string\n\t\tlineageTag: 43260b21-0847-455f-a853-e2233af2a62e\n\t\tsourceLineageTag: cd_education_status\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_education_status\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition customer_demographics = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: customer_demographics\n\t\t\tschemaName: tpcds_sf__SF___partition\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/date_dim.tmdl","text":"/// Date dimension providing calendar hierarchy with year, quarter, month, and day-of-week attributes for time-based analysis.\ntable date_dim\n\tlineageTag: 9e7ec2de-382c-4a3b-9589-1ece445bfa27\n\tsourceLineageTag: [dbo].[date_dim]\n\n\t/// Date surrogate key - unique identifier for each calendar date.\n\tcolumn d_date_sk_1\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 63841738-6970-46ac-8106-970384b1052b\n\t\tsourceLineageTag: d_date_sk_1\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_date_sk_1\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Calendar date value.\n\tcolumn d_date\n\t\tdataType: dateTime\n\t\tformatString: General Date\n\t\tlineageTag: dc08a5e6-1a30-4d5f-9b03-1c98c7bac892\n\t\tsourceLineageTag: d_date\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_date\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Calendar year (e.g., 2023).\n\tcolumn d_year\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: fcf4b9e9-c880-4ee0-907c-3a18951b0809\n\t\tsourceLineageTag: d_year\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_year\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Day of week (1=Sunday, 7=Saturday).\n\tcolumn d_dow\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 339ac74d-ab31-4d4e-93eb-524b9f7e211c\n\t\tsourceLineageTag: d_dow\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_dow\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Month of year (1-12).\n\tcolumn d_moy\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: faa3deb0-9def-4334-bd42-a1b9d60b51b7\n\t\tsourceLineageTag: d_moy\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_moy\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Day of month (1-31).\n\tcolumn d_dom\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 83cd6ebd-7811-4616-a884-9930f66f8bfe\n\t\tsourceLineageTag: d_dom\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_dom\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quarter of year (1-4).\n\tcolumn d_qoy\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 4962ee0a-fcb9-4bf8-a5c7-43a66a94e67f\n\t\tsourceLineageTag: d_qoy\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_qoy\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quarter name (e.g., 'Q1 2023').\n\tcolumn d_quarter_name\n\t\tdataType: string\n\t\tlineageTag: 2426fa03-107c-4f51-a0b4-c59bb4b85d4f\n\t\tsourceLineageTag: d_quarter_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_quarter_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition date_dim = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: date_dim\n\t\t\tschemaName: tpcds_sf__SF___partition\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n\tcalendar tpcds_calendar\n\t\tlineageTag: dd529b7a-99fe-4cb5-928c-df1abed499f1\n\n\t\tcalendarColumnGroup = year\n\t\t\tprimaryColumn: d_year\n\n\t\tcalendarColumnGroup = quarter\n\t\t\tprimaryColumn: d_quarter_name\n\n\t\tcalendarColumnGroup = quarterOfYear\n\t\t\tprimaryColumn: d_qoy\n\n\t\tcalendarColumnGroup = date\n\t\t\tprimaryColumn: d_date\n\n"},{"path":"definition/tables/item.tmdl","text":"/// Product dimension containing item details such as brand, category, class, color, pricing, and manufacturing information.\ntable item\n\tlineageTag: 0f958ae1-af31-4b29-87a3-7e9ee589887e\n\tsourceLineageTag: [dbo].[item]\n\n\t/// Item surrogate key - unique identifier for each product.\n\tcolumn i_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 23893369-a589-4284-afa3-b1e291864600\n\t\tsourceLineageTag: i_item_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Current retail price of the product.\n\tcolumn i_current_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 70b27235-eefb-4732-8e0d-64c9576f0776\n\t\tsourceLineageTag: i_current_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_current_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Wholesale cost paid for the product.\n\tcolumn i_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 90cbdd03-df2f-4439-941d-de295ceb5135\n\t\tsourceLineageTag: i_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Brand name of the product.\n\tcolumn i_brand\n\t\tdataType: string\n\t\tlineageTag: 0c701039-61ca-4f5f-872b-aab6677ef45c\n\t\tsourceLineageTag: i_brand\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_brand\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Product class or subcategory within the main category.\n\tcolumn i_class\n\t\tdataType: string\n\t\tlineageTag: 5bdf127c-d558-46b2-98e6-0e294148d466\n\t\tsourceLineageTag: i_class\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_class\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Product category classification.\n\tcolumn i_category\n\t\tdataType: string\n\t\tlineageTag: 138de914-55e9-4a46-b5b9-6707b88a38e9\n\t\tsourceLineageTag: i_category\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_category\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Manufacturer or producer of the product.\n\tcolumn i_manufact\n\t\tdataType: string\n\t\tlineageTag: e6852c2b-2bc8-4fb1-86cb-d7ebe7c18637\n\t\tsourceLineageTag: i_manufact\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_manufact\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Size specification of the product.\n\tcolumn i_size\n\t\tdataType: string\n\t\tlineageTag: 0aeb0963-bc02-4389-a3fd-ab7232caaf9c\n\t\tsourceLineageTag: i_size\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_size\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Primary color of the product.\n\tcolumn i_color\n\t\tdataType: string\n\t\tlineageTag: d6709cb2-852f-485a-a0de-8cdd07941038\n\t\tsourceLineageTag: i_color\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_color\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition item = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: item\n\t\t\tschemaName: tpcds_sf__SF___partition\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/promotion.tmdl","text":"/// Promotion dimension containing promotional campaign details including costs and promotional names.\ntable promotion\n\tlineageTag: 5942c68a-d07a-4a23-8864-b2c733be46c8\n\tsourceLineageTag: [dbo].[promotion]\n\n\t/// Promotion surrogate key - unique identifier for each promotional campaign.\n\tcolumn p_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 0299ca0a-f58f-4ebe-b752-ffa6b63f9760\n\t\tsourceLineageTag: p_promo_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension (for item-specific promotions).\n\tcolumn p_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 92f8bc42-df79-4a35-8a37-10a02a937c40\n\t\tsourceLineageTag: p_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: p_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Cost associated with running this promotional campaign.\n\tcolumn p_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: c42871df-1b39-4b1e-83cb-142e6864a1e0\n\t\tsourceLineageTag: p_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Name or title of the promotional campaign.\n\tcolumn p_promo_name\n\t\tdataType: string\n\t\tlineageTag: 6ceed2f6-4e12-4edc-b161-8251a356ec17\n\t\tsourceLineageTag: p_promo_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_promo_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition promotion = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: promotion\n\t\t\tschemaName: tpcds_sf__SF___partition\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/ship_mode.tmdl","text":"/// Shipping method dimension containing carrier information and shipping type classifications.\ntable ship_mode\n\tlineageTag: db1b87b5-f584-4f30-b70e-0eb00c8b45b8\n\tsourceLineageTag: [dbo].[ship_mode]\n\n\t/// Ship mode surrogate key - unique identifier for each shipping method.\n\tcolumn sm_ship_mode_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4c485c32-fa22-4df6-b4a8-a8df04e2f730\n\t\tsourceLineageTag: sm_ship_mode_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_ship_mode_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type or category of shipping method.\n\tcolumn sm_type\n\t\tdataType: string\n\t\tlineageTag: c49fc6f8-d030-49a2-8105-6a45d1a81ac2\n\t\tsourceLineageTag: sm_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Internal code for the shipping method.\n\tcolumn sm_code\n\t\tdataType: string\n\t\tlineageTag: f54a092d-e847-4cee-84e6-5b813cbf5280\n\t\tsourceLineageTag: sm_code\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_code\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Shipping carrier or company name.\n\tcolumn sm_carrier\n\t\tdataType: string\n\t\tlineageTag: aca70983-6b51-4291-83f8-77fac2e381fe\n\t\tsourceLineageTag: sm_carrier\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_carrier\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition ship_mode = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: ship_mode\n\t\t\tschemaName: tpcds_sf__SF___partition\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/store.tmdl","text":"/// Store dimension containing retail location information including geographic details, management, and tax rates.\ntable store\n\tlineageTag: 6835d4a1-f40a-40aa-8a65-98aa4e06a406\n\tsourceLineageTag: [dbo].[store]\n\n\t/// Store surrogate key - unique identifier for each store location.\n\tcolumn s_store_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 14502476-8801-4068-b7c4-875ce7cac939\n\t\tsourceLineageTag: s_store_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_store_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Store name or identifier for the retail location.\n\tcolumn s_store_name\n\t\tdataType: string\n\t\tlineageTag: 4e78d335-4974-4d41-8402-ea89f63f9005\n\t\tsourceLineageTag: s_store_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_store_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Name of the store manager.\n\tcolumn s_manager\n\t\tdataType: string\n\t\tlineageTag: 3a8ddb5b-2ada-42d0-9ef9-89cd96b435b6\n\t\tsourceLineageTag: s_manager\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_manager\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Name of the market manager overseeing this store.\n\tcolumn s_market_manager\n\t\tdataType: string\n\t\tlineageTag: 37c8562d-3ab6-47ad-9b99-2ad1d72e6cd6\n\t\tsourceLineageTag: s_market_manager\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_market_manager\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// City where the store is located.\n\tcolumn s_city\n\t\tdataType: string\n\t\tlineageTag: a1a9209c-8194-4f96-8e94-69e545e064ba\n\t\tsourceLineageTag: s_city\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_city\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// County where the store is located.\n\tcolumn s_county\n\t\tdataType: string\n\t\tlineageTag: e219113c-fe81-45b0-b6be-f02e7ce75403\n\t\tsourceLineageTag: s_county\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_county\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// State or province where the store is located.\n\tcolumn s_state\n\t\tdataType: string\n\t\tlineageTag: fbaeba14-56ed-496d-a004-83f851771750\n\t\tsourceLineageTag: s_state\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_state\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Postal code for the store location.\n\tcolumn s_zip\n\t\tdataType: string\n\t\tlineageTag: 44e7ea79-56d6-4d20-8243-92c35919dcc3\n\t\tsourceLineageTag: s_zip\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_zip\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Tax rate percentage applied at this store location.\n\tcolumn s_tax_percentage\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: ba6d903d-f426-46b7-90a8-756a18261f2a\n\t\tsourceLineageTag: s_tax_percentage\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_tax_percentage\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\tpartition store = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: store\n\t\t\tschemaName: tpcds_sf__SF___partition\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/store_sales.tmdl","text":"/// Fact table containing retail store sales transactions with pricing, quantities, profits, and foreign keys to related dimensions.\ntable store_sales\n\tlineageTag: 6c7a2caf-3c22-4f38-a6aa-895a77e2e1c9\n\tsourceLineageTag: [dbo].[store_sales]\n\n\t/// Foreign key to date dimension for sale date.\n\tcolumn ss_sold_date_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4e30d94a-216f-4eaf-9858-fb22c6e6d57e\n\t\tsourceLineageTag: ss_sold_date_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_sold_date_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension.\n\tcolumn ss_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 2b151014-7df5-45f0-9907-03cbe191952b\n\t\tsourceLineageTag: ss_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Customer surrogate key for the transaction.\n\tcolumn ss_customer_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 558da56d-ea73-4562-b35c-ff94e774cca3\n\t\tsourceLineageTag: ss_customer_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_customer_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension.\n\tcolumn ss_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: e65f473f-bb26-4c7d-90f1-139de12a51ab\n\t\tsourceLineageTag: ss_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension.\n\tcolumn ss_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: cb65f3a1-6240-4baa-bfb2-4b24d39e2e87\n\t\tsourceLineageTag: ss_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to store dimension.\n\tcolumn ss_store_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 809a4a04-3bbf-4067-b54f-ee4d5fbbf6d4\n\t\tsourceLineageTag: ss_store_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_store_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to promotion dimension.\n\tcolumn ss_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9a64bbfb-0ae6-4919-b4e9-73e6d90353b3\n\t\tsourceLineageTag: ss_promo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Transaction ticket or receipt number.\n\tcolumn ss_ticket_number\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: d8ce37f4-a12c-4558-b131-66631546b607\n\t\tsourceLineageTag: ss_ticket_number\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ticket_number\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quantity of items sold in this transaction.\n\tcolumn ss_quantity\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 318e6a7b-88a2-413c-8257-315f66313d79\n\t\tsourceLineageTag: ss_quantity\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_quantity\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Wholesale cost per unit for this transaction.\n\tcolumn ss_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: d02100cc-2804-43d7-b752-8e98d579e202\n\t\tsourceLineageTag: ss_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// List price per unit at time of sale.\n\tcolumn ss_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 4f32f293-c19e-497e-906e-089bc204cde9\n\t\tsourceLineageTag: ss_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Actual sales price per unit (after discounts).\n\tcolumn ss_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: e620a719-f6e0-4965-8326-f4a2d255999b\n\t\tsourceLineageTag: ss_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended sales price (sales price \u00d7 quantity).\n\tcolumn ss_ext_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 8b66819a-a151-4fc7-9ec0-cacf7b6e8e01\n\t\tsourceLineageTag: ss_ext_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ext_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended list price (list price \u00d7 quantity).\n\tcolumn ss_ext_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: bc8e9c57-98e2-4032-a3fd-69808e3cdd8b\n\t\tsourceLineageTag: ss_ext_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ext_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Net profit for this transaction (sales price - wholesale cost).\n\tcolumn ss_net_profit\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: d2d6233f-6944-40b5-97d7-2870da393d1b\n\t\tsourceLineageTag: ss_net_profit\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_net_profit\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Technical column used for cache invalidation in DirectLake mode.\n\tcolumn cache_buster\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 7ae60894-bf63-4fef-a327-bf8796466374\n\t\tsourceLineageTag: cache_buster\n\t\tsummarizeBy: none\n\t\tsourceColumn: cache_buster\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition store_sales = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: store_sales\n\t\t\tschemaName: tpcds_sf__SF___partition\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition.pbism","text":"{\"$schema\": \"https://developer.microsoft.com/json-schemas/fabric/item/semanticModel/definitionProperties/1.0.0/schema.json\", \"version\": \"5.0\", \"settings\": {}}"}]},"vorder":{"model":"tpcds_sf__SF___vorder","parts":[{"path":"definition/database.tmdl","text":"database tpcds_sf__SF___vorder\n\tcompatibilityLevel: 1702\n\tcompatibilityMode: powerBI\n\tlanguage: 1033\n\n"},{"path":"definition/expressions.tmdl","text":"expression 'DirectLake - tpcds_sf__SF__' = ```\n\t\tlet\n\t\t    Source = AzureStorage.DataLake(\"https://onelake.dfs.fabric.microsoft.com/51650f82-6bb5-4023-b0ab-db197d32e0be/f19d8f93-2956-4cf7-a1bb-c526d0a953cd\", [HierarchicalNavigation=true])\n\t\tin\n\t\t    Source\n\t\t\n\t\t```\n\tlineageTag: c6e3390d-cc84-4e13-ab93-53308a346336\n\n\tannotation PBI_IncludeFutureArtifacts = False\n\n"},{"path":"definition/model.tmdl","text":"model Model\n\tdirectLakeBehavior: directLakeOnly\n\tculture: en-US\n\tdefaultPowerBIDataSourceVersion: powerBI_V3\n\tsourceQueryCulture: en-US\n\tdataAccessOptions\n\t\tlegacyRedirects\n\t\treturnErrorValuesAsNull\n\nannotation PBI_QueryOrder = [\"DirectLake - tpcds_sf__SF__\"]\n\nannotation __PBI_TimeIntelligenceEnabled = 1\n\nannotation __LastRPTime = 134244136754838152\n\nannotation PBI_ProTooling = [\"DirectLakeOnOneLakeInWeb\",\"WebModelingEdit\"]\n\nannotation __TEdtr = 1\n\nannotation TabularEditor_SerializeOptions = {\"IgnoreInferredObjects\":true,\"IgnoreInferredProperties\":true,\"IgnoreTimestamps\":true,\"SplitMultilineStrings\":true,\"PrefixFilenames\":false,\"LocalTranslations\":true,\"LocalPerspectives\":true,\"LocalRelationships\":true,\"Levels\":[\"Data Sources\",\"Perspectives\",\"Relationships\",\"Roles\",\"Shared Expressions\",\"Tables\",\"Tables/Calculation Items\",\"Tables/Columns\",\"Tables/Hierarchies\",\"Tables/Measures\",\"Tables/Partitions\",\"Translations\"]}\n\nref table store\nref table item\nref table date_dim\nref table store_sales\nref table catalog_page\nref table promotion\nref table ship_mode\nref table catalog_sales\nref table customer_address\nref table customer_demographics\nref table 'Measures 1'\nref table 'Time Unit'\n\n"},{"path":"definition/relationships.tmdl","text":"relationship b3bd8d91-ae6b-4489-83cd-34ca2c3213d8\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_bill_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship b727abac-57ca-484a-8183-1b4c10921d84\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_bill_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship 41e21761-05b3-410b-960e-54805fb2c4f2\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_catalog_page_sk\n\ttoColumn: catalog_page.cp_catalog_page_sk\n\nrelationship c7ce7218-d702-40d6-94b4-80009e453fe9\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship 4d02a5f5-e67a-4f65-a203-192e1c2ef166\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_promo_sk\n\ttoColumn: promotion.p_promo_sk\n\nrelationship 545e95f8-e63b-4a27-9c91-3cd725ca10e0\n\tisActive: false\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship 685424a1-f6c7-42d6-8f44-3cccf5f08db4\n\tisActive: false\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship 2e0c0f18-79bf-48f8-be1a-2c53cc33e257\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_mode_sk\n\ttoColumn: ship_mode.sm_ship_mode_sk\n\nrelationship c5edc5f8-f418-44a4-a8aa-caff6e0bef29\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_sold_date_sk\n\ttoColumn: date_dim.d_date_sk_1\n\nrelationship 9f8c8977-b8ac-4985-aed9-7711ce5004af\n\tisActive: false\n\tfromColumn: promotion.p_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship f824425a-7dd8-4a70-9ee5-2042a74ed97d\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship be9c6d5e-0bed-4db1-8558-326d5dfba951\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship f172b575-b3ce-48a4-b264-2928516104a2\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship 95b347fa-cf7e-4885-8124-86eb2ff56351\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_promo_sk\n\ttoColumn: promotion.p_promo_sk\n\nrelationship c782badb-be54-4423-98b0-22eebad6aa29\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_sold_date_sk\n\ttoColumn: date_dim.d_date_sk_1\n\nrelationship 9e5ca2d3-a5aa-4d43-b00a-07e4a857b1e4\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_store_sk\n\ttoColumn: store.s_store_sk\n\n"},{"path":"definition/tables/Measures 1.tmdl","text":"/// Calculated measures table containing key business metrics for revenue, quantity, profit, tax, and performance analysis across store and catalog channels.\ntable 'Measures 1'\n\tlineageTag: ae7cf165-f530-4f98-a4c5-befd958ff771\n\n\t/// Revenue from catalog sales channel only, based on extended sales price\n\tmeasure 'Catalog Revenue' = SUM('catalog_sales'[cs_ext_sales_price])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 01. Revenue\n\t\tlineageTag: bed34121-c406-4e25-b931-6ea438aede2a\n\n\t/// Revenue from store sales channel only, based on extended sales price\n\tmeasure 'Store Revenue' = SUM('store_sales'[ss_ext_sales_price])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 01. Revenue\n\t\tlineageTag: 1755e841-2eeb-4e1e-9a98-c860eb583a79\n\n\t/// Total units sold through catalog sales channel\n\tmeasure 'Catalog Sales Quantity' = SUM('catalog_sales'[cs_quantity])\n\t\tformatString: #,0\n\t\tdisplayFolder: 02. Quantity\n\t\tlineageTag: 6da96e65-1eb2-4e76-afdb-50d9106f56c1\n\n\t/// Net profit from store sales channel only\n\tmeasure 'Store Net Profit' = SUM('store_sales'[ss_net_profit])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 03. Profit\n\t\tlineageTag: a5a04305-c986-4791-925e-b9115939e453\n\n\t/// Number of unique customers who made store purchases\n\tmeasure 'Store Distinct Customers' = DISTINCTCOUNT('store_sales'[ss_customer_sk])\n\t\tformatString: #,0\n\t\tdisplayFolder: 04. Distinct Counts\n\t\tlineageTag: ace0485f-176a-4401-bb53-41c99ce5c356\n\n\t/// Total revenue from beginning of year to current date selection\n\tmeasure 'Store Revenue YTD' = TOTALYTD([Store Revenue],'tpcds_calendar')\n\t\tformatString: $#,0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 0e0dbb36-1e75-4c77-8b7f-9a462a143dcb\n\n\t/// Revenue for the same period in the previous year\n\tmeasure 'Store Revenue Same Period LY' =\n\t\t\t\n\t\t\tCALCULATE(\n\t\t\t    [Store Revenue],\n\t\t\t    SAMEPERIODLASTYEAR('tpcds_calendar')\n\t\t\t)\n\t\tformatString: $#,0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 303dbdc1-0484-496c-b14f-a41b0ef53f82\n\n\t/// Revenue for the same period in the previous year\n\tmeasure 'Store Revenue YoY' =\n\t\t\tVAR CurrentYearRev = [Store Revenue]\n\t\t\tVAR PreviousYearRev = [Store Revenue Same Period LY]\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentYearRev - PreviousYearRev, PreviousYearRev, 0)\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 39648465-7aac-4652-8c2f-d6599df15204\n\n\t/// Catalog Sales Same Period LY\n\tmeasure 'Catalog Sales Same Period LY' =\n\t\t\t\n\t\t\tCALCULATE(\n\t\t\t    [Catalog Sales Quantity],\n\t\t\t    SAMEPERIODLASTYEAR('tpcds_calendar')\n\t\t\t)\n\t\tformatString: 0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: d1d5179d-688b-487c-8638-81780d95eb05\n\n\t/// Catalog Sales YoY\n\tmeasure 'Catalog Sales YoY' =\n\t\t\tVAR CurrentYearCatSales = [Catalog Sales Quantity]\n\t\t\tVAR PreviousYearCatSales = [Catalog Sales Same Period LY]\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentYearCatSales - PreviousYearCatSales, PreviousYearCatSales, 0)\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 0bc09d6d-1bdd-4f7e-ac26-a12e87ccf67d\n\n\t/// Store Profit % by Item Category\n\tmeasure 'Store Profit % by Item Category' = ```\n\t\t\t\n\t\t\tVAR CurrentProfit = [Store Net Profit]\n\t\t\tVAR TotalProfitAllCategories = \n\t\t\t    CALCULATE(\n\t\t\t        [Store Net Profit],\n\t\t\t        ALLEXCEPT(\n\t\t\t            'item',\n\t\t\t            'item'[i_brand],\n\t\t\t            'item'[i_manufact]\n\t\t\t        )\n\t\t\t    )\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentProfit, TotalProfitAllCategories, 0)\n\t\t\t```\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 05. Advanced % Share\n\t\tlineageTag: 04f417ef-ba48-4d9e-873d-76d516ec3a54\n\n\t/// Dummy\n\tcolumn Dummy\n\t\tformatString: 0\n\t\tlineageTag: c1dfb904-84a8-4dfc-a768-d5e10f70f255\n\t\tsummarizeBy: none\n\t\tisNameInferred\n\t\tsourceColumn: [Dummy]\n\n\tpartition 'Measures 1' = calculated\n\t\tmode: import\n\t\tsource = ROW(\"Dummy\", 1)\n\n"},{"path":"definition/tables/Time Unit.tmdl","text":"/// Field parameter table for dynamic time unit selection in reports, supporting Year and Quarter groupings.\ntable 'Time Unit'\n\tlineageTag: c100712f-ebcb-4d0f-a435-e1a4c8ce3229\n\n\t/// Display name for the time unit selection (Year, Quarter).\n\tcolumn 'Time Unit'\n\t\tlineageTag: bd5d5ab0-6269-4df0-bd39-9616f6e2b7d5\n\t\tsummarizeBy: none\n\t\tsourceColumn: [Value1]\n\t\tsortByColumn: 'Time Unit Order'\n\n\t\trelatedColumnDetails\n\t\t\tgroupByColumn: 'Time Unit Fields'\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// DAX column reference for the selected time unit.\n\tcolumn 'Time Unit Fields'\n\t\tisHidden\n\t\tlineageTag: 4be3ced6-f4db-40c4-84b2-5ef9decf7fee\n\t\tsummarizeBy: none\n\t\tsourceColumn: [Value2]\n\t\tsortByColumn: 'Time Unit Order'\n\n\t\textendedProperty ParameterMetadata =\n\t\t\t\t{\n\t\t\t\t  \"version\": 3,\n\t\t\t\t  \"kind\": 2\n\t\t\t\t}\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Sort order for time unit options in field parameter.\n\tcolumn 'Time Unit Order'\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 36566b71-2a4f-4ddb-86c0-b4c7b3d8ab7f\n\t\tsummarizeBy: sum\n\t\tsourceColumn: [Value3]\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition 'Time Unit' = calculated\n\t\tmode: import\n\t\tsource =\n\t\t\t\t{\n\t\t\t\t    (\"Year\", NAMEOF('date_dim'[d_year]), 0),\n\t\t\t\t    (\"Quarter\", NAMEOF('date_dim'[d_quarter_name]), 1)\n\t\t\t\t}\n\n\tannotation PBI_Id = 706957f972f8497896950148d2d5f0af\n\n"},{"path":"definition/tables/catalog_page.tmdl","text":"/// Catalog page dimension containing information about catalog pages used in catalog sales campaigns.\ntable catalog_page\n\tlineageTag: b4e5474e-8b18-42b8-9a56-9c2416950a0a\n\tsourceLineageTag: [dbo].[catalog_page]\n\n\t/// Catalog page surrogate key - unique identifier for each catalog page.\n\tcolumn cp_catalog_page_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 71a9dba7-e582-461e-b8de-f37322137084\n\t\tsourceLineageTag: cp_catalog_page_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: cp_catalog_page_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type or category of the catalog page.\n\tcolumn cp_type\n\t\tdataType: string\n\t\tlineageTag: bdf42608-dc88-489d-90ce-24a87bc40378\n\t\tsourceLineageTag: cp_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: cp_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition catalog_page = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: catalog_page\n\t\t\tschemaName: tpcds_sf__SF__\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/catalog_sales.tmdl","text":"/// Fact table containing catalog sales transactions with pricing, quantities, profits, and foreign keys to related dimensions.\ntable catalog_sales\n\tlineageTag: c38e1686-b3a0-473b-8604-8efe0410ea48\n\tsourceLineageTag: [dbo].[catalog_sales]\n\n\t/// Foreign key to date dimension for sale date.\n\tcolumn cs_sold_date_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: ac2ce865-caed-4039-bda1-a182c516cfdd\n\t\tsourceLineageTag: cs_sold_date_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_sold_date_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension for billing customer.\n\tcolumn cs_bill_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 310a5931-6249-4ad7-86b4-7aca5fbc39a9\n\t\tsourceLineageTag: cs_bill_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_bill_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension for billing address.\n\tcolumn cs_bill_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 51e6cbcf-e98d-4ea9-865e-a500419bec99\n\t\tsourceLineageTag: cs_bill_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_bill_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension for shipping customer.\n\tcolumn cs_ship_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4a433af2-d889-41d8-9c6f-1f05f94f0070\n\t\tsourceLineageTag: cs_ship_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension for shipping address.\n\tcolumn cs_ship_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 25b41390-6ff3-4f15-b0e6-f26a3da61ef5\n\t\tsourceLineageTag: cs_ship_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to catalog page dimension.\n\tcolumn cs_catalog_page_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: b01fc3a0-0a51-4660-a5d3-56d28ea386cd\n\t\tsourceLineageTag: cs_catalog_page_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_catalog_page_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to ship mode dimension.\n\tcolumn cs_ship_mode_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 0a85820a-e8f9-41f9-b50e-9610890408e8\n\t\tsourceLineageTag: cs_ship_mode_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_mode_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension.\n\tcolumn cs_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9024dff9-9629-4f0e-abae-0b29f00d568d\n\t\tsourceLineageTag: cs_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to promotion dimension.\n\tcolumn cs_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: dbfb7b81-9eef-4b86-aac0-a48cc36d0168\n\t\tsourceLineageTag: cs_promo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Catalog order number for this transaction.\n\tcolumn cs_order_number\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: ccec0342-6144-411f-bc2d-90234bcfb5ac\n\t\tsourceLineageTag: cs_order_number\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_order_number\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quantity of items ordered in this transaction.\n\tcolumn cs_quantity\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 0f5abf99-45b4-495b-bad3-795d0e8e5802\n\t\tsourceLineageTag: cs_quantity\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_quantity\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Wholesale cost per unit for this transaction.\n\tcolumn cs_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 37d760e7-f755-42e9-9dba-3a28cc211ae7\n\t\tsourceLineageTag: cs_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// List price per unit at time of order.\n\tcolumn cs_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 0eda1627-f565-444d-92e5-1a5bb0929e4d\n\t\tsourceLineageTag: cs_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Actual sales price per unit (after discounts).\n\tcolumn cs_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: eebce86e-da48-4799-8f93-1a8e69de6605\n\t\tsourceLineageTag: cs_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended sales price (sales price \u00d7 quantity).\n\tcolumn cs_ext_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: f0c74fde-bf40-47e4-9352-6bc4d4f219a0\n\t\tsourceLineageTag: cs_ext_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_ext_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended list price (list price \u00d7 quantity).\n\tcolumn cs_ext_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 845b8a06-567d-49c0-9455-1a469d81f646\n\t\tsourceLineageTag: cs_ext_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_ext_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Net profit for this transaction (sales price - wholesale cost).\n\tcolumn cs_net_profit\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 341a4a24-e726-4b20-8a50-967f689a5d01\n\t\tsourceLineageTag: cs_net_profit\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_net_profit\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Technical column used for cache invalidation in DirectLake mode.\n\tcolumn cache_buster\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 33947ffc-dc12-42ab-af62-5b00c42adf7a\n\t\tsourceLineageTag: cache_buster\n\t\tsummarizeBy: none\n\t\tsourceColumn: cache_buster\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition catalog_sales = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: catalog_sales\n\t\t\tschemaName: tpcds_sf__SF__\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/customer_address.tmdl","text":"/// Customer address dimension containing geographic information for billing and shipping addresses.\ntable customer_address\n\tlineageTag: 75246b34-d9f4-46fd-813f-bc5f0e47ab8c\n\tsourceLineageTag: [dbo].[customer_address]\n\n\t/// Customer address surrogate key - unique identifier for each address.\n\tcolumn ca_address_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 9084b5d7-d188-4fb9-9c92-211d5b68f397\n\t\tsourceLineageTag: ca_address_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_address_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// City name for the customer address.\n\tcolumn ca_city\n\t\tdataType: string\n\t\tlineageTag: 03b2fb94-a08c-4010-9157-b91edcb60daf\n\t\tsourceLineageTag: ca_city\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_city\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// County name for the customer address.\n\tcolumn ca_county\n\t\tdataType: string\n\t\tlineageTag: 0ef7c0d6-1e17-42c5-a9bb-fb014c91398a\n\t\tsourceLineageTag: ca_county\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_county\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// State or province for the customer address.\n\tcolumn ca_state\n\t\tdataType: string\n\t\tlineageTag: 24c6ee20-e4ab-4ca3-aa30-934ba66a55db\n\t\tsourceLineageTag: ca_state\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_state\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Postal code for the customer address.\n\tcolumn ca_zip\n\t\tdataType: string\n\t\tlineageTag: a57bf4fe-e3de-4848-81bd-e71dd885c99f\n\t\tsourceLineageTag: ca_zip\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_zip\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type of location (e.g., residential, commercial).\n\tcolumn ca_location_type\n\t\tdataType: string\n\t\tlineageTag: 9a90fb17-4314-40df-84ae-071cdd4eb3c7\n\t\tsourceLineageTag: ca_location_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_location_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition customer_address = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: customer_address\n\t\t\tschemaName: tpcds_sf__SF__\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/customer_demographics.tmdl","text":"/// Customer demographics dimension containing education status and marital status attributes for customer segmentation.\ntable customer_demographics\n\tlineageTag: 52f8f268-90f3-4f40-a527-07a74d9e9baf\n\tsourceLineageTag: [dbo].[customer_demographics]\n\n\t/// Customer demographics surrogate key - unique identifier for each demographic profile.\n\tcolumn cd_demo_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: d2cad4d0-4a88-41ab-b7ee-94ab73973694\n\t\tsourceLineageTag: cd_demo_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_demo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Marital status of the customer.\n\tcolumn cd_marital_status\n\t\tdataType: string\n\t\tlineageTag: e29a3681-2c0d-4579-9b35-5112d86c0f46\n\t\tsourceLineageTag: cd_marital_status\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_marital_status\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Education level of the customer.\n\tcolumn cd_education_status\n\t\tdataType: string\n\t\tlineageTag: 43260b21-0847-455f-a853-e2233af2a62e\n\t\tsourceLineageTag: cd_education_status\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_education_status\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition customer_demographics = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: customer_demographics\n\t\t\tschemaName: tpcds_sf__SF__\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/date_dim.tmdl","text":"/// Date dimension providing calendar hierarchy with year, quarter, month, and day-of-week attributes for time-based analysis.\ntable date_dim\n\tlineageTag: 9e7ec2de-382c-4a3b-9589-1ece445bfa27\n\tsourceLineageTag: [dbo].[date_dim]\n\n\t/// Date surrogate key - unique identifier for each calendar date.\n\tcolumn d_date_sk_1\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 63841738-6970-46ac-8106-970384b1052b\n\t\tsourceLineageTag: d_date_sk_1\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_date_sk_1\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Calendar date value.\n\tcolumn d_date\n\t\tdataType: dateTime\n\t\tformatString: General Date\n\t\tlineageTag: dc08a5e6-1a30-4d5f-9b03-1c98c7bac892\n\t\tsourceLineageTag: d_date\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_date\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Calendar year (e.g., 2023).\n\tcolumn d_year\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: fcf4b9e9-c880-4ee0-907c-3a18951b0809\n\t\tsourceLineageTag: d_year\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_year\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Day of week (1=Sunday, 7=Saturday).\n\tcolumn d_dow\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 339ac74d-ab31-4d4e-93eb-524b9f7e211c\n\t\tsourceLineageTag: d_dow\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_dow\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Month of year (1-12).\n\tcolumn d_moy\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: faa3deb0-9def-4334-bd42-a1b9d60b51b7\n\t\tsourceLineageTag: d_moy\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_moy\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Day of month (1-31).\n\tcolumn d_dom\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 83cd6ebd-7811-4616-a884-9930f66f8bfe\n\t\tsourceLineageTag: d_dom\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_dom\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quarter of year (1-4).\n\tcolumn d_qoy\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 4962ee0a-fcb9-4bf8-a5c7-43a66a94e67f\n\t\tsourceLineageTag: d_qoy\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_qoy\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quarter name (e.g., 'Q1 2023').\n\tcolumn d_quarter_name\n\t\tdataType: string\n\t\tlineageTag: 2426fa03-107c-4f51-a0b4-c59bb4b85d4f\n\t\tsourceLineageTag: d_quarter_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_quarter_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition date_dim = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: date_dim\n\t\t\tschemaName: tpcds_sf__SF__\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n\tcalendar tpcds_calendar\n\t\tlineageTag: dd529b7a-99fe-4cb5-928c-df1abed499f1\n\n\t\tcalendarColumnGroup = year\n\t\t\tprimaryColumn: d_year\n\n\t\tcalendarColumnGroup = quarter\n\t\t\tprimaryColumn: d_quarter_name\n\n\t\tcalendarColumnGroup = quarterOfYear\n\t\t\tprimaryColumn: d_qoy\n\n\t\tcalendarColumnGroup = date\n\t\t\tprimaryColumn: d_date\n\n"},{"path":"definition/tables/item.tmdl","text":"/// Product dimension containing item details such as brand, category, class, color, pricing, and manufacturing information.\ntable item\n\tlineageTag: 0f958ae1-af31-4b29-87a3-7e9ee589887e\n\tsourceLineageTag: [dbo].[item]\n\n\t/// Item surrogate key - unique identifier for each product.\n\tcolumn i_item_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 23893369-a589-4284-afa3-b1e291864600\n\t\tsourceLineageTag: i_item_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Current retail price of the product.\n\tcolumn i_current_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 70b27235-eefb-4732-8e0d-64c9576f0776\n\t\tsourceLineageTag: i_current_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_current_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Wholesale cost paid for the product.\n\tcolumn i_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 90cbdd03-df2f-4439-941d-de295ceb5135\n\t\tsourceLineageTag: i_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Brand name of the product.\n\tcolumn i_brand\n\t\tdataType: string\n\t\tlineageTag: 0c701039-61ca-4f5f-872b-aab6677ef45c\n\t\tsourceLineageTag: i_brand\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_brand\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Product class or subcategory within the main category.\n\tcolumn i_class\n\t\tdataType: string\n\t\tlineageTag: 5bdf127c-d558-46b2-98e6-0e294148d466\n\t\tsourceLineageTag: i_class\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_class\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Product category classification.\n\tcolumn i_category\n\t\tdataType: string\n\t\tlineageTag: 138de914-55e9-4a46-b5b9-6707b88a38e9\n\t\tsourceLineageTag: i_category\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_category\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Manufacturer or producer of the product.\n\tcolumn i_manufact\n\t\tdataType: string\n\t\tlineageTag: e6852c2b-2bc8-4fb1-86cb-d7ebe7c18637\n\t\tsourceLineageTag: i_manufact\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_manufact\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Size specification of the product.\n\tcolumn i_size\n\t\tdataType: string\n\t\tlineageTag: 0aeb0963-bc02-4389-a3fd-ab7232caaf9c\n\t\tsourceLineageTag: i_size\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_size\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Primary color of the product.\n\tcolumn i_color\n\t\tdataType: string\n\t\tlineageTag: d6709cb2-852f-485a-a0de-8cdd07941038\n\t\tsourceLineageTag: i_color\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_color\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition item = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: item\n\t\t\tschemaName: tpcds_sf__SF__\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/promotion.tmdl","text":"/// Promotion dimension containing promotional campaign details including costs and promotional names.\ntable promotion\n\tlineageTag: 5942c68a-d07a-4a23-8864-b2c733be46c8\n\tsourceLineageTag: [dbo].[promotion]\n\n\t/// Promotion surrogate key - unique identifier for each promotional campaign.\n\tcolumn p_promo_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 0299ca0a-f58f-4ebe-b752-ffa6b63f9760\n\t\tsourceLineageTag: p_promo_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension (for item-specific promotions).\n\tcolumn p_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 92f8bc42-df79-4a35-8a37-10a02a937c40\n\t\tsourceLineageTag: p_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: p_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Cost associated with running this promotional campaign.\n\tcolumn p_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: c42871df-1b39-4b1e-83cb-142e6864a1e0\n\t\tsourceLineageTag: p_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Name or title of the promotional campaign.\n\tcolumn p_promo_name\n\t\tdataType: string\n\t\tlineageTag: 6ceed2f6-4e12-4edc-b161-8251a356ec17\n\t\tsourceLineageTag: p_promo_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_promo_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition promotion = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: promotion\n\t\t\tschemaName: tpcds_sf__SF__\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/ship_mode.tmdl","text":"/// Shipping method dimension containing carrier information and shipping type classifications.\ntable ship_mode\n\tlineageTag: db1b87b5-f584-4f30-b70e-0eb00c8b45b8\n\tsourceLineageTag: [dbo].[ship_mode]\n\n\t/// Ship mode surrogate key - unique identifier for each shipping method.\n\tcolumn sm_ship_mode_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 4c485c32-fa22-4df6-b4a8-a8df04e2f730\n\t\tsourceLineageTag: sm_ship_mode_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_ship_mode_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type or category of shipping method.\n\tcolumn sm_type\n\t\tdataType: string\n\t\tlineageTag: c49fc6f8-d030-49a2-8105-6a45d1a81ac2\n\t\tsourceLineageTag: sm_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Internal code for the shipping method.\n\tcolumn sm_code\n\t\tdataType: string\n\t\tlineageTag: f54a092d-e847-4cee-84e6-5b813cbf5280\n\t\tsourceLineageTag: sm_code\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_code\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Shipping carrier or company name.\n\tcolumn sm_carrier\n\t\tdataType: string\n\t\tlineageTag: aca70983-6b51-4291-83f8-77fac2e381fe\n\t\tsourceLineageTag: sm_carrier\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_carrier\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition ship_mode = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: ship_mode\n\t\t\tschemaName: tpcds_sf__SF__\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/store.tmdl","text":"/// Store dimension containing retail location information including geographic details, management, and tax rates.\ntable store\n\tlineageTag: 6835d4a1-f40a-40aa-8a65-98aa4e06a406\n\tsourceLineageTag: [dbo].[store]\n\n\t/// Store surrogate key - unique identifier for each store location.\n\tcolumn s_store_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 14502476-8801-4068-b7c4-875ce7cac939\n\t\tsourceLineageTag: s_store_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_store_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Store name or identifier for the retail location.\n\tcolumn s_store_name\n\t\tdataType: string\n\t\tlineageTag: 4e78d335-4974-4d41-8402-ea89f63f9005\n\t\tsourceLineageTag: s_store_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_store_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Name of the store manager.\n\tcolumn s_manager\n\t\tdataType: string\n\t\tlineageTag: 3a8ddb5b-2ada-42d0-9ef9-89cd96b435b6\n\t\tsourceLineageTag: s_manager\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_manager\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Name of the market manager overseeing this store.\n\tcolumn s_market_manager\n\t\tdataType: string\n\t\tlineageTag: 37c8562d-3ab6-47ad-9b99-2ad1d72e6cd6\n\t\tsourceLineageTag: s_market_manager\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_market_manager\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// City where the store is located.\n\tcolumn s_city\n\t\tdataType: string\n\t\tlineageTag: a1a9209c-8194-4f96-8e94-69e545e064ba\n\t\tsourceLineageTag: s_city\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_city\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// County where the store is located.\n\tcolumn s_county\n\t\tdataType: string\n\t\tlineageTag: e219113c-fe81-45b0-b6be-f02e7ce75403\n\t\tsourceLineageTag: s_county\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_county\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// State or province where the store is located.\n\tcolumn s_state\n\t\tdataType: string\n\t\tlineageTag: fbaeba14-56ed-496d-a004-83f851771750\n\t\tsourceLineageTag: s_state\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_state\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Postal code for the store location.\n\tcolumn s_zip\n\t\tdataType: string\n\t\tlineageTag: 44e7ea79-56d6-4d20-8243-92c35919dcc3\n\t\tsourceLineageTag: s_zip\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_zip\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Tax rate percentage applied at this store location.\n\tcolumn s_tax_percentage\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: ba6d903d-f426-46b7-90a8-756a18261f2a\n\t\tsourceLineageTag: s_tax_percentage\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_tax_percentage\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\tpartition store = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: store\n\t\t\tschemaName: tpcds_sf__SF__\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/store_sales.tmdl","text":"/// Fact table containing retail store sales transactions with pricing, quantities, profits, and foreign keys to related dimensions.\ntable store_sales\n\tlineageTag: 6c7a2caf-3c22-4f38-a6aa-895a77e2e1c9\n\tsourceLineageTag: [dbo].[store_sales]\n\n\t/// Foreign key to date dimension for sale date.\n\tcolumn ss_sold_date_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4e30d94a-216f-4eaf-9858-fb22c6e6d57e\n\t\tsourceLineageTag: ss_sold_date_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_sold_date_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension.\n\tcolumn ss_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 2b151014-7df5-45f0-9907-03cbe191952b\n\t\tsourceLineageTag: ss_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Customer surrogate key for the transaction.\n\tcolumn ss_customer_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 558da56d-ea73-4562-b35c-ff94e774cca3\n\t\tsourceLineageTag: ss_customer_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_customer_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension.\n\tcolumn ss_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: e65f473f-bb26-4c7d-90f1-139de12a51ab\n\t\tsourceLineageTag: ss_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension.\n\tcolumn ss_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: cb65f3a1-6240-4baa-bfb2-4b24d39e2e87\n\t\tsourceLineageTag: ss_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to store dimension.\n\tcolumn ss_store_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 809a4a04-3bbf-4067-b54f-ee4d5fbbf6d4\n\t\tsourceLineageTag: ss_store_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_store_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to promotion dimension.\n\tcolumn ss_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9a64bbfb-0ae6-4919-b4e9-73e6d90353b3\n\t\tsourceLineageTag: ss_promo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Transaction ticket or receipt number.\n\tcolumn ss_ticket_number\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: d8ce37f4-a12c-4558-b131-66631546b607\n\t\tsourceLineageTag: ss_ticket_number\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ticket_number\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quantity of items sold in this transaction.\n\tcolumn ss_quantity\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 318e6a7b-88a2-413c-8257-315f66313d79\n\t\tsourceLineageTag: ss_quantity\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_quantity\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Wholesale cost per unit for this transaction.\n\tcolumn ss_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: d02100cc-2804-43d7-b752-8e98d579e202\n\t\tsourceLineageTag: ss_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// List price per unit at time of sale.\n\tcolumn ss_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 4f32f293-c19e-497e-906e-089bc204cde9\n\t\tsourceLineageTag: ss_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Actual sales price per unit (after discounts).\n\tcolumn ss_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: e620a719-f6e0-4965-8326-f4a2d255999b\n\t\tsourceLineageTag: ss_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended sales price (sales price \u00d7 quantity).\n\tcolumn ss_ext_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 8b66819a-a151-4fc7-9ec0-cacf7b6e8e01\n\t\tsourceLineageTag: ss_ext_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ext_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended list price (list price \u00d7 quantity).\n\tcolumn ss_ext_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: bc8e9c57-98e2-4032-a3fd-69808e3cdd8b\n\t\tsourceLineageTag: ss_ext_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ext_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Net profit for this transaction (sales price - wholesale cost).\n\tcolumn ss_net_profit\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: d2d6233f-6944-40b5-97d7-2870da393d1b\n\t\tsourceLineageTag: ss_net_profit\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_net_profit\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Technical column used for cache invalidation in DirectLake mode.\n\tcolumn cache_buster\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 7ae60894-bf63-4fef-a327-bf8796466374\n\t\tsourceLineageTag: cache_buster\n\t\tsummarizeBy: none\n\t\tsourceColumn: cache_buster\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition store_sales = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: store_sales\n\t\t\tschemaName: tpcds_sf__SF__\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition.pbism","text":"{\"$schema\": \"https://developer.microsoft.com/json-schemas/fabric/item/semanticModel/definitionProperties/1.0.0/schema.json\", \"version\": \"5.0\", \"settings\": {}}"}]},"vonly":{"model":"tpcds_sf__SF___vonly","parts":[{"path":"definition/database.tmdl","text":"database tpcds_sf__SF___vonly\n\tcompatibilityLevel: 1702\n\tcompatibilityMode: powerBI\n\tlanguage: 1033\n\n"},{"path":"definition/expressions.tmdl","text":"expression 'DirectLake - tpcds_sf__SF__' = ```\n\t\tlet\n\t\t    Source = AzureStorage.DataLake(\"https://onelake.dfs.fabric.microsoft.com/51650f82-6bb5-4023-b0ab-db197d32e0be/f19d8f93-2956-4cf7-a1bb-c526d0a953cd\", [HierarchicalNavigation=true])\n\t\tin\n\t\t    Source\n\t\t\n\t\t```\n\tlineageTag: c6e3390d-cc84-4e13-ab93-53308a346336\n\n\tannotation PBI_IncludeFutureArtifacts = False\n\n"},{"path":"definition/model.tmdl","text":"model Model\n\tdirectLakeBehavior: directLakeOnly\n\tculture: en-US\n\tdefaultPowerBIDataSourceVersion: powerBI_V3\n\tsourceQueryCulture: en-US\n\tdataAccessOptions\n\t\tlegacyRedirects\n\t\treturnErrorValuesAsNull\n\nannotation PBI_QueryOrder = [\"DirectLake - tpcds_sf__SF__\"]\n\nannotation __PBI_TimeIntelligenceEnabled = 1\n\nannotation __LastRPTime = 134244136754838152\n\nannotation PBI_ProTooling = [\"DirectLakeOnOneLakeInWeb\",\"WebModelingEdit\"]\n\nannotation __TEdtr = 1\n\nannotation TabularEditor_SerializeOptions = {\"IgnoreInferredObjects\":true,\"IgnoreInferredProperties\":true,\"IgnoreTimestamps\":true,\"SplitMultilineStrings\":true,\"PrefixFilenames\":false,\"LocalTranslations\":true,\"LocalPerspectives\":true,\"LocalRelationships\":true,\"Levels\":[\"Data Sources\",\"Perspectives\",\"Relationships\",\"Roles\",\"Shared Expressions\",\"Tables\",\"Tables/Calculation Items\",\"Tables/Columns\",\"Tables/Hierarchies\",\"Tables/Measures\",\"Tables/Partitions\",\"Translations\"]}\n\nref table store\nref table item\nref table date_dim\nref table store_sales\nref table catalog_page\nref table promotion\nref table ship_mode\nref table catalog_sales\nref table customer_address\nref table customer_demographics\nref table 'Measures 1'\nref table 'Time Unit'\n\n"},{"path":"definition/relationships.tmdl","text":"relationship b3bd8d91-ae6b-4489-83cd-34ca2c3213d8\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_bill_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship b727abac-57ca-484a-8183-1b4c10921d84\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_bill_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship 41e21761-05b3-410b-960e-54805fb2c4f2\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_catalog_page_sk\n\ttoColumn: catalog_page.cp_catalog_page_sk\n\nrelationship c7ce7218-d702-40d6-94b4-80009e453fe9\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship 4d02a5f5-e67a-4f65-a203-192e1c2ef166\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_promo_sk\n\ttoColumn: promotion.p_promo_sk\n\nrelationship 545e95f8-e63b-4a27-9c91-3cd725ca10e0\n\tisActive: false\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship 685424a1-f6c7-42d6-8f44-3cccf5f08db4\n\tisActive: false\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship 2e0c0f18-79bf-48f8-be1a-2c53cc33e257\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_mode_sk\n\ttoColumn: ship_mode.sm_ship_mode_sk\n\nrelationship c5edc5f8-f418-44a4-a8aa-caff6e0bef29\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_sold_date_sk\n\ttoColumn: date_dim.d_date_sk_1\n\nrelationship 9f8c8977-b8ac-4985-aed9-7711ce5004af\n\tisActive: false\n\tfromColumn: promotion.p_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship f824425a-7dd8-4a70-9ee5-2042a74ed97d\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship be9c6d5e-0bed-4db1-8558-326d5dfba951\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship f172b575-b3ce-48a4-b264-2928516104a2\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship 95b347fa-cf7e-4885-8124-86eb2ff56351\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_promo_sk\n\ttoColumn: promotion.p_promo_sk\n\nrelationship c782badb-be54-4423-98b0-22eebad6aa29\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_sold_date_sk\n\ttoColumn: date_dim.d_date_sk_1\n\nrelationship 9e5ca2d3-a5aa-4d43-b00a-07e4a857b1e4\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_store_sk\n\ttoColumn: store.s_store_sk\n\n"},{"path":"definition/tables/Measures 1.tmdl","text":"/// Calculated measures table containing key business metrics for revenue, quantity, profit, tax, and performance analysis across store and catalog channels.\ntable 'Measures 1'\n\tlineageTag: ae7cf165-f530-4f98-a4c5-befd958ff771\n\n\t/// Revenue from catalog sales channel only, based on extended sales price\n\tmeasure 'Catalog Revenue' = SUM('catalog_sales'[cs_ext_sales_price])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 01. Revenue\n\t\tlineageTag: bed34121-c406-4e25-b931-6ea438aede2a\n\n\t/// Revenue from store sales channel only, based on extended sales price\n\tmeasure 'Store Revenue' = SUM('store_sales'[ss_ext_sales_price])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 01. Revenue\n\t\tlineageTag: 1755e841-2eeb-4e1e-9a98-c860eb583a79\n\n\t/// Total units sold through catalog sales channel\n\tmeasure 'Catalog Sales Quantity' = SUM('catalog_sales'[cs_quantity])\n\t\tformatString: #,0\n\t\tdisplayFolder: 02. Quantity\n\t\tlineageTag: 6da96e65-1eb2-4e76-afdb-50d9106f56c1\n\n\t/// Net profit from store sales channel only\n\tmeasure 'Store Net Profit' = SUM('store_sales'[ss_net_profit])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 03. Profit\n\t\tlineageTag: a5a04305-c986-4791-925e-b9115939e453\n\n\t/// Number of unique customers who made store purchases\n\tmeasure 'Store Distinct Customers' = DISTINCTCOUNT('store_sales'[ss_customer_sk])\n\t\tformatString: #,0\n\t\tdisplayFolder: 04. Distinct Counts\n\t\tlineageTag: ace0485f-176a-4401-bb53-41c99ce5c356\n\n\t/// Total revenue from beginning of year to current date selection\n\tmeasure 'Store Revenue YTD' = TOTALYTD([Store Revenue],'tpcds_calendar')\n\t\tformatString: $#,0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 0e0dbb36-1e75-4c77-8b7f-9a462a143dcb\n\n\t/// Revenue for the same period in the previous year\n\tmeasure 'Store Revenue Same Period LY' =\n\t\t\t\n\t\t\tCALCULATE(\n\t\t\t    [Store Revenue],\n\t\t\t    SAMEPERIODLASTYEAR('tpcds_calendar')\n\t\t\t)\n\t\tformatString: $#,0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 303dbdc1-0484-496c-b14f-a41b0ef53f82\n\n\t/// Revenue for the same period in the previous year\n\tmeasure 'Store Revenue YoY' =\n\t\t\tVAR CurrentYearRev = [Store Revenue]\n\t\t\tVAR PreviousYearRev = [Store Revenue Same Period LY]\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentYearRev - PreviousYearRev, PreviousYearRev, 0)\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 39648465-7aac-4652-8c2f-d6599df15204\n\n\t/// Catalog Sales Same Period LY\n\tmeasure 'Catalog Sales Same Period LY' =\n\t\t\t\n\t\t\tCALCULATE(\n\t\t\t    [Catalog Sales Quantity],\n\t\t\t    SAMEPERIODLASTYEAR('tpcds_calendar')\n\t\t\t)\n\t\tformatString: 0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: d1d5179d-688b-487c-8638-81780d95eb05\n\n\t/// Catalog Sales YoY\n\tmeasure 'Catalog Sales YoY' =\n\t\t\tVAR CurrentYearCatSales = [Catalog Sales Quantity]\n\t\t\tVAR PreviousYearCatSales = [Catalog Sales Same Period LY]\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentYearCatSales - PreviousYearCatSales, PreviousYearCatSales, 0)\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 0bc09d6d-1bdd-4f7e-ac26-a12e87ccf67d\n\n\t/// Store Profit % by Item Category\n\tmeasure 'Store Profit % by Item Category' = ```\n\t\t\t\n\t\t\tVAR CurrentProfit = [Store Net Profit]\n\t\t\tVAR TotalProfitAllCategories = \n\t\t\t    CALCULATE(\n\t\t\t        [Store Net Profit],\n\t\t\t        ALLEXCEPT(\n\t\t\t            'item',\n\t\t\t            'item'[i_brand],\n\t\t\t            'item'[i_manufact]\n\t\t\t        )\n\t\t\t    )\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentProfit, TotalProfitAllCategories, 0)\n\t\t\t```\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 05. Advanced % Share\n\t\tlineageTag: 04f417ef-ba48-4d9e-873d-76d516ec3a54\n\n\t/// Dummy\n\tcolumn Dummy\n\t\tformatString: 0\n\t\tlineageTag: c1dfb904-84a8-4dfc-a768-d5e10f70f255\n\t\tsummarizeBy: none\n\t\tisNameInferred\n\t\tsourceColumn: [Dummy]\n\n\tpartition 'Measures 1' = calculated\n\t\tmode: import\n\t\tsource = ROW(\"Dummy\", 1)\n\n"},{"path":"definition/tables/Time Unit.tmdl","text":"/// Field parameter table for dynamic time unit selection in reports, supporting Year and Quarter groupings.\ntable 'Time Unit'\n\tlineageTag: c100712f-ebcb-4d0f-a435-e1a4c8ce3229\n\n\t/// Display name for the time unit selection (Year, Quarter).\n\tcolumn 'Time Unit'\n\t\tlineageTag: bd5d5ab0-6269-4df0-bd39-9616f6e2b7d5\n\t\tsummarizeBy: none\n\t\tsourceColumn: [Value1]\n\t\tsortByColumn: 'Time Unit Order'\n\n\t\trelatedColumnDetails\n\t\t\tgroupByColumn: 'Time Unit Fields'\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// DAX column reference for the selected time unit.\n\tcolumn 'Time Unit Fields'\n\t\tisHidden\n\t\tlineageTag: 4be3ced6-f4db-40c4-84b2-5ef9decf7fee\n\t\tsummarizeBy: none\n\t\tsourceColumn: [Value2]\n\t\tsortByColumn: 'Time Unit Order'\n\n\t\textendedProperty ParameterMetadata =\n\t\t\t\t{\n\t\t\t\t  \"version\": 3,\n\t\t\t\t  \"kind\": 2\n\t\t\t\t}\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Sort order for time unit options in field parameter.\n\tcolumn 'Time Unit Order'\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 36566b71-2a4f-4ddb-86c0-b4c7b3d8ab7f\n\t\tsummarizeBy: sum\n\t\tsourceColumn: [Value3]\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition 'Time Unit' = calculated\n\t\tmode: import\n\t\tsource =\n\t\t\t\t{\n\t\t\t\t    (\"Year\", NAMEOF('date_dim'[d_year]), 0),\n\t\t\t\t    (\"Quarter\", NAMEOF('date_dim'[d_quarter_name]), 1)\n\t\t\t\t}\n\n\tannotation PBI_Id = 706957f972f8497896950148d2d5f0af\n\n"},{"path":"definition/tables/catalog_page.tmdl","text":"/// Catalog page dimension containing information about catalog pages used in catalog sales campaigns.\ntable catalog_page\n\tlineageTag: b4e5474e-8b18-42b8-9a56-9c2416950a0a\n\tsourceLineageTag: [dbo].[catalog_page]\n\n\t/// Catalog page surrogate key - unique identifier for each catalog page.\n\tcolumn cp_catalog_page_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 71a9dba7-e582-461e-b8de-f37322137084\n\t\tsourceLineageTag: cp_catalog_page_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: cp_catalog_page_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type or category of the catalog page.\n\tcolumn cp_type\n\t\tdataType: string\n\t\tlineageTag: bdf42608-dc88-489d-90ce-24a87bc40378\n\t\tsourceLineageTag: cp_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: cp_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition catalog_page = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: catalog_page\n\t\t\tschemaName: tpcds_sf__SF___vonly\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/catalog_sales.tmdl","text":"/// Fact table containing catalog sales transactions with pricing, quantities, profits, and foreign keys to related dimensions.\ntable catalog_sales\n\tlineageTag: c38e1686-b3a0-473b-8604-8efe0410ea48\n\tsourceLineageTag: [dbo].[catalog_sales]\n\n\t/// Foreign key to date dimension for sale date.\n\tcolumn cs_sold_date_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: ac2ce865-caed-4039-bda1-a182c516cfdd\n\t\tsourceLineageTag: cs_sold_date_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_sold_date_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension for billing customer.\n\tcolumn cs_bill_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 310a5931-6249-4ad7-86b4-7aca5fbc39a9\n\t\tsourceLineageTag: cs_bill_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_bill_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension for billing address.\n\tcolumn cs_bill_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 51e6cbcf-e98d-4ea9-865e-a500419bec99\n\t\tsourceLineageTag: cs_bill_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_bill_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension for shipping customer.\n\tcolumn cs_ship_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4a433af2-d889-41d8-9c6f-1f05f94f0070\n\t\tsourceLineageTag: cs_ship_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension for shipping address.\n\tcolumn cs_ship_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 25b41390-6ff3-4f15-b0e6-f26a3da61ef5\n\t\tsourceLineageTag: cs_ship_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to catalog page dimension.\n\tcolumn cs_catalog_page_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: b01fc3a0-0a51-4660-a5d3-56d28ea386cd\n\t\tsourceLineageTag: cs_catalog_page_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_catalog_page_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to ship mode dimension.\n\tcolumn cs_ship_mode_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 0a85820a-e8f9-41f9-b50e-9610890408e8\n\t\tsourceLineageTag: cs_ship_mode_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_mode_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension.\n\tcolumn cs_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9024dff9-9629-4f0e-abae-0b29f00d568d\n\t\tsourceLineageTag: cs_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to promotion dimension.\n\tcolumn cs_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: dbfb7b81-9eef-4b86-aac0-a48cc36d0168\n\t\tsourceLineageTag: cs_promo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Catalog order number for this transaction.\n\tcolumn cs_order_number\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: ccec0342-6144-411f-bc2d-90234bcfb5ac\n\t\tsourceLineageTag: cs_order_number\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_order_number\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quantity of items ordered in this transaction.\n\tcolumn cs_quantity\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 0f5abf99-45b4-495b-bad3-795d0e8e5802\n\t\tsourceLineageTag: cs_quantity\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_quantity\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Wholesale cost per unit for this transaction.\n\tcolumn cs_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 37d760e7-f755-42e9-9dba-3a28cc211ae7\n\t\tsourceLineageTag: cs_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// List price per unit at time of order.\n\tcolumn cs_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 0eda1627-f565-444d-92e5-1a5bb0929e4d\n\t\tsourceLineageTag: cs_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Actual sales price per unit (after discounts).\n\tcolumn cs_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: eebce86e-da48-4799-8f93-1a8e69de6605\n\t\tsourceLineageTag: cs_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended sales price (sales price \u00d7 quantity).\n\tcolumn cs_ext_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: f0c74fde-bf40-47e4-9352-6bc4d4f219a0\n\t\tsourceLineageTag: cs_ext_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_ext_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended list price (list price \u00d7 quantity).\n\tcolumn cs_ext_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 845b8a06-567d-49c0-9455-1a469d81f646\n\t\tsourceLineageTag: cs_ext_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_ext_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Net profit for this transaction (sales price - wholesale cost).\n\tcolumn cs_net_profit\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 341a4a24-e726-4b20-8a50-967f689a5d01\n\t\tsourceLineageTag: cs_net_profit\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_net_profit\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Technical column used for cache invalidation in DirectLake mode.\n\tcolumn cache_buster\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 33947ffc-dc12-42ab-af62-5b00c42adf7a\n\t\tsourceLineageTag: cache_buster\n\t\tsummarizeBy: none\n\t\tsourceColumn: cache_buster\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition catalog_sales = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: catalog_sales\n\t\t\tschemaName: tpcds_sf__SF___vonly\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/customer_address.tmdl","text":"/// Customer address dimension containing geographic information for billing and shipping addresses.\ntable customer_address\n\tlineageTag: 75246b34-d9f4-46fd-813f-bc5f0e47ab8c\n\tsourceLineageTag: [dbo].[customer_address]\n\n\t/// Customer address surrogate key - unique identifier for each address.\n\tcolumn ca_address_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 9084b5d7-d188-4fb9-9c92-211d5b68f397\n\t\tsourceLineageTag: ca_address_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_address_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// City name for the customer address.\n\tcolumn ca_city\n\t\tdataType: string\n\t\tlineageTag: 03b2fb94-a08c-4010-9157-b91edcb60daf\n\t\tsourceLineageTag: ca_city\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_city\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// County name for the customer address.\n\tcolumn ca_county\n\t\tdataType: string\n\t\tlineageTag: 0ef7c0d6-1e17-42c5-a9bb-fb014c91398a\n\t\tsourceLineageTag: ca_county\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_county\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// State or province for the customer address.\n\tcolumn ca_state\n\t\tdataType: string\n\t\tlineageTag: 24c6ee20-e4ab-4ca3-aa30-934ba66a55db\n\t\tsourceLineageTag: ca_state\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_state\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Postal code for the customer address.\n\tcolumn ca_zip\n\t\tdataType: string\n\t\tlineageTag: a57bf4fe-e3de-4848-81bd-e71dd885c99f\n\t\tsourceLineageTag: ca_zip\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_zip\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type of location (e.g., residential, commercial).\n\tcolumn ca_location_type\n\t\tdataType: string\n\t\tlineageTag: 9a90fb17-4314-40df-84ae-071cdd4eb3c7\n\t\tsourceLineageTag: ca_location_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_location_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition customer_address = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: customer_address\n\t\t\tschemaName: tpcds_sf__SF___vonly\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/customer_demographics.tmdl","text":"/// Customer demographics dimension containing education status and marital status attributes for customer segmentation.\ntable customer_demographics\n\tlineageTag: 52f8f268-90f3-4f40-a527-07a74d9e9baf\n\tsourceLineageTag: [dbo].[customer_demographics]\n\n\t/// Customer demographics surrogate key - unique identifier for each demographic profile.\n\tcolumn cd_demo_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: d2cad4d0-4a88-41ab-b7ee-94ab73973694\n\t\tsourceLineageTag: cd_demo_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_demo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Marital status of the customer.\n\tcolumn cd_marital_status\n\t\tdataType: string\n\t\tlineageTag: e29a3681-2c0d-4579-9b35-5112d86c0f46\n\t\tsourceLineageTag: cd_marital_status\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_marital_status\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Education level of the customer.\n\tcolumn cd_education_status\n\t\tdataType: string\n\t\tlineageTag: 43260b21-0847-455f-a853-e2233af2a62e\n\t\tsourceLineageTag: cd_education_status\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_education_status\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition customer_demographics = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: customer_demographics\n\t\t\tschemaName: tpcds_sf__SF___vonly\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/date_dim.tmdl","text":"/// Date dimension providing calendar hierarchy with year, quarter, month, and day-of-week attributes for time-based analysis.\ntable date_dim\n\tlineageTag: 9e7ec2de-382c-4a3b-9589-1ece445bfa27\n\tsourceLineageTag: [dbo].[date_dim]\n\n\t/// Date surrogate key - unique identifier for each calendar date.\n\tcolumn d_date_sk_1\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 63841738-6970-46ac-8106-970384b1052b\n\t\tsourceLineageTag: d_date_sk_1\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_date_sk_1\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Calendar date value.\n\tcolumn d_date\n\t\tdataType: dateTime\n\t\tformatString: General Date\n\t\tlineageTag: dc08a5e6-1a30-4d5f-9b03-1c98c7bac892\n\t\tsourceLineageTag: d_date\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_date\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Calendar year (e.g., 2023).\n\tcolumn d_year\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: fcf4b9e9-c880-4ee0-907c-3a18951b0809\n\t\tsourceLineageTag: d_year\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_year\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Day of week (1=Sunday, 7=Saturday).\n\tcolumn d_dow\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 339ac74d-ab31-4d4e-93eb-524b9f7e211c\n\t\tsourceLineageTag: d_dow\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_dow\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Month of year (1-12).\n\tcolumn d_moy\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: faa3deb0-9def-4334-bd42-a1b9d60b51b7\n\t\tsourceLineageTag: d_moy\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_moy\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Day of month (1-31).\n\tcolumn d_dom\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 83cd6ebd-7811-4616-a884-9930f66f8bfe\n\t\tsourceLineageTag: d_dom\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_dom\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quarter of year (1-4).\n\tcolumn d_qoy\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 4962ee0a-fcb9-4bf8-a5c7-43a66a94e67f\n\t\tsourceLineageTag: d_qoy\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_qoy\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quarter name (e.g., 'Q1 2023').\n\tcolumn d_quarter_name\n\t\tdataType: string\n\t\tlineageTag: 2426fa03-107c-4f51-a0b4-c59bb4b85d4f\n\t\tsourceLineageTag: d_quarter_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_quarter_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition date_dim = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: date_dim\n\t\t\tschemaName: tpcds_sf__SF___vonly\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n\tcalendar tpcds_calendar\n\t\tlineageTag: dd529b7a-99fe-4cb5-928c-df1abed499f1\n\n\t\tcalendarColumnGroup = year\n\t\t\tprimaryColumn: d_year\n\n\t\tcalendarColumnGroup = quarter\n\t\t\tprimaryColumn: d_quarter_name\n\n\t\tcalendarColumnGroup = quarterOfYear\n\t\t\tprimaryColumn: d_qoy\n\n\t\tcalendarColumnGroup = date\n\t\t\tprimaryColumn: d_date\n\n"},{"path":"definition/tables/item.tmdl","text":"/// Product dimension containing item details such as brand, category, class, color, pricing, and manufacturing information.\ntable item\n\tlineageTag: 0f958ae1-af31-4b29-87a3-7e9ee589887e\n\tsourceLineageTag: [dbo].[item]\n\n\t/// Item surrogate key - unique identifier for each product.\n\tcolumn i_item_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 23893369-a589-4284-afa3-b1e291864600\n\t\tsourceLineageTag: i_item_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Current retail price of the product.\n\tcolumn i_current_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 70b27235-eefb-4732-8e0d-64c9576f0776\n\t\tsourceLineageTag: i_current_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_current_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Wholesale cost paid for the product.\n\tcolumn i_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 90cbdd03-df2f-4439-941d-de295ceb5135\n\t\tsourceLineageTag: i_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Brand name of the product.\n\tcolumn i_brand\n\t\tdataType: string\n\t\tlineageTag: 0c701039-61ca-4f5f-872b-aab6677ef45c\n\t\tsourceLineageTag: i_brand\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_brand\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Product class or subcategory within the main category.\n\tcolumn i_class\n\t\tdataType: string\n\t\tlineageTag: 5bdf127c-d558-46b2-98e6-0e294148d466\n\t\tsourceLineageTag: i_class\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_class\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Product category classification.\n\tcolumn i_category\n\t\tdataType: string\n\t\tlineageTag: 138de914-55e9-4a46-b5b9-6707b88a38e9\n\t\tsourceLineageTag: i_category\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_category\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Manufacturer or producer of the product.\n\tcolumn i_manufact\n\t\tdataType: string\n\t\tlineageTag: e6852c2b-2bc8-4fb1-86cb-d7ebe7c18637\n\t\tsourceLineageTag: i_manufact\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_manufact\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Size specification of the product.\n\tcolumn i_size\n\t\tdataType: string\n\t\tlineageTag: 0aeb0963-bc02-4389-a3fd-ab7232caaf9c\n\t\tsourceLineageTag: i_size\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_size\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Primary color of the product.\n\tcolumn i_color\n\t\tdataType: string\n\t\tlineageTag: d6709cb2-852f-485a-a0de-8cdd07941038\n\t\tsourceLineageTag: i_color\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_color\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition item = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: item\n\t\t\tschemaName: tpcds_sf__SF___vonly\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/promotion.tmdl","text":"/// Promotion dimension containing promotional campaign details including costs and promotional names.\ntable promotion\n\tlineageTag: 5942c68a-d07a-4a23-8864-b2c733be46c8\n\tsourceLineageTag: [dbo].[promotion]\n\n\t/// Promotion surrogate key - unique identifier for each promotional campaign.\n\tcolumn p_promo_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 0299ca0a-f58f-4ebe-b752-ffa6b63f9760\n\t\tsourceLineageTag: p_promo_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension (for item-specific promotions).\n\tcolumn p_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 92f8bc42-df79-4a35-8a37-10a02a937c40\n\t\tsourceLineageTag: p_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: p_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Cost associated with running this promotional campaign.\n\tcolumn p_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: c42871df-1b39-4b1e-83cb-142e6864a1e0\n\t\tsourceLineageTag: p_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Name or title of the promotional campaign.\n\tcolumn p_promo_name\n\t\tdataType: string\n\t\tlineageTag: 6ceed2f6-4e12-4edc-b161-8251a356ec17\n\t\tsourceLineageTag: p_promo_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_promo_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition promotion = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: promotion\n\t\t\tschemaName: tpcds_sf__SF___vonly\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/ship_mode.tmdl","text":"/// Shipping method dimension containing carrier information and shipping type classifications.\ntable ship_mode\n\tlineageTag: db1b87b5-f584-4f30-b70e-0eb00c8b45b8\n\tsourceLineageTag: [dbo].[ship_mode]\n\n\t/// Ship mode surrogate key - unique identifier for each shipping method.\n\tcolumn sm_ship_mode_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 4c485c32-fa22-4df6-b4a8-a8df04e2f730\n\t\tsourceLineageTag: sm_ship_mode_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_ship_mode_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type or category of shipping method.\n\tcolumn sm_type\n\t\tdataType: string\n\t\tlineageTag: c49fc6f8-d030-49a2-8105-6a45d1a81ac2\n\t\tsourceLineageTag: sm_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Internal code for the shipping method.\n\tcolumn sm_code\n\t\tdataType: string\n\t\tlineageTag: f54a092d-e847-4cee-84e6-5b813cbf5280\n\t\tsourceLineageTag: sm_code\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_code\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Shipping carrier or company name.\n\tcolumn sm_carrier\n\t\tdataType: string\n\t\tlineageTag: aca70983-6b51-4291-83f8-77fac2e381fe\n\t\tsourceLineageTag: sm_carrier\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_carrier\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition ship_mode = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: ship_mode\n\t\t\tschemaName: tpcds_sf__SF___vonly\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/store.tmdl","text":"/// Store dimension containing retail location information including geographic details, management, and tax rates.\ntable store\n\tlineageTag: 6835d4a1-f40a-40aa-8a65-98aa4e06a406\n\tsourceLineageTag: [dbo].[store]\n\n\t/// Store surrogate key - unique identifier for each store location.\n\tcolumn s_store_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 14502476-8801-4068-b7c4-875ce7cac939\n\t\tsourceLineageTag: s_store_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_store_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Store name or identifier for the retail location.\n\tcolumn s_store_name\n\t\tdataType: string\n\t\tlineageTag: 4e78d335-4974-4d41-8402-ea89f63f9005\n\t\tsourceLineageTag: s_store_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_store_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Name of the store manager.\n\tcolumn s_manager\n\t\tdataType: string\n\t\tlineageTag: 3a8ddb5b-2ada-42d0-9ef9-89cd96b435b6\n\t\tsourceLineageTag: s_manager\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_manager\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Name of the market manager overseeing this store.\n\tcolumn s_market_manager\n\t\tdataType: string\n\t\tlineageTag: 37c8562d-3ab6-47ad-9b99-2ad1d72e6cd6\n\t\tsourceLineageTag: s_market_manager\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_market_manager\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// City where the store is located.\n\tcolumn s_city\n\t\tdataType: string\n\t\tlineageTag: a1a9209c-8194-4f96-8e94-69e545e064ba\n\t\tsourceLineageTag: s_city\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_city\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// County where the store is located.\n\tcolumn s_county\n\t\tdataType: string\n\t\tlineageTag: e219113c-fe81-45b0-b6be-f02e7ce75403\n\t\tsourceLineageTag: s_county\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_county\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// State or province where the store is located.\n\tcolumn s_state\n\t\tdataType: string\n\t\tlineageTag: fbaeba14-56ed-496d-a004-83f851771750\n\t\tsourceLineageTag: s_state\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_state\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Postal code for the store location.\n\tcolumn s_zip\n\t\tdataType: string\n\t\tlineageTag: 44e7ea79-56d6-4d20-8243-92c35919dcc3\n\t\tsourceLineageTag: s_zip\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_zip\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Tax rate percentage applied at this store location.\n\tcolumn s_tax_percentage\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: ba6d903d-f426-46b7-90a8-756a18261f2a\n\t\tsourceLineageTag: s_tax_percentage\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_tax_percentage\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\tpartition store = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: store\n\t\t\tschemaName: tpcds_sf__SF___vonly\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/store_sales.tmdl","text":"/// Fact table containing retail store sales transactions with pricing, quantities, profits, and foreign keys to related dimensions.\ntable store_sales\n\tlineageTag: 6c7a2caf-3c22-4f38-a6aa-895a77e2e1c9\n\tsourceLineageTag: [dbo].[store_sales]\n\n\t/// Foreign key to date dimension for sale date.\n\tcolumn ss_sold_date_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4e30d94a-216f-4eaf-9858-fb22c6e6d57e\n\t\tsourceLineageTag: ss_sold_date_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_sold_date_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension.\n\tcolumn ss_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 2b151014-7df5-45f0-9907-03cbe191952b\n\t\tsourceLineageTag: ss_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Customer surrogate key for the transaction.\n\tcolumn ss_customer_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 558da56d-ea73-4562-b35c-ff94e774cca3\n\t\tsourceLineageTag: ss_customer_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_customer_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension.\n\tcolumn ss_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: e65f473f-bb26-4c7d-90f1-139de12a51ab\n\t\tsourceLineageTag: ss_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension.\n\tcolumn ss_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: cb65f3a1-6240-4baa-bfb2-4b24d39e2e87\n\t\tsourceLineageTag: ss_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to store dimension.\n\tcolumn ss_store_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 809a4a04-3bbf-4067-b54f-ee4d5fbbf6d4\n\t\tsourceLineageTag: ss_store_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_store_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to promotion dimension.\n\tcolumn ss_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9a64bbfb-0ae6-4919-b4e9-73e6d90353b3\n\t\tsourceLineageTag: ss_promo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Transaction ticket or receipt number.\n\tcolumn ss_ticket_number\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: d8ce37f4-a12c-4558-b131-66631546b607\n\t\tsourceLineageTag: ss_ticket_number\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ticket_number\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quantity of items sold in this transaction.\n\tcolumn ss_quantity\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 318e6a7b-88a2-413c-8257-315f66313d79\n\t\tsourceLineageTag: ss_quantity\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_quantity\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Wholesale cost per unit for this transaction.\n\tcolumn ss_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: d02100cc-2804-43d7-b752-8e98d579e202\n\t\tsourceLineageTag: ss_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// List price per unit at time of sale.\n\tcolumn ss_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 4f32f293-c19e-497e-906e-089bc204cde9\n\t\tsourceLineageTag: ss_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Actual sales price per unit (after discounts).\n\tcolumn ss_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: e620a719-f6e0-4965-8326-f4a2d255999b\n\t\tsourceLineageTag: ss_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended sales price (sales price \u00d7 quantity).\n\tcolumn ss_ext_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 8b66819a-a151-4fc7-9ec0-cacf7b6e8e01\n\t\tsourceLineageTag: ss_ext_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ext_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended list price (list price \u00d7 quantity).\n\tcolumn ss_ext_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: bc8e9c57-98e2-4032-a3fd-69808e3cdd8b\n\t\tsourceLineageTag: ss_ext_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ext_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Net profit for this transaction (sales price - wholesale cost).\n\tcolumn ss_net_profit\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: d2d6233f-6944-40b5-97d7-2870da393d1b\n\t\tsourceLineageTag: ss_net_profit\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_net_profit\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Technical column used for cache invalidation in DirectLake mode.\n\tcolumn cache_buster\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 7ae60894-bf63-4fef-a327-bf8796466374\n\t\tsourceLineageTag: cache_buster\n\t\tsummarizeBy: none\n\t\tsourceColumn: cache_buster\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition store_sales = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: store_sales\n\t\t\tschemaName: tpcds_sf__SF___vonly\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition.pbism","text":"{\"$schema\": \"https://developer.microsoft.com/json-schemas/fabric/item/semanticModel/definitionProperties/1.0.0/schema.json\", \"version\": \"5.0\", \"settings\": {}}"}]},"duckdb":{"model":"tpcds_sf__SF___duckdb","parts":[{"path":"definition/database.tmdl","text":"database tpcds_sf__SF___duckdb\n\tcompatibilityLevel: 1702\n\tcompatibilityMode: powerBI\n\tlanguage: 1033\n\n"},{"path":"definition/expressions.tmdl","text":"expression 'DirectLake - tpcds_sf__SF__' = ```\n\t\tlet\n\t\t    Source = AzureStorage.DataLake(\"https://onelake.dfs.fabric.microsoft.com/51650f82-6bb5-4023-b0ab-db197d32e0be/f19d8f93-2956-4cf7-a1bb-c526d0a953cd\", [HierarchicalNavigation=true])\n\t\tin\n\t\t    Source\n\t\t\n\t\t```\n\tlineageTag: c6e3390d-cc84-4e13-ab93-53308a346336\n\n\tannotation PBI_IncludeFutureArtifacts = False\n\n"},{"path":"definition/model.tmdl","text":"model Model\n\tdirectLakeBehavior: directLakeOnly\n\tculture: en-US\n\tdefaultPowerBIDataSourceVersion: powerBI_V3\n\tsourceQueryCulture: en-US\n\tdataAccessOptions\n\t\tlegacyRedirects\n\t\treturnErrorValuesAsNull\n\nannotation PBI_QueryOrder = [\"DirectLake - tpcds_sf__SF__\"]\n\nannotation __PBI_TimeIntelligenceEnabled = 1\n\nannotation __LastRPTime = 134244136754838152\n\nannotation PBI_ProTooling = [\"DirectLakeOnOneLakeInWeb\",\"WebModelingEdit\"]\n\nannotation __TEdtr = 1\n\nannotation TabularEditor_SerializeOptions = {\"IgnoreInferredObjects\":true,\"IgnoreInferredProperties\":true,\"IgnoreTimestamps\":true,\"SplitMultilineStrings\":true,\"PrefixFilenames\":false,\"LocalTranslations\":true,\"LocalPerspectives\":true,\"LocalRelationships\":true,\"Levels\":[\"Data Sources\",\"Perspectives\",\"Relationships\",\"Roles\",\"Shared Expressions\",\"Tables\",\"Tables/Calculation Items\",\"Tables/Columns\",\"Tables/Hierarchies\",\"Tables/Measures\",\"Tables/Partitions\",\"Translations\"]}\n\nref table store\nref table item\nref table date_dim\nref table store_sales\nref table catalog_page\nref table promotion\nref table ship_mode\nref table catalog_sales\nref table customer_address\nref table customer_demographics\nref table 'Measures 1'\nref table 'Time Unit'\n\n"},{"path":"definition/relationships.tmdl","text":"relationship b3bd8d91-ae6b-4489-83cd-34ca2c3213d8\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_bill_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship b727abac-57ca-484a-8183-1b4c10921d84\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_bill_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship 41e21761-05b3-410b-960e-54805fb2c4f2\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_catalog_page_sk\n\ttoColumn: catalog_page.cp_catalog_page_sk\n\nrelationship c7ce7218-d702-40d6-94b4-80009e453fe9\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship 4d02a5f5-e67a-4f65-a203-192e1c2ef166\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_promo_sk\n\ttoColumn: promotion.p_promo_sk\n\nrelationship 545e95f8-e63b-4a27-9c91-3cd725ca10e0\n\tisActive: false\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship 685424a1-f6c7-42d6-8f44-3cccf5f08db4\n\tisActive: false\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship 2e0c0f18-79bf-48f8-be1a-2c53cc33e257\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_mode_sk\n\ttoColumn: ship_mode.sm_ship_mode_sk\n\nrelationship c5edc5f8-f418-44a4-a8aa-caff6e0bef29\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_sold_date_sk\n\ttoColumn: date_dim.d_date_sk_1\n\nrelationship 9f8c8977-b8ac-4985-aed9-7711ce5004af\n\tisActive: false\n\tfromColumn: promotion.p_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship f824425a-7dd8-4a70-9ee5-2042a74ed97d\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship be9c6d5e-0bed-4db1-8558-326d5dfba951\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship f172b575-b3ce-48a4-b264-2928516104a2\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship 95b347fa-cf7e-4885-8124-86eb2ff56351\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_promo_sk\n\ttoColumn: promotion.p_promo_sk\n\nrelationship c782badb-be54-4423-98b0-22eebad6aa29\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_sold_date_sk\n\ttoColumn: date_dim.d_date_sk_1\n\nrelationship 9e5ca2d3-a5aa-4d43-b00a-07e4a857b1e4\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_store_sk\n\ttoColumn: store.s_store_sk\n\n"},{"path":"definition/tables/Measures 1.tmdl","text":"/// Calculated measures table containing key business metrics for revenue, quantity, profit, tax, and performance analysis across store and catalog channels.\ntable 'Measures 1'\n\tlineageTag: ae7cf165-f530-4f98-a4c5-befd958ff771\n\n\t/// Revenue from catalog sales channel only, based on extended sales price\n\tmeasure 'Catalog Revenue' = SUM('catalog_sales'[cs_ext_sales_price])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 01. Revenue\n\t\tlineageTag: bed34121-c406-4e25-b931-6ea438aede2a\n\n\t/// Revenue from store sales channel only, based on extended sales price\n\tmeasure 'Store Revenue' = SUM('store_sales'[ss_ext_sales_price])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 01. Revenue\n\t\tlineageTag: 1755e841-2eeb-4e1e-9a98-c860eb583a79\n\n\t/// Total units sold through catalog sales channel\n\tmeasure 'Catalog Sales Quantity' = SUM('catalog_sales'[cs_quantity])\n\t\tformatString: #,0\n\t\tdisplayFolder: 02. Quantity\n\t\tlineageTag: 6da96e65-1eb2-4e76-afdb-50d9106f56c1\n\n\t/// Net profit from store sales channel only\n\tmeasure 'Store Net Profit' = SUM('store_sales'[ss_net_profit])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 03. Profit\n\t\tlineageTag: a5a04305-c986-4791-925e-b9115939e453\n\n\t/// Number of unique customers who made store purchases\n\tmeasure 'Store Distinct Customers' = DISTINCTCOUNT('store_sales'[ss_customer_sk])\n\t\tformatString: #,0\n\t\tdisplayFolder: 04. Distinct Counts\n\t\tlineageTag: ace0485f-176a-4401-bb53-41c99ce5c356\n\n\t/// Total revenue from beginning of year to current date selection\n\tmeasure 'Store Revenue YTD' = TOTALYTD([Store Revenue],'tpcds_calendar')\n\t\tformatString: $#,0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 0e0dbb36-1e75-4c77-8b7f-9a462a143dcb\n\n\t/// Revenue for the same period in the previous year\n\tmeasure 'Store Revenue Same Period LY' =\n\t\t\t\n\t\t\tCALCULATE(\n\t\t\t    [Store Revenue],\n\t\t\t    SAMEPERIODLASTYEAR('tpcds_calendar')\n\t\t\t)\n\t\tformatString: $#,0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 303dbdc1-0484-496c-b14f-a41b0ef53f82\n\n\t/// Revenue for the same period in the previous year\n\tmeasure 'Store Revenue YoY' =\n\t\t\tVAR CurrentYearRev = [Store Revenue]\n\t\t\tVAR PreviousYearRev = [Store Revenue Same Period LY]\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentYearRev - PreviousYearRev, PreviousYearRev, 0)\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 39648465-7aac-4652-8c2f-d6599df15204\n\n\t/// Catalog Sales Same Period LY\n\tmeasure 'Catalog Sales Same Period LY' =\n\t\t\t\n\t\t\tCALCULATE(\n\t\t\t    [Catalog Sales Quantity],\n\t\t\t    SAMEPERIODLASTYEAR('tpcds_calendar')\n\t\t\t)\n\t\tformatString: 0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: d1d5179d-688b-487c-8638-81780d95eb05\n\n\t/// Catalog Sales YoY\n\tmeasure 'Catalog Sales YoY' =\n\t\t\tVAR CurrentYearCatSales = [Catalog Sales Quantity]\n\t\t\tVAR PreviousYearCatSales = [Catalog Sales Same Period LY]\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentYearCatSales - PreviousYearCatSales, PreviousYearCatSales, 0)\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 0bc09d6d-1bdd-4f7e-ac26-a12e87ccf67d\n\n\t/// Store Profit % by Item Category\n\tmeasure 'Store Profit % by Item Category' = ```\n\t\t\t\n\t\t\tVAR CurrentProfit = [Store Net Profit]\n\t\t\tVAR TotalProfitAllCategories = \n\t\t\t    CALCULATE(\n\t\t\t        [Store Net Profit],\n\t\t\t        ALLEXCEPT(\n\t\t\t            'item',\n\t\t\t            'item'[i_brand],\n\t\t\t            'item'[i_manufact]\n\t\t\t        )\n\t\t\t    )\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentProfit, TotalProfitAllCategories, 0)\n\t\t\t```\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 05. Advanced % Share\n\t\tlineageTag: 04f417ef-ba48-4d9e-873d-76d516ec3a54\n\n\t/// Dummy\n\tcolumn Dummy\n\t\tformatString: 0\n\t\tlineageTag: c1dfb904-84a8-4dfc-a768-d5e10f70f255\n\t\tsummarizeBy: none\n\t\tisNameInferred\n\t\tsourceColumn: [Dummy]\n\n\tpartition 'Measures 1' = calculated\n\t\tmode: import\n\t\tsource = ROW(\"Dummy\", 1)\n\n"},{"path":"definition/tables/Time Unit.tmdl","text":"/// Field parameter table for dynamic time unit selection in reports, supporting Year and Quarter groupings.\ntable 'Time Unit'\n\tlineageTag: c100712f-ebcb-4d0f-a435-e1a4c8ce3229\n\n\t/// Display name for the time unit selection (Year, Quarter).\n\tcolumn 'Time Unit'\n\t\tlineageTag: bd5d5ab0-6269-4df0-bd39-9616f6e2b7d5\n\t\tsummarizeBy: none\n\t\tsourceColumn: [Value1]\n\t\tsortByColumn: 'Time Unit Order'\n\n\t\trelatedColumnDetails\n\t\t\tgroupByColumn: 'Time Unit Fields'\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// DAX column reference for the selected time unit.\n\tcolumn 'Time Unit Fields'\n\t\tisHidden\n\t\tlineageTag: 4be3ced6-f4db-40c4-84b2-5ef9decf7fee\n\t\tsummarizeBy: none\n\t\tsourceColumn: [Value2]\n\t\tsortByColumn: 'Time Unit Order'\n\n\t\textendedProperty ParameterMetadata =\n\t\t\t\t{\n\t\t\t\t  \"version\": 3,\n\t\t\t\t  \"kind\": 2\n\t\t\t\t}\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Sort order for time unit options in field parameter.\n\tcolumn 'Time Unit Order'\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 36566b71-2a4f-4ddb-86c0-b4c7b3d8ab7f\n\t\tsummarizeBy: sum\n\t\tsourceColumn: [Value3]\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition 'Time Unit' = calculated\n\t\tmode: import\n\t\tsource =\n\t\t\t\t{\n\t\t\t\t    (\"Year\", NAMEOF('date_dim'[d_year]), 0),\n\t\t\t\t    (\"Quarter\", NAMEOF('date_dim'[d_quarter_name]), 1)\n\t\t\t\t}\n\n\tannotation PBI_Id = 706957f972f8497896950148d2d5f0af\n\n"},{"path":"definition/tables/catalog_page.tmdl","text":"/// Catalog page dimension containing information about catalog pages used in catalog sales campaigns.\ntable catalog_page\n\tlineageTag: b4e5474e-8b18-42b8-9a56-9c2416950a0a\n\tsourceLineageTag: [dbo].[catalog_page]\n\n\t/// Catalog page surrogate key - unique identifier for each catalog page.\n\tcolumn cp_catalog_page_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 71a9dba7-e582-461e-b8de-f37322137084\n\t\tsourceLineageTag: cp_catalog_page_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: cp_catalog_page_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type or category of the catalog page.\n\tcolumn cp_type\n\t\tdataType: string\n\t\tlineageTag: bdf42608-dc88-489d-90ce-24a87bc40378\n\t\tsourceLineageTag: cp_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: cp_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition catalog_page = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: catalog_page\n\t\t\tschemaName: tpcds_sf__SF___duckdb\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/catalog_sales.tmdl","text":"/// Fact table containing catalog sales transactions with pricing, quantities, profits, and foreign keys to related dimensions.\ntable catalog_sales\n\tlineageTag: c38e1686-b3a0-473b-8604-8efe0410ea48\n\tsourceLineageTag: [dbo].[catalog_sales]\n\n\t/// Foreign key to date dimension for sale date.\n\tcolumn cs_sold_date_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: ac2ce865-caed-4039-bda1-a182c516cfdd\n\t\tsourceLineageTag: cs_sold_date_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_sold_date_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension for billing customer.\n\tcolumn cs_bill_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 310a5931-6249-4ad7-86b4-7aca5fbc39a9\n\t\tsourceLineageTag: cs_bill_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_bill_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension for billing address.\n\tcolumn cs_bill_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 51e6cbcf-e98d-4ea9-865e-a500419bec99\n\t\tsourceLineageTag: cs_bill_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_bill_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension for shipping customer.\n\tcolumn cs_ship_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4a433af2-d889-41d8-9c6f-1f05f94f0070\n\t\tsourceLineageTag: cs_ship_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension for shipping address.\n\tcolumn cs_ship_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 25b41390-6ff3-4f15-b0e6-f26a3da61ef5\n\t\tsourceLineageTag: cs_ship_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to catalog page dimension.\n\tcolumn cs_catalog_page_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: b01fc3a0-0a51-4660-a5d3-56d28ea386cd\n\t\tsourceLineageTag: cs_catalog_page_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_catalog_page_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to ship mode dimension.\n\tcolumn cs_ship_mode_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 0a85820a-e8f9-41f9-b50e-9610890408e8\n\t\tsourceLineageTag: cs_ship_mode_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_mode_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension.\n\tcolumn cs_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9024dff9-9629-4f0e-abae-0b29f00d568d\n\t\tsourceLineageTag: cs_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to promotion dimension.\n\tcolumn cs_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: dbfb7b81-9eef-4b86-aac0-a48cc36d0168\n\t\tsourceLineageTag: cs_promo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Catalog order number for this transaction.\n\tcolumn cs_order_number\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: ccec0342-6144-411f-bc2d-90234bcfb5ac\n\t\tsourceLineageTag: cs_order_number\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_order_number\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quantity of items ordered in this transaction.\n\tcolumn cs_quantity\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 0f5abf99-45b4-495b-bad3-795d0e8e5802\n\t\tsourceLineageTag: cs_quantity\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_quantity\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Wholesale cost per unit for this transaction.\n\tcolumn cs_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 37d760e7-f755-42e9-9dba-3a28cc211ae7\n\t\tsourceLineageTag: cs_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// List price per unit at time of order.\n\tcolumn cs_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 0eda1627-f565-444d-92e5-1a5bb0929e4d\n\t\tsourceLineageTag: cs_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Actual sales price per unit (after discounts).\n\tcolumn cs_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: eebce86e-da48-4799-8f93-1a8e69de6605\n\t\tsourceLineageTag: cs_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended sales price (sales price \u00d7 quantity).\n\tcolumn cs_ext_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: f0c74fde-bf40-47e4-9352-6bc4d4f219a0\n\t\tsourceLineageTag: cs_ext_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_ext_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended list price (list price \u00d7 quantity).\n\tcolumn cs_ext_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 845b8a06-567d-49c0-9455-1a469d81f646\n\t\tsourceLineageTag: cs_ext_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_ext_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Net profit for this transaction (sales price - wholesale cost).\n\tcolumn cs_net_profit\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 341a4a24-e726-4b20-8a50-967f689a5d01\n\t\tsourceLineageTag: cs_net_profit\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_net_profit\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Technical column used for cache invalidation in DirectLake mode.\n\tcolumn cache_buster\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 33947ffc-dc12-42ab-af62-5b00c42adf7a\n\t\tsourceLineageTag: cache_buster\n\t\tsummarizeBy: none\n\t\tsourceColumn: cache_buster\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition catalog_sales = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: catalog_sales\n\t\t\tschemaName: tpcds_sf__SF___duckdb\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/customer_address.tmdl","text":"/// Customer address dimension containing geographic information for billing and shipping addresses.\ntable customer_address\n\tlineageTag: 75246b34-d9f4-46fd-813f-bc5f0e47ab8c\n\tsourceLineageTag: [dbo].[customer_address]\n\n\t/// Customer address surrogate key - unique identifier for each address.\n\tcolumn ca_address_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 9084b5d7-d188-4fb9-9c92-211d5b68f397\n\t\tsourceLineageTag: ca_address_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_address_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// City name for the customer address.\n\tcolumn ca_city\n\t\tdataType: string\n\t\tlineageTag: 03b2fb94-a08c-4010-9157-b91edcb60daf\n\t\tsourceLineageTag: ca_city\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_city\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// County name for the customer address.\n\tcolumn ca_county\n\t\tdataType: string\n\t\tlineageTag: 0ef7c0d6-1e17-42c5-a9bb-fb014c91398a\n\t\tsourceLineageTag: ca_county\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_county\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// State or province for the customer address.\n\tcolumn ca_state\n\t\tdataType: string\n\t\tlineageTag: 24c6ee20-e4ab-4ca3-aa30-934ba66a55db\n\t\tsourceLineageTag: ca_state\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_state\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Postal code for the customer address.\n\tcolumn ca_zip\n\t\tdataType: string\n\t\tlineageTag: a57bf4fe-e3de-4848-81bd-e71dd885c99f\n\t\tsourceLineageTag: ca_zip\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_zip\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type of location (e.g., residential, commercial).\n\tcolumn ca_location_type\n\t\tdataType: string\n\t\tlineageTag: 9a90fb17-4314-40df-84ae-071cdd4eb3c7\n\t\tsourceLineageTag: ca_location_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_location_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition customer_address = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: customer_address\n\t\t\tschemaName: tpcds_sf__SF___duckdb\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/customer_demographics.tmdl","text":"/// Customer demographics dimension containing education status and marital status attributes for customer segmentation.\ntable customer_demographics\n\tlineageTag: 52f8f268-90f3-4f40-a527-07a74d9e9baf\n\tsourceLineageTag: [dbo].[customer_demographics]\n\n\t/// Customer demographics surrogate key - unique identifier for each demographic profile.\n\tcolumn cd_demo_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: d2cad4d0-4a88-41ab-b7ee-94ab73973694\n\t\tsourceLineageTag: cd_demo_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_demo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Marital status of the customer.\n\tcolumn cd_marital_status\n\t\tdataType: string\n\t\tlineageTag: e29a3681-2c0d-4579-9b35-5112d86c0f46\n\t\tsourceLineageTag: cd_marital_status\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_marital_status\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Education level of the customer.\n\tcolumn cd_education_status\n\t\tdataType: string\n\t\tlineageTag: 43260b21-0847-455f-a853-e2233af2a62e\n\t\tsourceLineageTag: cd_education_status\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_education_status\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition customer_demographics = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: customer_demographics\n\t\t\tschemaName: tpcds_sf__SF___duckdb\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/date_dim.tmdl","text":"/// Date dimension providing calendar hierarchy with year, quarter, month, and day-of-week attributes for time-based analysis.\ntable date_dim\n\tlineageTag: 9e7ec2de-382c-4a3b-9589-1ece445bfa27\n\tsourceLineageTag: [dbo].[date_dim]\n\n\t/// Date surrogate key - unique identifier for each calendar date.\n\tcolumn d_date_sk_1\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 63841738-6970-46ac-8106-970384b1052b\n\t\tsourceLineageTag: d_date_sk_1\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_date_sk_1\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Calendar date value.\n\tcolumn d_date\n\t\tdataType: dateTime\n\t\tformatString: General Date\n\t\tlineageTag: dc08a5e6-1a30-4d5f-9b03-1c98c7bac892\n\t\tsourceLineageTag: d_date\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_date\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Calendar year (e.g., 2023).\n\tcolumn d_year\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: fcf4b9e9-c880-4ee0-907c-3a18951b0809\n\t\tsourceLineageTag: d_year\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_year\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Day of week (1=Sunday, 7=Saturday).\n\tcolumn d_dow\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 339ac74d-ab31-4d4e-93eb-524b9f7e211c\n\t\tsourceLineageTag: d_dow\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_dow\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Month of year (1-12).\n\tcolumn d_moy\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: faa3deb0-9def-4334-bd42-a1b9d60b51b7\n\t\tsourceLineageTag: d_moy\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_moy\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Day of month (1-31).\n\tcolumn d_dom\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 83cd6ebd-7811-4616-a884-9930f66f8bfe\n\t\tsourceLineageTag: d_dom\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_dom\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quarter of year (1-4).\n\tcolumn d_qoy\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 4962ee0a-fcb9-4bf8-a5c7-43a66a94e67f\n\t\tsourceLineageTag: d_qoy\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_qoy\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quarter name (e.g., 'Q1 2023').\n\tcolumn d_quarter_name\n\t\tdataType: string\n\t\tlineageTag: 2426fa03-107c-4f51-a0b4-c59bb4b85d4f\n\t\tsourceLineageTag: d_quarter_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_quarter_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition date_dim = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: date_dim\n\t\t\tschemaName: tpcds_sf__SF___duckdb\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n\tcalendar tpcds_calendar\n\t\tlineageTag: dd529b7a-99fe-4cb5-928c-df1abed499f1\n\n\t\tcalendarColumnGroup = year\n\t\t\tprimaryColumn: d_year\n\n\t\tcalendarColumnGroup = quarter\n\t\t\tprimaryColumn: d_quarter_name\n\n\t\tcalendarColumnGroup = quarterOfYear\n\t\t\tprimaryColumn: d_qoy\n\n\t\tcalendarColumnGroup = date\n\t\t\tprimaryColumn: d_date\n\n"},{"path":"definition/tables/item.tmdl","text":"/// Product dimension containing item details such as brand, category, class, color, pricing, and manufacturing information.\ntable item\n\tlineageTag: 0f958ae1-af31-4b29-87a3-7e9ee589887e\n\tsourceLineageTag: [dbo].[item]\n\n\t/// Item surrogate key - unique identifier for each product.\n\tcolumn i_item_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 23893369-a589-4284-afa3-b1e291864600\n\t\tsourceLineageTag: i_item_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Current retail price of the product.\n\tcolumn i_current_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 70b27235-eefb-4732-8e0d-64c9576f0776\n\t\tsourceLineageTag: i_current_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_current_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Wholesale cost paid for the product.\n\tcolumn i_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 90cbdd03-df2f-4439-941d-de295ceb5135\n\t\tsourceLineageTag: i_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Brand name of the product.\n\tcolumn i_brand\n\t\tdataType: string\n\t\tlineageTag: 0c701039-61ca-4f5f-872b-aab6677ef45c\n\t\tsourceLineageTag: i_brand\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_brand\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Product class or subcategory within the main category.\n\tcolumn i_class\n\t\tdataType: string\n\t\tlineageTag: 5bdf127c-d558-46b2-98e6-0e294148d466\n\t\tsourceLineageTag: i_class\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_class\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Product category classification.\n\tcolumn i_category\n\t\tdataType: string\n\t\tlineageTag: 138de914-55e9-4a46-b5b9-6707b88a38e9\n\t\tsourceLineageTag: i_category\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_category\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Manufacturer or producer of the product.\n\tcolumn i_manufact\n\t\tdataType: string\n\t\tlineageTag: e6852c2b-2bc8-4fb1-86cb-d7ebe7c18637\n\t\tsourceLineageTag: i_manufact\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_manufact\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Size specification of the product.\n\tcolumn i_size\n\t\tdataType: string\n\t\tlineageTag: 0aeb0963-bc02-4389-a3fd-ab7232caaf9c\n\t\tsourceLineageTag: i_size\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_size\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Primary color of the product.\n\tcolumn i_color\n\t\tdataType: string\n\t\tlineageTag: d6709cb2-852f-485a-a0de-8cdd07941038\n\t\tsourceLineageTag: i_color\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_color\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition item = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: item\n\t\t\tschemaName: tpcds_sf__SF___duckdb\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/promotion.tmdl","text":"/// Promotion dimension containing promotional campaign details including costs and promotional names.\ntable promotion\n\tlineageTag: 5942c68a-d07a-4a23-8864-b2c733be46c8\n\tsourceLineageTag: [dbo].[promotion]\n\n\t/// Promotion surrogate key - unique identifier for each promotional campaign.\n\tcolumn p_promo_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 0299ca0a-f58f-4ebe-b752-ffa6b63f9760\n\t\tsourceLineageTag: p_promo_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension (for item-specific promotions).\n\tcolumn p_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 92f8bc42-df79-4a35-8a37-10a02a937c40\n\t\tsourceLineageTag: p_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: p_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Cost associated with running this promotional campaign.\n\tcolumn p_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: c42871df-1b39-4b1e-83cb-142e6864a1e0\n\t\tsourceLineageTag: p_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Name or title of the promotional campaign.\n\tcolumn p_promo_name\n\t\tdataType: string\n\t\tlineageTag: 6ceed2f6-4e12-4edc-b161-8251a356ec17\n\t\tsourceLineageTag: p_promo_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_promo_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition promotion = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: promotion\n\t\t\tschemaName: tpcds_sf__SF___duckdb\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/ship_mode.tmdl","text":"/// Shipping method dimension containing carrier information and shipping type classifications.\ntable ship_mode\n\tlineageTag: db1b87b5-f584-4f30-b70e-0eb00c8b45b8\n\tsourceLineageTag: [dbo].[ship_mode]\n\n\t/// Ship mode surrogate key - unique identifier for each shipping method.\n\tcolumn sm_ship_mode_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 4c485c32-fa22-4df6-b4a8-a8df04e2f730\n\t\tsourceLineageTag: sm_ship_mode_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_ship_mode_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type or category of shipping method.\n\tcolumn sm_type\n\t\tdataType: string\n\t\tlineageTag: c49fc6f8-d030-49a2-8105-6a45d1a81ac2\n\t\tsourceLineageTag: sm_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Internal code for the shipping method.\n\tcolumn sm_code\n\t\tdataType: string\n\t\tlineageTag: f54a092d-e847-4cee-84e6-5b813cbf5280\n\t\tsourceLineageTag: sm_code\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_code\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Shipping carrier or company name.\n\tcolumn sm_carrier\n\t\tdataType: string\n\t\tlineageTag: aca70983-6b51-4291-83f8-77fac2e381fe\n\t\tsourceLineageTag: sm_carrier\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_carrier\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition ship_mode = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: ship_mode\n\t\t\tschemaName: tpcds_sf__SF___duckdb\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/store.tmdl","text":"/// Store dimension containing retail location information including geographic details, management, and tax rates.\ntable store\n\tlineageTag: 6835d4a1-f40a-40aa-8a65-98aa4e06a406\n\tsourceLineageTag: [dbo].[store]\n\n\t/// Store surrogate key - unique identifier for each store location.\n\tcolumn s_store_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 14502476-8801-4068-b7c4-875ce7cac939\n\t\tsourceLineageTag: s_store_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_store_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Store name or identifier for the retail location.\n\tcolumn s_store_name\n\t\tdataType: string\n\t\tlineageTag: 4e78d335-4974-4d41-8402-ea89f63f9005\n\t\tsourceLineageTag: s_store_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_store_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Name of the store manager.\n\tcolumn s_manager\n\t\tdataType: string\n\t\tlineageTag: 3a8ddb5b-2ada-42d0-9ef9-89cd96b435b6\n\t\tsourceLineageTag: s_manager\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_manager\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Name of the market manager overseeing this store.\n\tcolumn s_market_manager\n\t\tdataType: string\n\t\tlineageTag: 37c8562d-3ab6-47ad-9b99-2ad1d72e6cd6\n\t\tsourceLineageTag: s_market_manager\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_market_manager\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// City where the store is located.\n\tcolumn s_city\n\t\tdataType: string\n\t\tlineageTag: a1a9209c-8194-4f96-8e94-69e545e064ba\n\t\tsourceLineageTag: s_city\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_city\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// County where the store is located.\n\tcolumn s_county\n\t\tdataType: string\n\t\tlineageTag: e219113c-fe81-45b0-b6be-f02e7ce75403\n\t\tsourceLineageTag: s_county\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_county\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// State or province where the store is located.\n\tcolumn s_state\n\t\tdataType: string\n\t\tlineageTag: fbaeba14-56ed-496d-a004-83f851771750\n\t\tsourceLineageTag: s_state\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_state\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Postal code for the store location.\n\tcolumn s_zip\n\t\tdataType: string\n\t\tlineageTag: 44e7ea79-56d6-4d20-8243-92c35919dcc3\n\t\tsourceLineageTag: s_zip\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_zip\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Tax rate percentage applied at this store location.\n\tcolumn s_tax_percentage\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: ba6d903d-f426-46b7-90a8-756a18261f2a\n\t\tsourceLineageTag: s_tax_percentage\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_tax_percentage\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\tpartition store = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: store\n\t\t\tschemaName: tpcds_sf__SF___duckdb\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/store_sales.tmdl","text":"/// Fact table containing retail store sales transactions with pricing, quantities, profits, and foreign keys to related dimensions.\ntable store_sales\n\tlineageTag: 6c7a2caf-3c22-4f38-a6aa-895a77e2e1c9\n\tsourceLineageTag: [dbo].[store_sales]\n\n\t/// Foreign key to date dimension for sale date.\n\tcolumn ss_sold_date_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4e30d94a-216f-4eaf-9858-fb22c6e6d57e\n\t\tsourceLineageTag: ss_sold_date_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_sold_date_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension.\n\tcolumn ss_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 2b151014-7df5-45f0-9907-03cbe191952b\n\t\tsourceLineageTag: ss_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Customer surrogate key for the transaction.\n\tcolumn ss_customer_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 558da56d-ea73-4562-b35c-ff94e774cca3\n\t\tsourceLineageTag: ss_customer_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_customer_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension.\n\tcolumn ss_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: e65f473f-bb26-4c7d-90f1-139de12a51ab\n\t\tsourceLineageTag: ss_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension.\n\tcolumn ss_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: cb65f3a1-6240-4baa-bfb2-4b24d39e2e87\n\t\tsourceLineageTag: ss_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to store dimension.\n\tcolumn ss_store_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 809a4a04-3bbf-4067-b54f-ee4d5fbbf6d4\n\t\tsourceLineageTag: ss_store_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_store_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to promotion dimension.\n\tcolumn ss_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9a64bbfb-0ae6-4919-b4e9-73e6d90353b3\n\t\tsourceLineageTag: ss_promo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Transaction ticket or receipt number.\n\tcolumn ss_ticket_number\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: d8ce37f4-a12c-4558-b131-66631546b607\n\t\tsourceLineageTag: ss_ticket_number\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ticket_number\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quantity of items sold in this transaction.\n\tcolumn ss_quantity\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 318e6a7b-88a2-413c-8257-315f66313d79\n\t\tsourceLineageTag: ss_quantity\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_quantity\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Wholesale cost per unit for this transaction.\n\tcolumn ss_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: d02100cc-2804-43d7-b752-8e98d579e202\n\t\tsourceLineageTag: ss_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// List price per unit at time of sale.\n\tcolumn ss_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 4f32f293-c19e-497e-906e-089bc204cde9\n\t\tsourceLineageTag: ss_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Actual sales price per unit (after discounts).\n\tcolumn ss_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: e620a719-f6e0-4965-8326-f4a2d255999b\n\t\tsourceLineageTag: ss_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended sales price (sales price \u00d7 quantity).\n\tcolumn ss_ext_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 8b66819a-a151-4fc7-9ec0-cacf7b6e8e01\n\t\tsourceLineageTag: ss_ext_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ext_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended list price (list price \u00d7 quantity).\n\tcolumn ss_ext_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: bc8e9c57-98e2-4032-a3fd-69808e3cdd8b\n\t\tsourceLineageTag: ss_ext_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ext_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Net profit for this transaction (sales price - wholesale cost).\n\tcolumn ss_net_profit\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: d2d6233f-6944-40b5-97d7-2870da393d1b\n\t\tsourceLineageTag: ss_net_profit\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_net_profit\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Technical column used for cache invalidation in DirectLake mode.\n\tcolumn cache_buster\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 7ae60894-bf63-4fef-a327-bf8796466374\n\t\tsourceLineageTag: cache_buster\n\t\tsummarizeBy: none\n\t\tsourceColumn: cache_buster\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition store_sales = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: store_sales\n\t\t\tschemaName: tpcds_sf__SF___duckdb\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition.pbism","text":"{\"$schema\": \"https://developer.microsoft.com/json-schemas/fabric/item/semanticModel/definitionProperties/1.0.0/schema.json\", \"version\": \"5.0\", \"settings\": {}}"}]},"ducksort":{"model":"tpcds_sf__SF___ducksort","parts":[{"path":"definition/database.tmdl","text":"database tpcds_sf__SF___ducksort\n\tcompatibilityLevel: 1702\n\tcompatibilityMode: powerBI\n\tlanguage: 1033\n\n"},{"path":"definition/expressions.tmdl","text":"expression 'DirectLake - tpcds_sf__SF__' = ```\n\t\tlet\n\t\t    Source = AzureStorage.DataLake(\"https://onelake.dfs.fabric.microsoft.com/51650f82-6bb5-4023-b0ab-db197d32e0be/f19d8f93-2956-4cf7-a1bb-c526d0a953cd\", [HierarchicalNavigation=true])\n\t\tin\n\t\t    Source\n\t\t\n\t\t```\n\tlineageTag: c6e3390d-cc84-4e13-ab93-53308a346336\n\n\tannotation PBI_IncludeFutureArtifacts = False\n\n"},{"path":"definition/model.tmdl","text":"model Model\n\tdirectLakeBehavior: directLakeOnly\n\tculture: en-US\n\tdefaultPowerBIDataSourceVersion: powerBI_V3\n\tsourceQueryCulture: en-US\n\tdataAccessOptions\n\t\tlegacyRedirects\n\t\treturnErrorValuesAsNull\n\nannotation PBI_QueryOrder = [\"DirectLake - tpcds_sf__SF__\"]\n\nannotation __PBI_TimeIntelligenceEnabled = 1\n\nannotation __LastRPTime = 134244136754838152\n\nannotation PBI_ProTooling = [\"DirectLakeOnOneLakeInWeb\",\"WebModelingEdit\"]\n\nannotation __TEdtr = 1\n\nannotation TabularEditor_SerializeOptions = {\"IgnoreInferredObjects\":true,\"IgnoreInferredProperties\":true,\"IgnoreTimestamps\":true,\"SplitMultilineStrings\":true,\"PrefixFilenames\":false,\"LocalTranslations\":true,\"LocalPerspectives\":true,\"LocalRelationships\":true,\"Levels\":[\"Data Sources\",\"Perspectives\",\"Relationships\",\"Roles\",\"Shared Expressions\",\"Tables\",\"Tables/Calculation Items\",\"Tables/Columns\",\"Tables/Hierarchies\",\"Tables/Measures\",\"Tables/Partitions\",\"Translations\"]}\n\nref table store\nref table item\nref table date_dim\nref table store_sales\nref table catalog_page\nref table promotion\nref table ship_mode\nref table catalog_sales\nref table customer_address\nref table customer_demographics\nref table 'Measures 1'\nref table 'Time Unit'\n\n"},{"path":"definition/relationships.tmdl","text":"relationship b3bd8d91-ae6b-4489-83cd-34ca2c3213d8\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_bill_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship b727abac-57ca-484a-8183-1b4c10921d84\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_bill_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship 41e21761-05b3-410b-960e-54805fb2c4f2\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_catalog_page_sk\n\ttoColumn: catalog_page.cp_catalog_page_sk\n\nrelationship c7ce7218-d702-40d6-94b4-80009e453fe9\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship 4d02a5f5-e67a-4f65-a203-192e1c2ef166\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_promo_sk\n\ttoColumn: promotion.p_promo_sk\n\nrelationship 545e95f8-e63b-4a27-9c91-3cd725ca10e0\n\tisActive: false\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship 685424a1-f6c7-42d6-8f44-3cccf5f08db4\n\tisActive: false\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship 2e0c0f18-79bf-48f8-be1a-2c53cc33e257\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_ship_mode_sk\n\ttoColumn: ship_mode.sm_ship_mode_sk\n\nrelationship c5edc5f8-f418-44a4-a8aa-caff6e0bef29\n\trelyOnReferentialIntegrity\n\tfromColumn: catalog_sales.cs_sold_date_sk\n\ttoColumn: date_dim.d_date_sk_1\n\nrelationship 9f8c8977-b8ac-4985-aed9-7711ce5004af\n\tisActive: false\n\tfromColumn: promotion.p_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship f824425a-7dd8-4a70-9ee5-2042a74ed97d\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_addr_sk\n\ttoColumn: customer_address.ca_address_sk\n\nrelationship be9c6d5e-0bed-4db1-8558-326d5dfba951\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_cdemo_sk\n\ttoColumn: customer_demographics.cd_demo_sk\n\nrelationship f172b575-b3ce-48a4-b264-2928516104a2\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_item_sk\n\ttoColumn: item.i_item_sk\n\nrelationship 95b347fa-cf7e-4885-8124-86eb2ff56351\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_promo_sk\n\ttoColumn: promotion.p_promo_sk\n\nrelationship c782badb-be54-4423-98b0-22eebad6aa29\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_sold_date_sk\n\ttoColumn: date_dim.d_date_sk_1\n\nrelationship 9e5ca2d3-a5aa-4d43-b00a-07e4a857b1e4\n\trelyOnReferentialIntegrity\n\tfromColumn: store_sales.ss_store_sk\n\ttoColumn: store.s_store_sk\n\n"},{"path":"definition/tables/Measures 1.tmdl","text":"/// Calculated measures table containing key business metrics for revenue, quantity, profit, tax, and performance analysis across store and catalog channels.\ntable 'Measures 1'\n\tlineageTag: ae7cf165-f530-4f98-a4c5-befd958ff771\n\n\t/// Revenue from catalog sales channel only, based on extended sales price\n\tmeasure 'Catalog Revenue' = SUM('catalog_sales'[cs_ext_sales_price])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 01. Revenue\n\t\tlineageTag: bed34121-c406-4e25-b931-6ea438aede2a\n\n\t/// Revenue from store sales channel only, based on extended sales price\n\tmeasure 'Store Revenue' = SUM('store_sales'[ss_ext_sales_price])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 01. Revenue\n\t\tlineageTag: 1755e841-2eeb-4e1e-9a98-c860eb583a79\n\n\t/// Total units sold through catalog sales channel\n\tmeasure 'Catalog Sales Quantity' = SUM('catalog_sales'[cs_quantity])\n\t\tformatString: #,0\n\t\tdisplayFolder: 02. Quantity\n\t\tlineageTag: 6da96e65-1eb2-4e76-afdb-50d9106f56c1\n\n\t/// Net profit from store sales channel only\n\tmeasure 'Store Net Profit' = SUM('store_sales'[ss_net_profit])\n\t\tformatString: $#,0\n\t\tdisplayFolder: 03. Profit\n\t\tlineageTag: a5a04305-c986-4791-925e-b9115939e453\n\n\t/// Number of unique customers who made store purchases\n\tmeasure 'Store Distinct Customers' = DISTINCTCOUNT('store_sales'[ss_customer_sk])\n\t\tformatString: #,0\n\t\tdisplayFolder: 04. Distinct Counts\n\t\tlineageTag: ace0485f-176a-4401-bb53-41c99ce5c356\n\n\t/// Total revenue from beginning of year to current date selection\n\tmeasure 'Store Revenue YTD' = TOTALYTD([Store Revenue],'tpcds_calendar')\n\t\tformatString: $#,0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 0e0dbb36-1e75-4c77-8b7f-9a462a143dcb\n\n\t/// Revenue for the same period in the previous year\n\tmeasure 'Store Revenue Same Period LY' =\n\t\t\t\n\t\t\tCALCULATE(\n\t\t\t    [Store Revenue],\n\t\t\t    SAMEPERIODLASTYEAR('tpcds_calendar')\n\t\t\t)\n\t\tformatString: $#,0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 303dbdc1-0484-496c-b14f-a41b0ef53f82\n\n\t/// Revenue for the same period in the previous year\n\tmeasure 'Store Revenue YoY' =\n\t\t\tVAR CurrentYearRev = [Store Revenue]\n\t\t\tVAR PreviousYearRev = [Store Revenue Same Period LY]\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentYearRev - PreviousYearRev, PreviousYearRev, 0)\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 39648465-7aac-4652-8c2f-d6599df15204\n\n\t/// Catalog Sales Same Period LY\n\tmeasure 'Catalog Sales Same Period LY' =\n\t\t\t\n\t\t\tCALCULATE(\n\t\t\t    [Catalog Sales Quantity],\n\t\t\t    SAMEPERIODLASTYEAR('tpcds_calendar')\n\t\t\t)\n\t\tformatString: 0\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: d1d5179d-688b-487c-8638-81780d95eb05\n\n\t/// Catalog Sales YoY\n\tmeasure 'Catalog Sales YoY' =\n\t\t\tVAR CurrentYearCatSales = [Catalog Sales Quantity]\n\t\t\tVAR PreviousYearCatSales = [Catalog Sales Same Period LY]\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentYearCatSales - PreviousYearCatSales, PreviousYearCatSales, 0)\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 06. Time Intelligence\n\t\tlineageTag: 0bc09d6d-1bdd-4f7e-ac26-a12e87ccf67d\n\n\t/// Store Profit % by Item Category\n\tmeasure 'Store Profit % by Item Category' = ```\n\t\t\t\n\t\t\tVAR CurrentProfit = [Store Net Profit]\n\t\t\tVAR TotalProfitAllCategories = \n\t\t\t    CALCULATE(\n\t\t\t        [Store Net Profit],\n\t\t\t        ALLEXCEPT(\n\t\t\t            'item',\n\t\t\t            'item'[i_brand],\n\t\t\t            'item'[i_manufact]\n\t\t\t        )\n\t\t\t    )\n\t\t\tRETURN\n\t\t\t    DIVIDE(CurrentProfit, TotalProfitAllCategories, 0)\n\t\t\t```\n\t\tformatString: 0.00%;-0.00%;0.00%\n\t\tdisplayFolder: 05. Advanced % Share\n\t\tlineageTag: 04f417ef-ba48-4d9e-873d-76d516ec3a54\n\n\t/// Dummy\n\tcolumn Dummy\n\t\tformatString: 0\n\t\tlineageTag: c1dfb904-84a8-4dfc-a768-d5e10f70f255\n\t\tsummarizeBy: none\n\t\tisNameInferred\n\t\tsourceColumn: [Dummy]\n\n\tpartition 'Measures 1' = calculated\n\t\tmode: import\n\t\tsource = ROW(\"Dummy\", 1)\n\n"},{"path":"definition/tables/Time Unit.tmdl","text":"/// Field parameter table for dynamic time unit selection in reports, supporting Year and Quarter groupings.\ntable 'Time Unit'\n\tlineageTag: c100712f-ebcb-4d0f-a435-e1a4c8ce3229\n\n\t/// Display name for the time unit selection (Year, Quarter).\n\tcolumn 'Time Unit'\n\t\tlineageTag: bd5d5ab0-6269-4df0-bd39-9616f6e2b7d5\n\t\tsummarizeBy: none\n\t\tsourceColumn: [Value1]\n\t\tsortByColumn: 'Time Unit Order'\n\n\t\trelatedColumnDetails\n\t\t\tgroupByColumn: 'Time Unit Fields'\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// DAX column reference for the selected time unit.\n\tcolumn 'Time Unit Fields'\n\t\tisHidden\n\t\tlineageTag: 4be3ced6-f4db-40c4-84b2-5ef9decf7fee\n\t\tsummarizeBy: none\n\t\tsourceColumn: [Value2]\n\t\tsortByColumn: 'Time Unit Order'\n\n\t\textendedProperty ParameterMetadata =\n\t\t\t\t{\n\t\t\t\t  \"version\": 3,\n\t\t\t\t  \"kind\": 2\n\t\t\t\t}\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Sort order for time unit options in field parameter.\n\tcolumn 'Time Unit Order'\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 36566b71-2a4f-4ddb-86c0-b4c7b3d8ab7f\n\t\tsummarizeBy: sum\n\t\tsourceColumn: [Value3]\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition 'Time Unit' = calculated\n\t\tmode: import\n\t\tsource =\n\t\t\t\t{\n\t\t\t\t    (\"Year\", NAMEOF('date_dim'[d_year]), 0),\n\t\t\t\t    (\"Quarter\", NAMEOF('date_dim'[d_quarter_name]), 1)\n\t\t\t\t}\n\n\tannotation PBI_Id = 706957f972f8497896950148d2d5f0af\n\n"},{"path":"definition/tables/catalog_page.tmdl","text":"/// Catalog page dimension containing information about catalog pages used in catalog sales campaigns.\ntable catalog_page\n\tlineageTag: b4e5474e-8b18-42b8-9a56-9c2416950a0a\n\tsourceLineageTag: [dbo].[catalog_page]\n\n\t/// Catalog page surrogate key - unique identifier for each catalog page.\n\tcolumn cp_catalog_page_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 71a9dba7-e582-461e-b8de-f37322137084\n\t\tsourceLineageTag: cp_catalog_page_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: cp_catalog_page_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type or category of the catalog page.\n\tcolumn cp_type\n\t\tdataType: string\n\t\tlineageTag: bdf42608-dc88-489d-90ce-24a87bc40378\n\t\tsourceLineageTag: cp_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: cp_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition catalog_page = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: catalog_page\n\t\t\tschemaName: tpcds_sf__SF___ducksort\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/catalog_sales.tmdl","text":"/// Fact table containing catalog sales transactions with pricing, quantities, profits, and foreign keys to related dimensions.\ntable catalog_sales\n\tlineageTag: c38e1686-b3a0-473b-8604-8efe0410ea48\n\tsourceLineageTag: [dbo].[catalog_sales]\n\n\t/// Foreign key to date dimension for sale date.\n\tcolumn cs_sold_date_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: ac2ce865-caed-4039-bda1-a182c516cfdd\n\t\tsourceLineageTag: cs_sold_date_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_sold_date_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension for billing customer.\n\tcolumn cs_bill_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 310a5931-6249-4ad7-86b4-7aca5fbc39a9\n\t\tsourceLineageTag: cs_bill_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_bill_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension for billing address.\n\tcolumn cs_bill_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 51e6cbcf-e98d-4ea9-865e-a500419bec99\n\t\tsourceLineageTag: cs_bill_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_bill_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension for shipping customer.\n\tcolumn cs_ship_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4a433af2-d889-41d8-9c6f-1f05f94f0070\n\t\tsourceLineageTag: cs_ship_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension for shipping address.\n\tcolumn cs_ship_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 25b41390-6ff3-4f15-b0e6-f26a3da61ef5\n\t\tsourceLineageTag: cs_ship_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to catalog page dimension.\n\tcolumn cs_catalog_page_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: b01fc3a0-0a51-4660-a5d3-56d28ea386cd\n\t\tsourceLineageTag: cs_catalog_page_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_catalog_page_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to ship mode dimension.\n\tcolumn cs_ship_mode_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 0a85820a-e8f9-41f9-b50e-9610890408e8\n\t\tsourceLineageTag: cs_ship_mode_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_ship_mode_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension.\n\tcolumn cs_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9024dff9-9629-4f0e-abae-0b29f00d568d\n\t\tsourceLineageTag: cs_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to promotion dimension.\n\tcolumn cs_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: dbfb7b81-9eef-4b86-aac0-a48cc36d0168\n\t\tsourceLineageTag: cs_promo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: cs_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Catalog order number for this transaction.\n\tcolumn cs_order_number\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: ccec0342-6144-411f-bc2d-90234bcfb5ac\n\t\tsourceLineageTag: cs_order_number\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_order_number\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quantity of items ordered in this transaction.\n\tcolumn cs_quantity\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 0f5abf99-45b4-495b-bad3-795d0e8e5802\n\t\tsourceLineageTag: cs_quantity\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_quantity\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Wholesale cost per unit for this transaction.\n\tcolumn cs_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 37d760e7-f755-42e9-9dba-3a28cc211ae7\n\t\tsourceLineageTag: cs_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// List price per unit at time of order.\n\tcolumn cs_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 0eda1627-f565-444d-92e5-1a5bb0929e4d\n\t\tsourceLineageTag: cs_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Actual sales price per unit (after discounts).\n\tcolumn cs_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: eebce86e-da48-4799-8f93-1a8e69de6605\n\t\tsourceLineageTag: cs_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended sales price (sales price \u00d7 quantity).\n\tcolumn cs_ext_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: f0c74fde-bf40-47e4-9352-6bc4d4f219a0\n\t\tsourceLineageTag: cs_ext_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_ext_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended list price (list price \u00d7 quantity).\n\tcolumn cs_ext_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 845b8a06-567d-49c0-9455-1a469d81f646\n\t\tsourceLineageTag: cs_ext_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_ext_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Net profit for this transaction (sales price - wholesale cost).\n\tcolumn cs_net_profit\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 341a4a24-e726-4b20-8a50-967f689a5d01\n\t\tsourceLineageTag: cs_net_profit\n\t\tsummarizeBy: none\n\t\tsourceColumn: cs_net_profit\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Technical column used for cache invalidation in DirectLake mode.\n\tcolumn cache_buster\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 33947ffc-dc12-42ab-af62-5b00c42adf7a\n\t\tsourceLineageTag: cache_buster\n\t\tsummarizeBy: none\n\t\tsourceColumn: cache_buster\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition catalog_sales = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: catalog_sales\n\t\t\tschemaName: tpcds_sf__SF___ducksort\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/customer_address.tmdl","text":"/// Customer address dimension containing geographic information for billing and shipping addresses.\ntable customer_address\n\tlineageTag: 75246b34-d9f4-46fd-813f-bc5f0e47ab8c\n\tsourceLineageTag: [dbo].[customer_address]\n\n\t/// Customer address surrogate key - unique identifier for each address.\n\tcolumn ca_address_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 9084b5d7-d188-4fb9-9c92-211d5b68f397\n\t\tsourceLineageTag: ca_address_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_address_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// City name for the customer address.\n\tcolumn ca_city\n\t\tdataType: string\n\t\tlineageTag: 03b2fb94-a08c-4010-9157-b91edcb60daf\n\t\tsourceLineageTag: ca_city\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_city\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// County name for the customer address.\n\tcolumn ca_county\n\t\tdataType: string\n\t\tlineageTag: 0ef7c0d6-1e17-42c5-a9bb-fb014c91398a\n\t\tsourceLineageTag: ca_county\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_county\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// State or province for the customer address.\n\tcolumn ca_state\n\t\tdataType: string\n\t\tlineageTag: 24c6ee20-e4ab-4ca3-aa30-934ba66a55db\n\t\tsourceLineageTag: ca_state\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_state\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Postal code for the customer address.\n\tcolumn ca_zip\n\t\tdataType: string\n\t\tlineageTag: a57bf4fe-e3de-4848-81bd-e71dd885c99f\n\t\tsourceLineageTag: ca_zip\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_zip\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type of location (e.g., residential, commercial).\n\tcolumn ca_location_type\n\t\tdataType: string\n\t\tlineageTag: 9a90fb17-4314-40df-84ae-071cdd4eb3c7\n\t\tsourceLineageTag: ca_location_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: ca_location_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition customer_address = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: customer_address\n\t\t\tschemaName: tpcds_sf__SF___ducksort\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/customer_demographics.tmdl","text":"/// Customer demographics dimension containing education status and marital status attributes for customer segmentation.\ntable customer_demographics\n\tlineageTag: 52f8f268-90f3-4f40-a527-07a74d9e9baf\n\tsourceLineageTag: [dbo].[customer_demographics]\n\n\t/// Customer demographics surrogate key - unique identifier for each demographic profile.\n\tcolumn cd_demo_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: d2cad4d0-4a88-41ab-b7ee-94ab73973694\n\t\tsourceLineageTag: cd_demo_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_demo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Marital status of the customer.\n\tcolumn cd_marital_status\n\t\tdataType: string\n\t\tlineageTag: e29a3681-2c0d-4579-9b35-5112d86c0f46\n\t\tsourceLineageTag: cd_marital_status\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_marital_status\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Education level of the customer.\n\tcolumn cd_education_status\n\t\tdataType: string\n\t\tlineageTag: 43260b21-0847-455f-a853-e2233af2a62e\n\t\tsourceLineageTag: cd_education_status\n\t\tsummarizeBy: none\n\t\tsourceColumn: cd_education_status\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition customer_demographics = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: customer_demographics\n\t\t\tschemaName: tpcds_sf__SF___ducksort\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/date_dim.tmdl","text":"/// Date dimension providing calendar hierarchy with year, quarter, month, and day-of-week attributes for time-based analysis.\ntable date_dim\n\tlineageTag: 9e7ec2de-382c-4a3b-9589-1ece445bfa27\n\tsourceLineageTag: [dbo].[date_dim]\n\n\t/// Date surrogate key - unique identifier for each calendar date.\n\tcolumn d_date_sk_1\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 63841738-6970-46ac-8106-970384b1052b\n\t\tsourceLineageTag: d_date_sk_1\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_date_sk_1\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Calendar date value.\n\tcolumn d_date\n\t\tdataType: dateTime\n\t\tformatString: General Date\n\t\tlineageTag: dc08a5e6-1a30-4d5f-9b03-1c98c7bac892\n\t\tsourceLineageTag: d_date\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_date\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Calendar year (e.g., 2023).\n\tcolumn d_year\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: fcf4b9e9-c880-4ee0-907c-3a18951b0809\n\t\tsourceLineageTag: d_year\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_year\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Day of week (1=Sunday, 7=Saturday).\n\tcolumn d_dow\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 339ac74d-ab31-4d4e-93eb-524b9f7e211c\n\t\tsourceLineageTag: d_dow\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_dow\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Month of year (1-12).\n\tcolumn d_moy\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: faa3deb0-9def-4334-bd42-a1b9d60b51b7\n\t\tsourceLineageTag: d_moy\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_moy\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Day of month (1-31).\n\tcolumn d_dom\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 83cd6ebd-7811-4616-a884-9930f66f8bfe\n\t\tsourceLineageTag: d_dom\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_dom\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quarter of year (1-4).\n\tcolumn d_qoy\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 4962ee0a-fcb9-4bf8-a5c7-43a66a94e67f\n\t\tsourceLineageTag: d_qoy\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_qoy\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quarter name (e.g., 'Q1 2023').\n\tcolumn d_quarter_name\n\t\tdataType: string\n\t\tlineageTag: 2426fa03-107c-4f51-a0b4-c59bb4b85d4f\n\t\tsourceLineageTag: d_quarter_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: d_quarter_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition date_dim = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: date_dim\n\t\t\tschemaName: tpcds_sf__SF___ducksort\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n\tcalendar tpcds_calendar\n\t\tlineageTag: dd529b7a-99fe-4cb5-928c-df1abed499f1\n\n\t\tcalendarColumnGroup = year\n\t\t\tprimaryColumn: d_year\n\n\t\tcalendarColumnGroup = quarter\n\t\t\tprimaryColumn: d_quarter_name\n\n\t\tcalendarColumnGroup = quarterOfYear\n\t\t\tprimaryColumn: d_qoy\n\n\t\tcalendarColumnGroup = date\n\t\t\tprimaryColumn: d_date\n\n"},{"path":"definition/tables/item.tmdl","text":"/// Product dimension containing item details such as brand, category, class, color, pricing, and manufacturing information.\ntable item\n\tlineageTag: 0f958ae1-af31-4b29-87a3-7e9ee589887e\n\tsourceLineageTag: [dbo].[item]\n\n\t/// Item surrogate key - unique identifier for each product.\n\tcolumn i_item_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 23893369-a589-4284-afa3-b1e291864600\n\t\tsourceLineageTag: i_item_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Current retail price of the product.\n\tcolumn i_current_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 70b27235-eefb-4732-8e0d-64c9576f0776\n\t\tsourceLineageTag: i_current_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_current_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Wholesale cost paid for the product.\n\tcolumn i_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 90cbdd03-df2f-4439-941d-de295ceb5135\n\t\tsourceLineageTag: i_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Brand name of the product.\n\tcolumn i_brand\n\t\tdataType: string\n\t\tlineageTag: 0c701039-61ca-4f5f-872b-aab6677ef45c\n\t\tsourceLineageTag: i_brand\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_brand\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Product class or subcategory within the main category.\n\tcolumn i_class\n\t\tdataType: string\n\t\tlineageTag: 5bdf127c-d558-46b2-98e6-0e294148d466\n\t\tsourceLineageTag: i_class\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_class\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Product category classification.\n\tcolumn i_category\n\t\tdataType: string\n\t\tlineageTag: 138de914-55e9-4a46-b5b9-6707b88a38e9\n\t\tsourceLineageTag: i_category\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_category\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Manufacturer or producer of the product.\n\tcolumn i_manufact\n\t\tdataType: string\n\t\tlineageTag: e6852c2b-2bc8-4fb1-86cb-d7ebe7c18637\n\t\tsourceLineageTag: i_manufact\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_manufact\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Size specification of the product.\n\tcolumn i_size\n\t\tdataType: string\n\t\tlineageTag: 0aeb0963-bc02-4389-a3fd-ab7232caaf9c\n\t\tsourceLineageTag: i_size\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_size\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Primary color of the product.\n\tcolumn i_color\n\t\tdataType: string\n\t\tlineageTag: d6709cb2-852f-485a-a0de-8cdd07941038\n\t\tsourceLineageTag: i_color\n\t\tsummarizeBy: none\n\t\tsourceColumn: i_color\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition item = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: item\n\t\t\tschemaName: tpcds_sf__SF___ducksort\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/promotion.tmdl","text":"/// Promotion dimension containing promotional campaign details including costs and promotional names.\ntable promotion\n\tlineageTag: 5942c68a-d07a-4a23-8864-b2c733be46c8\n\tsourceLineageTag: [dbo].[promotion]\n\n\t/// Promotion surrogate key - unique identifier for each promotional campaign.\n\tcolumn p_promo_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 0299ca0a-f58f-4ebe-b752-ffa6b63f9760\n\t\tsourceLineageTag: p_promo_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension (for item-specific promotions).\n\tcolumn p_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 92f8bc42-df79-4a35-8a37-10a02a937c40\n\t\tsourceLineageTag: p_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: p_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Cost associated with running this promotional campaign.\n\tcolumn p_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: c42871df-1b39-4b1e-83cb-142e6864a1e0\n\t\tsourceLineageTag: p_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Name or title of the promotional campaign.\n\tcolumn p_promo_name\n\t\tdataType: string\n\t\tlineageTag: 6ceed2f6-4e12-4edc-b161-8251a356ec17\n\t\tsourceLineageTag: p_promo_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: p_promo_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition promotion = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: promotion\n\t\t\tschemaName: tpcds_sf__SF___ducksort\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/ship_mode.tmdl","text":"/// Shipping method dimension containing carrier information and shipping type classifications.\ntable ship_mode\n\tlineageTag: db1b87b5-f584-4f30-b70e-0eb00c8b45b8\n\tsourceLineageTag: [dbo].[ship_mode]\n\n\t/// Ship mode surrogate key - unique identifier for each shipping method.\n\tcolumn sm_ship_mode_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 4c485c32-fa22-4df6-b4a8-a8df04e2f730\n\t\tsourceLineageTag: sm_ship_mode_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_ship_mode_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Type or category of shipping method.\n\tcolumn sm_type\n\t\tdataType: string\n\t\tlineageTag: c49fc6f8-d030-49a2-8105-6a45d1a81ac2\n\t\tsourceLineageTag: sm_type\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_type\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Internal code for the shipping method.\n\tcolumn sm_code\n\t\tdataType: string\n\t\tlineageTag: f54a092d-e847-4cee-84e6-5b813cbf5280\n\t\tsourceLineageTag: sm_code\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_code\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Shipping carrier or company name.\n\tcolumn sm_carrier\n\t\tdataType: string\n\t\tlineageTag: aca70983-6b51-4291-83f8-77fac2e381fe\n\t\tsourceLineageTag: sm_carrier\n\t\tsummarizeBy: none\n\t\tsourceColumn: sm_carrier\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition ship_mode = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: ship_mode\n\t\t\tschemaName: tpcds_sf__SF___ducksort\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/store.tmdl","text":"/// Store dimension containing retail location information including geographic details, management, and tax rates.\ntable store\n\tlineageTag: 6835d4a1-f40a-40aa-8a65-98aa4e06a406\n\tsourceLineageTag: [dbo].[store]\n\n\t/// Store surrogate key - unique identifier for each store location.\n\tcolumn s_store_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 14502476-8801-4068-b7c4-875ce7cac939\n\t\tsourceLineageTag: s_store_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_store_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Store name or identifier for the retail location.\n\tcolumn s_store_name\n\t\tdataType: string\n\t\tlineageTag: 4e78d335-4974-4d41-8402-ea89f63f9005\n\t\tsourceLineageTag: s_store_name\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_store_name\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Name of the store manager.\n\tcolumn s_manager\n\t\tdataType: string\n\t\tlineageTag: 3a8ddb5b-2ada-42d0-9ef9-89cd96b435b6\n\t\tsourceLineageTag: s_manager\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_manager\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Name of the market manager overseeing this store.\n\tcolumn s_market_manager\n\t\tdataType: string\n\t\tlineageTag: 37c8562d-3ab6-47ad-9b99-2ad1d72e6cd6\n\t\tsourceLineageTag: s_market_manager\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_market_manager\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// City where the store is located.\n\tcolumn s_city\n\t\tdataType: string\n\t\tlineageTag: a1a9209c-8194-4f96-8e94-69e545e064ba\n\t\tsourceLineageTag: s_city\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_city\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// County where the store is located.\n\tcolumn s_county\n\t\tdataType: string\n\t\tlineageTag: e219113c-fe81-45b0-b6be-f02e7ce75403\n\t\tsourceLineageTag: s_county\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_county\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// State or province where the store is located.\n\tcolumn s_state\n\t\tdataType: string\n\t\tlineageTag: fbaeba14-56ed-496d-a004-83f851771750\n\t\tsourceLineageTag: s_state\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_state\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Postal code for the store location.\n\tcolumn s_zip\n\t\tdataType: string\n\t\tlineageTag: 44e7ea79-56d6-4d20-8243-92c35919dcc3\n\t\tsourceLineageTag: s_zip\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_zip\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Tax rate percentage applied at this store location.\n\tcolumn s_tax_percentage\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: ba6d903d-f426-46b7-90a8-756a18261f2a\n\t\tsourceLineageTag: s_tax_percentage\n\t\tsummarizeBy: none\n\t\tsourceColumn: s_tax_percentage\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\tpartition store = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: store\n\t\t\tschemaName: tpcds_sf__SF___ducksort\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition/tables/store_sales.tmdl","text":"/// Fact table containing retail store sales transactions with pricing, quantities, profits, and foreign keys to related dimensions.\ntable store_sales\n\tlineageTag: 6c7a2caf-3c22-4f38-a6aa-895a77e2e1c9\n\tsourceLineageTag: [dbo].[store_sales]\n\n\t/// Foreign key to date dimension for sale date.\n\tcolumn ss_sold_date_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 4e30d94a-216f-4eaf-9858-fb22c6e6d57e\n\t\tsourceLineageTag: ss_sold_date_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_sold_date_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to item dimension.\n\tcolumn ss_item_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 2b151014-7df5-45f0-9907-03cbe191952b\n\t\tsourceLineageTag: ss_item_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_item_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Customer surrogate key for the transaction.\n\tcolumn ss_customer_sk\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 558da56d-ea73-4562-b35c-ff94e774cca3\n\t\tsourceLineageTag: ss_customer_sk\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_customer_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer demographics dimension.\n\tcolumn ss_cdemo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: e65f473f-bb26-4c7d-90f1-139de12a51ab\n\t\tsourceLineageTag: ss_cdemo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_cdemo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to customer address dimension.\n\tcolumn ss_addr_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: cb65f3a1-6240-4baa-bfb2-4b24d39e2e87\n\t\tsourceLineageTag: ss_addr_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_addr_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to store dimension.\n\tcolumn ss_store_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 809a4a04-3bbf-4067-b54f-ee4d5fbbf6d4\n\t\tsourceLineageTag: ss_store_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_store_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Foreign key to promotion dimension.\n\tcolumn ss_promo_sk\n\t\tdataType: int64\n\t\tisHidden\n\t\tformatString: 0\n\t\tlineageTag: 9a64bbfb-0ae6-4919-b4e9-73e6d90353b3\n\t\tsourceLineageTag: ss_promo_sk\n\t\tsummarizeBy: sum\n\t\tsourceColumn: ss_promo_sk\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Transaction ticket or receipt number.\n\tcolumn ss_ticket_number\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: d8ce37f4-a12c-4558-b131-66631546b607\n\t\tsourceLineageTag: ss_ticket_number\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ticket_number\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Quantity of items sold in this transaction.\n\tcolumn ss_quantity\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 318e6a7b-88a2-413c-8257-315f66313d79\n\t\tsourceLineageTag: ss_quantity\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_quantity\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t/// Wholesale cost per unit for this transaction.\n\tcolumn ss_wholesale_cost\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: d02100cc-2804-43d7-b752-8e98d579e202\n\t\tsourceLineageTag: ss_wholesale_cost\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_wholesale_cost\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// List price per unit at time of sale.\n\tcolumn ss_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 4f32f293-c19e-497e-906e-089bc204cde9\n\t\tsourceLineageTag: ss_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Actual sales price per unit (after discounts).\n\tcolumn ss_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: e620a719-f6e0-4965-8326-f4a2d255999b\n\t\tsourceLineageTag: ss_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended sales price (sales price \u00d7 quantity).\n\tcolumn ss_ext_sales_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: 8b66819a-a151-4fc7-9ec0-cacf7b6e8e01\n\t\tsourceLineageTag: ss_ext_sales_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ext_sales_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Extended list price (list price \u00d7 quantity).\n\tcolumn ss_ext_list_price\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: bc8e9c57-98e2-4032-a3fd-69808e3cdd8b\n\t\tsourceLineageTag: ss_ext_list_price\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_ext_list_price\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Net profit for this transaction (sales price - wholesale cost).\n\tcolumn ss_net_profit\n\t\tdataType: decimal\n\t\tformatString: \"\u00a3\"#,0.###############;-\"\u00a3\"#,0.###############;\"\u00a3\"#,0.###############\n\t\tlineageTag: d2d6233f-6944-40b5-97d7-2870da393d1b\n\t\tsourceLineageTag: ss_net_profit\n\t\tsummarizeBy: none\n\t\tsourceColumn: ss_net_profit\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\t\tannotation PBI_FormatHint = {\"currencyCulture\":\"en-GB\"}\n\n\t/// Technical column used for cache invalidation in DirectLake mode.\n\tcolumn cache_buster\n\t\tdataType: int64\n\t\tformatString: 0\n\t\tlineageTag: 7ae60894-bf63-4fef-a327-bf8796466374\n\t\tsourceLineageTag: cache_buster\n\t\tsummarizeBy: none\n\t\tsourceColumn: cache_buster\n\n\t\tannotation SummarizationSetBy = Automatic\n\n\tpartition store_sales = entity\n\t\tmode: directLake\n\t\tsource\n\t\t\tentityName: store_sales\n\t\t\tschemaName: tpcds_sf__SF___ducksort\n\t\t\texpressionSource: 'DirectLake - tpcds_sf__SF__'\n\n"},{"path":"definition.pbism","text":"{\"$schema\": \"https://developer.microsoft.com/json-schemas/fabric/item/semanticModel/definitionProperties/1.0.0/schema.json\", \"version\": \"5.0\", \"settings\": {}}"}]}}''')
ARM_OF = {'tpcds_sf__SF___default': 'default', 'tpcds_sf__SF___defaultf8': 'defaultf8', 'tpcds_sf__SF___default2rg': 'default2rg', 'tpcds_sf__SF___cluster': 'cluster', 'tpcds_sf__SF___clustersn': 'clustersn', 'tpcds_sf__SF___partition': 'partition', 'tpcds_sf__SF___vorder': 'vorder', 'tpcds_sf__SF___vonly': 'vonly', 'tpcds_sf__SF___duckdb': 'duckdb', 'tpcds_sf__SF___ducksort': 'ducksort'}

def model_name(arm, sf):
    """This arm's semantic model name at this scale factor."""
    return EMBEDDED_MODELS[arm]["model"].replace("__SF__", str(sf))


def model_parts(arm, sf):
    """The arm's TMDL as definition parts, with this run's scale factor filled in.

    The definitions are embedded DECODED with the scale factor left as a token, so one build of
    this notebook runs at any sf. A token rather than str.format() because TMDL is full of braces
    (DAX table constructors, `Measures 1`).
    """
    return [{"path": p["path"], "payloadType": "InlineBase64",
             "payload": base64.b64encode(
                 p["text"].replace("__SF__", str(sf)).encode()).decode()}
            for p in EMBEDDED_MODELS[arm]["parts"]]
# NOT part of the protocol any more: the three runs ARE the measurement, and a
# probe before run 1 would page in every column the suite reads and flatten run 1
# into run 2. Kept because it is still the honest way to time a transcode BY
# ITSELF -- every column the suite reads, generated from the capture, `Measures 1`
# and the relationships -- run by hand against a model nothing has touched.
TRANSCODE_DAX = 'EVALUATE ROW(\n    "c0", SUM(\'catalog_page\'[cp_catalog_page_sk]),\n    "c1", MIN(\'catalog_page\'[cp_type]),\n    "c2", SUM(\'catalog_sales\'[cache_buster]),\n    "c3", SUM(\'catalog_sales\'[cs_bill_addr_sk]),\n    "c4", SUM(\'catalog_sales\'[cs_bill_cdemo_sk]),\n    "c5", SUM(\'catalog_sales\'[cs_catalog_page_sk]),\n    "c6", SUM(\'catalog_sales\'[cs_ext_sales_price]),\n    "c7", SUM(\'catalog_sales\'[cs_item_sk]),\n    "c8", SUM(\'catalog_sales\'[cs_promo_sk]),\n    "c9", SUM(\'catalog_sales\'[cs_quantity]),\n    "c10", SUM(\'catalog_sales\'[cs_ship_addr_sk]),\n    "c11", SUM(\'catalog_sales\'[cs_ship_cdemo_sk]),\n    "c12", SUM(\'catalog_sales\'[cs_ship_mode_sk]),\n    "c13", SUM(\'catalog_sales\'[cs_sold_date_sk]),\n    "c14", SUM(\'customer_address\'[ca_address_sk]),\n    "c15", MIN(\'customer_address\'[ca_state]),\n    "c16", SUM(\'customer_demographics\'[cd_demo_sk]),\n    "c17", MIN(\'customer_demographics\'[cd_education_status]),\n    "c18", SUM(\'date_dim\'[d_date_sk_1]),\n    "c19", MIN(\'date_dim\'[d_quarter_name]),\n    "c20", SUM(\'date_dim\'[d_year]),\n    "c21", MIN(\'item\'[i_brand]),\n    "c22", MIN(\'item\'[i_category]),\n    "c23", SUM(\'item\'[i_item_sk]),\n    "c24", MIN(\'item\'[i_manufact]),\n    "c25", SUM(\'promotion\'[p_item_sk]),\n    "c26", MIN(\'promotion\'[p_promo_name]),\n    "c27", SUM(\'promotion\'[p_promo_sk]),\n    "c28", MIN(\'ship_mode\'[sm_carrier]),\n    "c29", SUM(\'ship_mode\'[sm_ship_mode_sk]),\n    "c30", MIN(\'store\'[s_manager]),\n    "c31", SUM(\'store\'[s_store_sk]),\n    "c32", SUM(\'store_sales\'[cache_buster]),\n    "c33", SUM(\'store_sales\'[ss_addr_sk]),\n    "c34", SUM(\'store_sales\'[ss_cdemo_sk]),\n    "c35", SUM(\'store_sales\'[ss_customer_sk]),\n    "c36", SUM(\'store_sales\'[ss_ext_sales_price]),\n    "c37", SUM(\'store_sales\'[ss_item_sk]),\n    "c38", SUM(\'store_sales\'[ss_net_profit]),\n    "c39", SUM(\'store_sales\'[ss_promo_sk]),\n    "c40", SUM(\'store_sales\'[ss_sold_date_sk]),\n    "c41", SUM(\'store_sales\'[ss_store_sk])\n)'

In [ ]:
def _report_children(outcome):
    """Print each child's own failure. `runMultiple` returns a per-activity dict and does NOT
    always raise, so without this a pass where all 20 users died the same way reports only that
    nothing was written -- never what went wrong inside them."""
    if not isinstance(outcome, dict):
        return
    items = outcome.get("results", outcome) if "results" in outcome else outcome
    if not isinstance(items, dict):
        return
    failed = 0
    for act, res in items.items():
        if not isinstance(res, dict):
            continue
        err = res.get("exception") or res.get("error") or res.get("errorMessage")
        if err:
            failed += 1
            if failed <= 3:      # they fail identically; three samples is the diagnosis
                print(f"  {act}: {str(err).splitlines()[0][:300]}", flush=True)
    if failed > 3:
        print(f"  ... and {failed - 3} more child(ren) failed the same way", flush=True)


def run_dax(workspace, dataset, run_index, threads, model_id="", model_created_utc="",
            single_query=""):
    """One load test over the suite at `threads` concurrency -- one of the three.

    `threads` is an argument, not the global: every run goes at the rung's concurrency, and it is
    what each result row records as its concurrent_threads.

    `run_index` is an ARGUMENT too, not the notebook parameter it used to be: the three runs happen
    inside one notebook invocation now, over one model, so it moves per call.

    `model_id` / `model_created_utc` describe the model these three runs share, and ride down to
    every row, so run 1 can be verified against the moment its model came into existence.

    Every activity is an independent child notebook session, so each virtual user opens its own
    XMLA connection -- which is what makes this a concurrency test rather than a loop.
    """
    # `cache` is DERIVED from run_index, never passed: two arguments that must agree is one that
    # can disagree, and a row claiming to be a warm run 1 would be undetectable afterwards.
    cache = int(run_index > 1)
    ts = time.strftime("%Y%m%d-%H%M%S")
    # Their analysis notebook parses loadtest_id with a Pattern_SF_Run-timestamp regex into
    # Pattern, SF, Run -- so keep that shape and our results drop straight into their comparison.
    # The pattern names are OURS (dbxcluster / fabvorder), not their db_dq / fab_dl, so nothing
    # masquerades as their data.
    loadtest_id = f"{PATTERN.get(dataset, dataset)}_{sf_label}_{run_index:02d}-{ts}"
    print(f"  load test {loadtest_id}: run {run_index}, {threads} thread(s), cache={cache}",
          flush=True)

    args = {
        "xmla_endpoint": f"powerbi://api.powerbi.com/v1.0/myorg/{workspace}",
        "perf_analyzer_filename": queryfile,
        "model": dataset,
        "roles": None,
        "customdata": None,
        "effective_username": None,
        "iterations": iterations,
        "delay_sec": delay_sec,
        "loadtestId": loadtest_id,
        "threadId": 0,
        "concurrent_threads": threads,
        "run_index": run_index,
        "useRootDefaultLakehouse": True,
        "cache": cache,
        "delta_path": delta_path,
        "nbr_queries": nbr_queries,
        "model_id": model_id,
        "model_created_utc": model_created_utc,
        "single_query": single_query,
    }
    activity = {
        "name": "RunPerfScenario",
        "path": "RunPerfScenario",
        "timeoutPerCellInSeconds": 90000,
        "args": {},
        "workspace": None,
        "retry": 0,                 # a dead thread stays dead; results.py drops the short rung
        "retryIntervalInSeconds": 0,
        "dependencies": [],         # no ordering: they all start together
    }
    DAG = {"activities": [], "timeoutInSeconds": 43200, "concurrency": threads}
    for i in range(threads):
        a = dict(activity)
        a["name"] = f"RunPerfScenario_{i}"
        a["args"] = {**args, "threadId": i}
        DAG["activities"].append(a)

    before = _results_rows()
    outcome = None
    try:
        outcome = notebookutils.notebook.runMultiple(DAG)
    except Exception as e:
        # Not fatal here: a thread that dies writes nothing and is simply absent from the table,
        # which results.py turns into a dropped short rung rather than a load test at a
        # concurrency that never happened. But it IS printed, and so is every child's own
        # exception below -- "see the snapshots" is useless advice when the snapshots are three
        # clicks deep in the monitoring UI and the job has already moved on.
        print(f"  load test error: {e}", flush=True)
    _report_children(outcome)
    after = _results_rows()
    print(f"  load test complete: {after - before:,} rows written "
          f"({before:,} -> {after:,})", flush=True)
    # A pass that wrote NOTHING is not a degraded rung, it is a broken harness -- every thread
    # failed the same way. Swallowing runMultiple's exception once turned exactly that into a
    # Completed job with an empty table, so the row count is the thing that decides.
    if after == before:
        raise RuntimeError(
            f"{dataset} run {run_index} at {threads} thread(s) wrote no rows to {results_table}. "
            f"Every virtual user failed identically -- see the RunPerfScenario snapshots above.")
    return loadtest_id

In [ ]:
from datetime import datetime, timezone

import requests

FABRIC_API = "https://api.fabric.microsoft.com"
POWERBI_API = "https://api.powerbi.com/v1.0/myorg"
MODEL_FOLDER = "dbx"       # workspace folder the recreated models are parked in


def _headers():
    return {"Authorization": f"Bearer {notebookutils.credentials.getToken('pbi')}",
            "Content-Type": "application/json"}


def _dataset_id(name):
    r = requests.get(f"{FABRIC_API}/v1/workspaces/{workspace_id}/semanticModels",
                     headers=_headers())
    r.raise_for_status()
    return next((i["id"] for i in r.json().get("value", []) if i["displayName"] == name), None)


def _reframe(dataset_id):
    """Full refresh so the Direct Lake tables are loaded. Everything here is ASYNC and that is the
    whole difficulty:

      * the POST itself can be REFUSED while a just-created model is still provisioning, so it is
        retried rather than raised on -- a 4xx here used to kill the run seconds after a
        successful create;
      * the refresh then runs in the background, so poll the specific request id from this POST.
        Do NOT trust `$top=1`: it can return a previous run's 'Completed' and hand back a model
        that has loaded nothing.

    Only when the poll says Completed has anything been loaded -- and even then the XMLA catalog
    lags, which is what `_wait_queryable` is for.
    """
    h = _headers()
    base = f"{POWERBI_API}/groups/{workspace_id}/datasets/{dataset_id}/refreshes"
    r = None
    for attempt in range(40):                 # up to ~5 min of "not ready to refresh yet"
        r = requests.post(base, headers=h, json={"type": "full"})
        if r.status_code in (200, 201, 202):
            break
        print(f"  refresh not accepted yet (attempt {attempt + 1}): "
              f"HTTP {r.status_code} {r.text[:140]}", flush=True)
        time.sleep(8)
        h = _headers()                        # the token can also have been the problem
    else:
        raise Exception(f"refresh never accepted: HTTP {r.status_code} {r.text[:200]}")
    req_id = r.headers.get("Location", "").rstrip("/").split("/")[-1] or r.headers.get("RequestId", "")
    for _ in range(240):
        time.sleep(5)
        if req_id:
            st = requests.get(f"{base}/{req_id}", headers=h).json().get("status")
        else:
            st = requests.get(f"{base}?$top=1", headers=h).json().get("value", [{}])[0].get("status")
        if st == "Completed":
            # A reframe reports Completed before the XMLA catalog is reliably queryable, so
            # settle before the first connect.
            time.sleep(30)
            return
        if st in ("Failed", "Disabled"):
            raise Exception(f"reframe {st}")
    raise Exception("reframe timed out")


def _delete_model(ds_id, name):
    r = requests.delete(f"{FABRIC_API}/v1/workspaces/{workspace_id}/semanticModels/{ds_id}",
                        headers=_headers())
    print(f"  deleted existing '{name}' ({ds_id}) -> HTTP {r.status_code}", flush=True)
    if r.status_code not in (200, 202, 204, 404):
        raise Exception(f"could not delete '{name}': HTTP {r.status_code} {r.text[:200]}")
    # The name has to be free before the create, and a delete is not instant.
    for _ in range(60):
        if _dataset_id(name) is None:
            return
        time.sleep(5)
    raise Exception(f"'{name}' still present 5 minutes after delete")


def _create_model(name, parts):
    """Create the semantic model from the paper's TMDL, in the `dbx` workspace folder.

    Inline on purpose -- NO duckrun import. This notebook is the one that writes the results, and
    installing duckrun here would drag a different duckdb/deltalake pair into the session doing the
    Delta write, which is the class of problem the pinned python3.12 kernel exists to avoid.

    The whole difficulty is the 202. A create that returns one has NOT applied the definition yet,
    and the item name resolves before it has: reading the id back at that point produced a HOLLOW
    model -- item present, refresh Completed in nine seconds, `store_sales` and `Measures 1` not
    resolving, and every virtual user dying on its first real query. So a 202 is polled to
    `Succeeded` and the id is taken from the operation result, never from a name lookup.
    """
    body = {"displayName": name, "definition": {"parts": parts}}
    if FOLDER_ID:
        body["folderId"] = FOLDER_ID
    url = f"{FABRIC_API}/v1/workspaces/{workspace_id}/semanticModels"
    last = None
    for fmt in (None, "TMDL"):          # TMSL goes in with no format; TMDL may want one
        b = dict(body)
        if fmt:
            b["definition"] = dict(body["definition"], format=fmt)
        r = requests.post(url, headers=_headers(), json=b)
        if r.status_code in (200, 201):
            return r.json()["id"]
        if r.status_code == 202:
            return _await_created_item(r)
        last = f"HTTP {r.status_code} {r.text[:200]}"
        print(f"  create rejected (format={fmt}): {last}", flush=True)
    raise Exception(f"could not create semantic model '{name}': {last}")


def _await_created_item(resp):
    """Poll a create long-running-operation to Succeeded and return the id it reports."""
    location = resp.headers.get("Location")
    if not location:
        raise Exception("create returned 202 with no Location to poll")
    for _ in range(120):                # 10 minutes
        time.sleep(5)
        r = requests.get(location, headers=_headers())
        r.raise_for_status()
        body = r.json()
        status = body.get("status")
        if status == "Succeeded":
            if body.get("id"):
                return body["id"]
            # Some tenants return the item from a /result sub-url rather than inline.
            rr = requests.get(location.rstrip("/") + "/result", headers=_headers())
            rr.raise_for_status()
            return rr.json()["id"]
        if status in ("Failed", "Undetermined"):
            raise Exception(f"item create failed: {body}")
    raise Exception("timed out creating the semantic model")


def _folder_id(name):
    """The id of the root-level workspace folder `name`, or "" if it cannot be resolved. Cosmetic:
    a model in the workspace root measures exactly the same, so this never fails the run."""
    try:
        r = requests.get(f"{FABRIC_API}/v1/workspaces/{workspace_id}/folders", headers=_headers())
        for f in r.json().get("value", []):
            if f.get("displayName") == name and not f.get("parentFolderId"):
                return f["id"]
    except Exception as e:                                          # noqa: BLE001
        print(f"  folder lookup failed ({e}); creating in the workspace root", flush=True)
    return ""



def _wait_queryable(name, minutes=15):
    """Block until the DRIVER can open an XMLA connection to `name` and get an answer.

    A model that was just CREATED is not immediately on the XMLA endpoint -- the item exists, the
    refresh reports Completed, and `Initial Catalog=<name>` still fails for a while. Without this
    gate the 20 children each hit that window, each retries for five minutes, and each dies having
    written nothing: the pass reports "every virtual user failed identically" and names no cause.
    One connection from the driver is the cheap way to find out, and it fails HERE, with the
    reason, instead of twenty times in child sessions whose logs are hard to reach.

    Deliberately the SAME mechanism the children use (AdomdConnection over the workspace XMLA
    endpoint), not the executeQueries REST API: REST can answer while XMLA is still catching up,
    and the children speak XMLA.
    """
    import sempy.fabric as fabric
    try:
        fabric.create_tom_server()
    except Exception:
        pass
    from Microsoft.AnalysisServices.AdomdClient import AdomdConnection

    endpoint = f"powerbi://api.powerbi.com/v1.0/myorg/{ws_name}"
    deadline = time.time() + minutes * 60
    attempt, last = 0, None
    while time.time() < deadline:
        attempt += 1
        con = AdomdConnection(f"Data Source={endpoint};Initial Catalog={name};"
                              f"password={notebookutils.credentials.getToken('pbi')};Timeout=600;")
        try:
            con.Open()
            cmd = con.CreateCommand()
            # NOT `ROW("n", 1)`: that resolves against nothing and happily passes on a model
            # with no tables in it, which is exactly the failure this gate exists to catch. Query
            # what a child needs -- a fact table and a measure off `Measures 1` -- so a hollow
            # model keeps failing here instead of killing every virtual user five minutes later.
            cmd.CommandText = ('EVALUATE ROW("rows", COUNTROWS(store_sales), '
                               '"rev", [Store Revenue])')
            rdr = cmd.ExecuteReader()
            rdr.Close()
            con.Close()
            print(f"  '{name}' answers over XMLA (attempt {attempt})", flush=True)
            return
        except Exception as e:
            last = str(e)
            if attempt == 1 or attempt % 5 == 0:
                print(f"  not queryable yet (attempt {attempt}): {last[:140]}", flush=True)
            time.sleep(15)
    raise Exception(f"'{name}' never became queryable over {endpoint} within {minutes} min. "
                    f"Last error: {last}")


def prepare_model(name):
    """DELETE the model, recreate it from the embedded TMDL, reframe it. Returns (id, created_utc).

    Recreating is what makes the first pass genuinely cold: a reframe alone leaves whatever the
    previous run paged in, so pass 1 would measure a half-warm model and call it a transcode. The
    definition is the paper's own TMDL, embedded by build_notebooks.py, so what is measured here
    and what `fabric/deploy_paper_model.py` deploys cannot drift apart.
    """
    # ARM_OF is keyed on the TOKENISED model name, so resolve the arm by shape at this sf
    # rather than by literal -- the definitions are scale-factor agnostic now.
    arm = next((a for a in EMBEDDED_MODELS if model_name(a, sf) == name), None)
    if arm is None:
        raise Exception(f"no embedded definition for '{name}' at sf={sf} - expected one of "
                        f"{[model_name(a, sf) for a in EMBEDDED_MODELS]}")
    existing = _dataset_id(name)
    if existing:
        _delete_model(existing, name)
    print(f"  creating '{name}' from the paper's TMDL...", flush=True)
    created_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")
    ds_id = _create_model(name, model_parts(arm, sf))
    print(f"  created '{name}' ({ds_id}) at {created_utc}", flush=True)
    print(f"  reframing '{name}'...", flush=True)
    _reframe(ds_id)
    print("  reframe complete", flush=True)
    _wait_queryable(name)
    return ds_id, created_utc


FOLDER_ID = ""             # resolved once, in run_test


def run_test():
    global FOLDER_ID
    FOLDER_ID = _folder_id(MODEL_FOLDER)
    print(f"models will be created in folder {MODEL_FOLDER!r} ({FOLDER_ID or 'workspace root'})",
          flush=True)
    for dataset in model_to_test:
        print(f"\n=== {dataset}", flush=True)
        # Recreating the model is FATAL if it fails, never skipped. Skipping produced a rung that
        # finished cleanly with one arm in it -- and a one-armed rung is indistinguishable,
        # downstream, from an arm that was never asked to run.
        model_id, created_utc = prepare_model(dataset)

        # THE PAPER'S PROTOCOL: three load tests over this ONE model, back to back. Run 1 is
        # cold in the only sense that matters -- the model came into existence moments ago, so
        # every column is transcoded inside the run by whichever query touches it first. That is
        # exactly what their Run 1 measures, which is why ours can be set beside theirs.
        #
        # NO sleep between runs. The gap is already runMultiple tearing down N child sessions and
        # starting N more; adding to it would only give the capacity time to evict what run 1 paid
        # for, and run 2 would come back part-cold.
        #
        # There is deliberately no transcode probe before run 1. It would page in every column the
        # suite reads and flatten run 1 into run 2, which destroys the warming curve -- the one
        # thing this protocol exists to produce.
        for r in range(1, runs + 1):
            print(f"  run {r} of {runs}: {concurrent_threads} thread(s), "
                  f"model created {created_utc}", flush=True)
            run_dax(ws_name, dataset, run_index=r, threads=concurrent_threads,
                    model_id=model_id, model_created_utc=created_utc)

        # All three runs are recorded, so the model has no further use -- drop it. This is the ONE
        # deviation from the paper's protocol (they left theirs standing): nothing then holds
        # capacity memory through the cool-down, and the next arm creates into an empty workspace
        # instead of racing a delete. The delete at the START of prepare_model stays as the safety
        # net: a run that died mid-pass leaves its model behind on purpose, for inspection, and the
        # next run clears it.
        # Deliberately NOT in a finally: a failed run keeps its model so there is something to
        # look at.
        _delete_model(model_id, dataset)

        time.sleep(300)   # cool-down after each model so back-to-back rungs start rested
    return "done"

In [ ]:
run_test()